# CS781 Phase 2 – LNER (all 4 Phase-1 backbones)

Each architecture section includes:
1. **Phase-1 model + tokenizer classes** (needed to reload checkpoints)
2. Full Phase-2 LNER cells (LoRA / BitFit / full-FT / KD, train with resumption, analysis, plots, submission)

Set paths via the activation cell, ensure tokenizer + Phase-1 checkpoint exist, Run All for that section.


In [3]:
# !mkdir -p /kaggle/working/cs781_phase1/
# !cp -rv /kaggle/input/datasets/vidya6364/phase1-models/* /kaggle/working/cs781_phase1/
# !cp -v /kaggle/input/datasets/vidya6364/phase1-tokenizers/* /kaggle/working/phase1_models/
# !ls /kaggle/input/datasets/vidya6364/models/

# !mkdir -p /kaggle/working/phase2-alibi-gqa/
# !mkdir -p /kaggle/working/phase2-custom-mha/
# !mkdir -p /kaggle/working/phase2-gqa-conv/
# !mkdir -p /kaggle/working/phase2-mla-moe/
# !cp -rv /kaggle/input/datasets/vidya6364/phase2-models/phase2_bundle_alibi_gqa/* /kaggle/working/phase2-alibi-gqa/
# !cp -rv /kaggle/input/datasets/vidya6364/phase2-models/phase2_bundle_custom_mha/* /kaggle/working/phase2-custom-mha/
# !cp -rv /kaggle/input/datasets/vidya6364/phase2-models/phase2_bundle_gqa_conv/* /kaggle/working/phase2-gqa-conv/
# !cp -rv /kaggle/input/datasets/vidya6364/phase2-models/phase2_bundle_mla_moe/* /kaggle/working/phase2-mla-moe/
# !ls

In [4]:
# !cp -rv /kaggle/input/datasets/vidya6364/phase1-models-1k/* /kaggle/working/cs781_phase1/
# !rm -rf /kaggle/working/cs781_phase1/phase1-models-1k/

In [5]:
# Shared path resolution for all 4 Phase-1 backbones

import os

OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATASET_DIR = os.environ.get("DATASET_DIR", OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

def _first(*paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return paths[0]

TOK_PATHS = {
    "custom_mha": _first(
        os.path.join(DATASET_DIR, "legal_bpe_custom_mha.json"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
    "mla_moe": _first(
        os.path.join(DATASET_DIR, "legal_bpe_mla_moe.json"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
    "gqa_conv": _first(
        os.path.join(DATASET_DIR, "legal_bpe_gqa_conv.json"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
    "gqa_alibi": _first(
        os.path.join(DATASET_DIR, "legal_bpe_tokenizer_gqa_alibi.pkl"),
        os.path.join(DATASET_DIR, "legal_bpe_alibi_gqa.pkl"),
        os.path.join(DATASET_DIR, "legal_bpe_v2.json"),
    ),
}

CKPT_V2 = _first(
    os.path.join(DATASET_DIR, "legal_lm_custom_mha.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_custom_mha.pt"),
    os.path.join(OUT_DIR, "legal_lm_custom_mha.pt"),
)
CKPT_V3 = _first(
    os.path.join(DATASET_DIR, "latest_checkpoint_mla_moe_1k.pt"),
    os.path.join(DATASET_DIR, "legal_lm_mla_moe.pt"),
    os.path.join(DATASET_DIR, "latest_checkpoint_mla_moe.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_mla_moe.pt"),
    os.path.join(OUT_DIR, "latest_checkpoint_v3.pt"),
)
CKPT_GQA = _first(
    os.path.join(DATASET_DIR, "legal_lm_gqa_conv.pt"),
    os.path.join(DATASET_DIR, "latest_checkpoint_gqa_conv.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_gqa_conv.pt"),
    os.path.join(OUT_DIR, "latest_checkpoint_v3_fixed.pt"),
)
CKPT_ALIBI = _first(
    os.path.join(DATASET_DIR, "legal_lm_alibi_gqa.pt"),
    os.path.join(DATASET_DIR, "latest_checkpoint_alibi_gqa.pt"),
    os.path.join(DATASET_DIR, "legal_lm_alibi.pt"),
    os.path.join(DATASET_DIR, "phase1_models", "legal_lm_alibi_gqa.pt"),
    os.path.join(OUT_DIR, "latest_checkpoint_alibi.pt"),
)
CKPTS = {
    "custom_mha": CKPT_V2,
    "mla_moe": CKPT_V3,
    "gqa_conv": CKPT_GQA,
    "gqa_alibi": CKPT_ALIBI,
}

print("Available tokenizers / checkpoints:")
for k in TOK_PATHS:
    print(f"  tok [{k:12s}] exists={str(os.path.exists(TOK_PATHS[k])):5s}  {TOK_PATHS[k]}")
for k in CKPTS:
    print(f"  ckpt[{k:12s}] exists={str(os.path.exists(CKPTS[k])):5s}  {CKPTS[k]}")


Available tokenizers / checkpoints:
  tok [custom_mha  ] exists=True   /kaggle/working/cs781_phase1/legal_bpe_custom_mha.json
  tok [mla_moe     ] exists=True   /kaggle/working/cs781_phase1/legal_bpe_mla_moe.json
  tok [gqa_conv    ] exists=True   /kaggle/working/cs781_phase1/legal_bpe_gqa_conv.json
  tok [gqa_alibi   ] exists=True   /kaggle/working/cs781_phase1/legal_bpe_tokenizer_gqa_alibi.pkl
  ckpt[custom_mha  ] exists=True   /kaggle/working/cs781_phase1/legal_lm_custom_mha.pt
  ckpt[mla_moe     ] exists=True   /kaggle/working/cs781_phase1/legal_lm_mla_moe.pt
  ckpt[gqa_conv    ] exists=True   /kaggle/working/cs781_phase1/legal_lm_gqa_conv.pt
  ckpt[gqa_alibi   ] exists=True   /kaggle/working/cs781_phase1/legal_lm_alibi_gqa.pt


**Reasoning**:
Standardize checkpoint and tokenizer loading paths for all four architectures and introduce the `LOAD_PRETRAINED` configuration flag.



In [6]:
# Dual-mode pretraining configuration
LOAD_PRETRAINED = True

import os

# Standardized path configuration verifying Phase-1 checkpoint and tokenizer presence
OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"[Configuration] LOAD_PRETRAINED set to: {LOAD_PRETRAINED}")
print(f"[Configuration] Standardized OUT_DIR: {OUT_DIR}")


[Configuration] LOAD_PRETRAINED set to: True
[Configuration] Standardized OUT_DIR: /kaggle/working/cs781_phase1


---
## Architecture: `custom_mha`



In [7]:
# ── Activate backbone: custom_mha ──────────────────────────────────────────────
BACKBONE = "custom_mha"
TOKENIZER_PATH = TOK_PATHS["custom_mha"]
PHASE1_CKPT = CKPTS["custom_mha"]
MODEL_PATH = PHASE1_CKPT
CKPT_PATH = PHASE1_CKPT
if "OUT_DIR" not in dir() or not OUT_DIR:
    OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"[active] BACKBONE={BACKBONE}")
print(f"[active] TOKENIZER_PATH={TOKENIZER_PATH}  exists={os.path.exists(TOKENIZER_PATH)}")
print(f"[active] PHASE1_CKPT={PHASE1_CKPT}  exists={os.path.exists(PHASE1_CKPT)}")


[active] BACKBONE=custom_mha
[active] TOKENIZER_PATH=/kaggle/working/cs781_phase1/legal_bpe_custom_mha.json  exists=True
[active] PHASE1_CKPT=/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt  exists=True


### Phase-1 classes for `custom_mha` (tokenizer + model, needed to reload checkpoints)

These cells are taken from Phase 1 of the same source notebook. They define `BPETokenizer`,
the model stack (`LegalLM` / `DualPathLM` / …), and `load_model`. Internal bodies are unchanged.


In [18]:
import os, sys, re, gc, json, math, time, heapq, random, hashlib, unicodedata, zipfile, subprocess
import multiprocessing as mp
from collections import Counter, defaultdict, deque
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint
from tqdm.auto import tqdm

try:
    import ijson
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ijson"])
    import ijson
from huggingface_hub import hf_hub_download, login

# ------------------------------- configuration -------------------------------
SEED = 42
HF_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# How many tokens to EXTRACT from each file (files differ a lot in size, so caps differ).
# Documents are sampled spread across the whole file, not just the first N rows.
SOURCE_TOKENS = {"acts": 15_000_000, "supreme_court": 35_000_000, "high_courts": 35_000_000, "tribunals": 25_000_000}
# How often each source is SAMPLED during training (equal by default).
SAMPLE_MIX = {"acts": 1.0, "supreme_court": 1.0, "high_courts": 0.15, "tribunals": 0.2}
VAL_TOKENS_PER_SOURCE = 500_000   # held-out docs (≈2% of docs, never used for training)
VAL_MOD = 50                      # doc goes to validation if hash % VAL_MOD == 0
BYTES_PER_TOKEN = 4.0             # rough, only used to decide how sparsely to sample a file
NUM_WORKERS = 3                   # tokenizer processes for preprocessing (0 = single process)

VOCAB_SIZE = 24000
TOK_CHARS_PER_SOURCE = 4_000_000  # text used to train the BPE, per source

# model (~66M params with SwiGLU)
BLOCK_SIZE, N_EMBD, N_LAYER, N_HEAD = 1024, 768, 8, 12
BATCH_SIZE, GRAD_ACCUM = 8, 8    # 16*4*1024 = 65k tokens / optimizer step
TRAIN_TOKENS = 150_000_000        # total tokens seen in training (several passes over small sources is fine)
LR, MIN_LR, WEIGHT_DECAY, WARMUP_STEPS = 6e-4, 6e-5, 0.1, 100
EVAL_EVERY, EVAL_WINDOWS, LOG_EVERY, CKPT_EVERY = 250, 48, 10, 250
MAX_TRAIN_HOURS = 8.0             # schedule is shortened automatically if throughput is too low
USE_GRAD_CKPT = True             # set True only if you hit OOM (about 30% slower)
USE_COMPILE = False

# generation
TARGET_WORDS, MAX_NEW_TOKENS, GEN_BATCH = 128, 320, 16
TEMPERATURE, TOP_P, TOP_K, REP_PENALTY = 0.7, 0.9, 50, 1.15

OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
TOKENIZER_PATH = os.path.join(OUT_DIR, "legal_bpe_v2.json")
MODEL_PATH = os.path.join(OUT_DIR, "legal_lm_custom_mha.pt")
CKPT_PATH = os.path.join(OUT_DIR, "legal_lm_custom_mha.pt")
PREDICTION_PATH = os.path.join(OUT_DIR, "prediction.json")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Never print the token (your previous notebook printed it - rotate that token on huggingface.co).
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    if HF_TOKEN: login(token=HF_TOKEN)
except Exception as e:
    print("HF login skipped:", e)
print("device:", DEVICE, "| amp dtype:", AMP_DTYPE if USE_AMP else None)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda | amp dtype: torch.bfloat16


## 1. Text cleaning + byte-level BPE tokenizer

In [19]:
_QUOTES = str.maketrans({"“": '"', "”": '"', "„": '"', "‘": "'", "’": "'", "‚": "'", "`": "'", "´": "'",
                         "–": "-", "—": "-", "−": "-", "\u00ad": ""})
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
_WS = re.compile(r"[ \t\u00a0\u1680\u2000-\u200b\u202f\u205f\u3000\ufeff]+")
_DOTS = re.compile(r"\.{4,}|(?:\. ){4,}")
_RULE = re.compile(r"[_\-=*~#]{3,}")
_PAGE = re.compile(r"(?mi)^[ \t]*(?:page\s+)?\d{1,4}[ \t]*$\n?")
_HYPH = re.compile(r"(?<=[a-z])-\n(?=[a-z])")
_WRAP = re.compile(r"(?<=[a-z,;])\n(?=[a-z(])")

def clean_text(t):
    """Light, conservative normalisation of legal text."""
    t = unicodedata.normalize("NFKC", str(t)).replace("\r\n", "\n").replace("\r", "\n")
    t = t.translate(_QUOTES)
    t = _CTRL.sub(" ", t)
    t = _HYPH.sub("", t)
    t = _DOTS.sub("...", t)
    t = _RULE.sub(" ", t)
    t = _PAGE.sub("", t)
    t = _WS.sub(" ", t)
    t = re.sub(r" *\n *", "\n", t)
    t = _WRAP.sub(" ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


class BPETokenizer:
    SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
    _split = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+")

    def __init__(self, vocab_size=12000):
        self.vocab_size = vocab_size
        self.merges = []
        self._reset()

    def _reset(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        self._cache = {}

    def _build(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        seen = set(self.itos[4:])
        for a, b in self.merges:
            m = a + b
            if m not in seen:
                seen.add(m); self.itos.append(m)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.ranks = {pair: r for r, pair in enumerate(self.merges)}
        self.merged = {pair: pair[0] + pair[1] for pair in self.merges}
        self._cache = {}
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        self.vocab_size = len(self.itos)

    def _word(self, w):
        r = self._cache.get(w)
        if r is not None: return r
        toks = [bytes([b]) for b in w.encode("utf-8")]
        while len(toks) > 1:
            best_i, best_r = -1, 1 << 60
            for i in range(len(toks) - 1):
                r_ = self.ranks.get((toks[i], toks[i + 1]), 1 << 60)
                if r_ < best_r: best_i, best_r = i, r_
            if best_i < 0: break
            toks[best_i:best_i + 2] = [toks[best_i] + toks[best_i + 1]]
        r = [self.stoi.get(t, self.unk_id) for t in toks]
        if len(self._cache) > 500_000: self._cache.clear()
        self._cache[w] = r
        return r

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        for w in self._split.findall(text):
            ids.extend(self._word(w))
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        if isinstance(ids, torch.Tensor): ids = ids.tolist()
        parts = []
        for i in ids:
            if i < 4: continue
            t = self.itos[i] if i < len(self.itos) else b""
            if isinstance(t, bytes):
                try: parts.append(t.decode("utf-8"))
                except Exception: parts.append("�")
            else:
                parts.append(str(t))
        return "".join(parts)

    def save(self, path):
        with open(path, "w") as f:
            json.dump({"merges": [[a.hex(), b.hex()] for a, b in self.merges]}, f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            data = json.load(f)
        tok = cls()
        tok.merges = [(bytes.fromhex(a), bytes.fromhex(b)) for a, b in data["merges"]]
        tok._build()
        return tok

    @classmethod
    def load_any(cls, path):
        """Load from either the original JSON format or a pickle dump.

        Why this exists
        ---------------
        Phase-1 models were trained with different BPE tables.  Some exports
        are the original JSON (list of hex merges); others are full pickle
        dumps of the tokenizer object or of a dict containing "merges".
        This helper accepts both so you can drop .pkl files straight into
        DATASET_DIR without converting them first.
        """
        path = str(path)
        if path.endswith((".pkl", ".pickle")):
            import pickle
            with open(path, "rb") as f:
                obj = pickle.load(f)
            if isinstance(obj, cls):
                return obj
            if isinstance(obj, dict):
                tok = cls()
                if "merges" in obj:
                    merges = obj["merges"]
                    # hex-encoded (from .json export) or raw bytes
                    if merges and isinstance(merges[0][0], str):
                        tok.merges = [(bytes.fromhex(a), bytes.fromhex(b)) for a, b in merges]
                    else:
                        tok.merges = [(bytes(a), bytes(b)) for a, b in merges]
                    tok._build()
                    return tok
                # full state-dict style
                for k, v in obj.items():
                    setattr(tok, k, v)
                if not getattr(tok, "stoi", None):
                    tok._build()
                return tok
            raise TypeError(f"Unrecognised pickle content in {path}: {type(obj)}")
        # original JSON path
        return cls.load(path)


assert os.path.exists(TOKENIZER_PATH), f"Tokenizer not found at {TOKENIZER_PATH}"
tokenizer = BPETokenizer.load_any(TOKENIZER_PATH)
print(f"Tokenizer loaded | vocab_size = {tokenizer.vocab_size} | from {TOKENIZER_PATH}")

# Pre-load every available tokenizer so Phase 4 can switch per experiment.
# Specials (pad/unk/bos/eos = 0/1/2/3) are identical across files; only merges/vocab differ.
TOKENIZERS = {}
for _name, _path in TOK_PATHS.items():
    if os.path.exists(_path):
        TOKENIZERS[_name] = BPETokenizer.load_any(_path)
        print(f"  tok[{_name}]: vocab={TOKENIZERS[_name].vocab_size}  <- {_path}")
    else:
        print(f"  tok[{_name}]: MISSING {_path}")
# keep global `tokenizer` pointing at the BACKBONE selected in the config cell
tokenizer = TOKENIZERS.get(BACKBONE, tokenizer)
print(f"Active tokenizer for BACKBONE={BACKBONE}: vocab={tokenizer.vocab_size}")


Tokenizer loaded | vocab_size = 30000 | from /kaggle/working/cs781_phase1/legal_bpe_v2.json
  tok[custom_mha]: vocab=24000  <- /kaggle/working/cs781_phase1/legal_bpe_custom_mha.json
  tok[mla_moe]: vocab=30000  <- /kaggle/working/cs781_phase1/legal_bpe_mla_moe.json
  tok[gqa_conv]: vocab=30000  <- /kaggle/working/cs781_phase1/legal_bpe_gqa_conv.json
  tok[gqa_alibi]: vocab=32000  <- /kaggle/working/cs781_phase1/legal_bpe_tokenizer_gqa_alibi.pkl
Active tokenizer for BACKBONE=custom_mha: vocab=24000


## 4. Model: RoPE + RMSNorm + SwiGLU decoder, with KV-cache generation

In [20]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.w

class RoPE(nn.Module):
    def __init__(self, hd, max_len, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, hd, 2).float() / hd))
        fr = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", fr.cos(), persistent=False)
        self.register_buffer("sin", fr.sin(), persistent=False)
    def forward(self, x, start=0):                      # x: (b, heads, t, hd)
        t = x.size(2)
        cos, sin = self.cos[start:start + t].to(x.dtype), self.sin[start:start + t].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((x1 * cos - x2 * sin, x2 * cos + x1 * sin), dim=-1)

class Attention(nn.Module):
    def __init__(self, dim, heads, block_size):
        super().__init__()
        assert dim % heads == 0
        self.h, self.hd = heads, dim // heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.rope = RoPE(self.hd, block_size)
    def forward(self, x, mask=None, cache=None, start=0):
        b, t, c = x.shape
        q, k, v = self.qkv(x).reshape(b, t, 3, self.h, self.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope(q, start), self.rope(k, start)
        if cache is not None:
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            cache = (k, v)
        if mask is not None:   y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        elif t > 1:            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:                  y = F.scaled_dot_product_attention(q, k, v)
        return self.proj(y.transpose(1, 2).reshape(b, t, c)), cache

class SwiGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hid = int(8 * dim / 3 / 64) * 64
        self.w13 = nn.Linear(dim, 2 * hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
    def forward(self, x):
        a, b = self.w13(x).chunk(2, dim=-1)
        return self.w2(F.silu(a) * b)

class Block(nn.Module):
    def __init__(self, dim, heads, block_size):
        super().__init__()
        self.n1, self.n2 = RMSNorm(dim), RMSNorm(dim)
        self.attn, self.mlp = Attention(dim, heads, block_size), SwiGLU(dim)
    def forward(self, x, mask=None, cache=None, start=0):
        h, cache = self.attn(self.n1(x), mask, cache, start)
        x = x + h
        return x + self.mlp(self.n2(x)), cache

class LegalLM(nn.Module):
    def __init__(self, vocab_size, dim=N_EMBD, layers=N_LAYER, heads=N_HEAD, block_size=BLOCK_SIZE):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, heads, block_size) for _ in range(layers)])
        self.ln_f = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight                      # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():                        # GPT-2 style scaled residual init
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * layers))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for blk in self.blocks:
            if self.training and USE_GRAD_CKPT:
                def run_block(t, block=blk):
                    out, _ = block(t, mask=None, cache=None, start=0)
                    return out

                x = torch.utils.checkpoint.checkpoint(
                    run_block, x, use_reentrant=False
                )
            else:
                x, _ = blk(x, mask=None, cache=None, start=0)
        logits = self.lm_head(self.ln_f(x))
        loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1)) if targets is not None else None
        return logits, loss

    def step(self, idx, caches, start, mask):
        """Inference step with KV cache. Returns logits of the LAST position and the updated caches."""
        x = self.wte(idx); new = []
        for blk, c in zip(self.blocks, caches):
            x, c = blk(x, mask, c, start); new.append(c)
        return self.lm_head(self.ln_f(x[:, -1:]))[:, 0], new


@torch.inference_mode()
def generate_batch(model, tok, prompts, max_new, temperature, top_p, top_k, rep_pen, target_words):
    """Batched sampling with left-padding. Returns a list of generated id-lists."""
    model.eval()
    dev = next(model.parameters()).device
    b, L = len(prompts), max(len(p) for p in prompts)
    assert L + max_new <= model.block_size, "prompt too long for the context window"
    idx = torch.full((b, L), tok.pad_id, dtype=torch.long)
    valid = torch.zeros(b, L, dtype=torch.bool)
    for i, p in enumerate(prompts):
        idx[i, L - len(p):] = torch.tensor(p); valid[i, L - len(p):] = True
    idx, valid = idx.to(dev), valid.to(dev)
    causal = torch.ones(L, L, dtype=torch.bool, device=dev).tril()
    eye = torch.eye(L, dtype=torch.bool, device=dev)
    mask = causal[None, None] & (valid[:, None, None, :] | eye[None, None])   # pad rows attend to themselves (no NaN)
    V = len(tok.itos)
    seen = torch.zeros(b, V, dtype=torch.bool, device=dev)
    gen = []
    with torch.autocast(device_type=dev.type, dtype=AMP_DTYPE, enabled=dev.type == "cuda"):
        logits, caches = model.step(idx, [(None, None)] * len(model.blocks), 0, mask)
        for n in range(max_new):
            lg = logits.float()
            lg[:, [tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id]] = float("-inf")   # always keep writing text
            if rep_pen != 1.0:
                lg = torch.where(seen, torch.where(lg > 0, lg / rep_pen, lg * rep_pen), lg)
            lg = lg / max(temperature, 1e-5)
            if top_k:
                kth = lg.topk(min(top_k, V), dim=-1).values[:, -1:]
                lg = lg.masked_fill(lg < kth, float("-inf"))
            if top_p < 1.0:
                sl, si = lg.sort(descending=True)
                cp = sl.softmax(-1).cumsum(-1)
                rm = cp - sl.softmax(-1) > top_p
                lg = lg.masked_fill(rm.scatter(1, si, rm), float("-inf"))
            nxt = torch.multinomial(lg.softmax(-1), 1)
            gen.append(nxt); seen.scatter_(1, nxt, True)
            valid = torch.cat([valid, torch.ones(b, 1, dtype=torch.bool, device=dev)], 1)
            if (n + 1) % 32 == 0:
                g = torch.cat(gen, 1).tolist()
                if all(len(tok.decode(r).split()) >= target_words + 8 for r in g): break
            if n + 1 < max_new:
                logits, caches = model.step(nxt, caches, L + n, valid[:, None, None, :])
    return torch.cat(gen, 1).tolist()

## 6. Generation: exactly 128 words per prompt

In [21]:
def load_model(path=MODEL_PATH):
    if os.path.exists(MODEL_PATH):
        path = MODEL_PATH
    else:
        path = "/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt"
    print(path)
    ck = torch.load(path, map_location=DEVICE, weights_only=False); c = ck["config"]
    m = LegalLM(c["vocab_size"], c["dim"], c["layers"], c["heads"], c["block_size"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()

def continue_texts(model, prompts, target_words=TARGET_WORDS, bs=GEN_BATCH, rounds=4, progress=True):
    max_ctx = BLOCK_SIZE - MAX_NEW_TOKENS
    ctx = [tokenizer.encode(clean_text(p), add_bos=True)[-max_ctx:] for p in prompts]   # no </s> after the prompt
    gen = [[] for _ in prompts]
    todo = list(range(len(prompts)))
    for _ in range(rounds):
        if not todo: break
        todo.sort(key=lambda i: len(ctx[i]))
        bar = tqdm(range(0, len(todo), bs), desc="generating", disable=not progress)
        for s in bar:
            chunk = todo[s:s + bs]
            outs = generate_batch(model, tokenizer, [ctx[i] for i in chunk], MAX_NEW_TOKENS,
                                  TEMPERATURE, TOP_P, TOP_K, REP_PENALTY, target_words)
            for i, o in zip(chunk, outs):
                gen[i] += o; ctx[i] = (ctx[i] + o)[-max_ctx:]
        todo = [i for i in todo if len(tokenizer.decode(gen[i]).split()) < target_words]
    return [" ".join(tokenizer.decode(g).split()[:target_words]) for g in gen]

model = load_model()
samples = ["The respondent submits that the appeal should be dismissed.",
           "Section 302 of the Indian Penal Code shall apply.",
           "The Court, having considered the evidence, hereby orders"]
for p, o in zip(samples, continue_texts(model, samples, progress=False)):
    print(f"PROMPT: {p}\n -> [{len(o.split())} words] {o}\n")

/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt
PROMPT: The respondent submits that the appeal should be dismissed.
 -> [128 words] The appeal is disposed of accordingly. (Dr. P.K. Sharma) (S.R. GAVAKSH) (ORAL) P.M.V.J.C.: I.A.G.P.(MD) No. 12 of 2009(D) W.P.No.665 of 2008(G) ORDER: 1. The respondent no.2 has filed this application in the Court of Additional Chief Judicial Magistrate, J&M, Bangalore against the order dated 19th September, 1999 passed by the trial court and as such the case may be is pending before the learned District Judge. The said order reads as under:- "It is hereby directed that all parties are to file an application in respect of the matter for consideration of this case." 2. The facts stated therein have been mentioned in the impugned judgment and are not relevant. We have heard Mr. Justice D.N. Agarwal, learned counsel appearing on behalf of the respondents-respondents. In support of

PROMPT: Section 302 of the Indian Penal Code shall apply.
 -> [128 words] 

### Phase content (`p2`) – cells [23:44] from source


# Phase 2

In [10]:
import os, json, pickle, random, math, time
from pathlib import Path
from dataclasses import fields
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# SEED = 42
HF_REPO = "Exploration-Lab/CS-781-Capstone"
# IMPORTANT: point these to your actual Phase-1 LEGAL model artifacts.
# Example only: change this directory to the folder containing your LegalLM.
LEGAL_MODEL_DIR = Path("/kaggle/working/cs781_phase1")
# MODEL_CONFIG_PATH = LEGAL_MODEL_DIR / "config.json"
MODEL_WEIGHTS_PATH = LEGAL_MODEL_DIR / "legal_lm_custom_mha.pt"
TOKENIZER_STATE_PATH = LEGAL_MODEL_DIR / "legal_bpe_v2.json"

OUTPUT_DIR = Path("/kaggle/working/lner_full_finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DOCS = None             # None = use all available annotated documents
VALIDATION_FRACTION = 0.10
EPOCHS = 4
BATCH_SIZE = 4              # reduce to 1 or 2 if GPU memory is limited
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WINDOW_OVERLAP = 64

In [11]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold1-gold.json"]
VAL_FILES = ["Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json"]

In [12]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [13]:
DATA_DIR = Path("/kaggle/working/Phase_2_LNER")
TRAIN_DIR = DATA_DIR / "Train"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def download_lner_file(filename):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename=f"Phase_2_LNER/Train/{filename}",
        local_dir="/kaggle/working/hf_legal_data"
    )

paths = {
    "fold1_text": download_lner_file("fold1-text.json"),
    "fold1_gold": download_lner_file("fold1-gold.json"),
    "fold2_text": download_lner_file("fold2-text.json"),
    "fold2_gold": download_lner_file("fold2-gold.json"),
    "labels": download_lner_file("labels.txt"),
}

with open(paths["fold1_text"], encoding="utf-8") as f:
    text1 = json.load(f)
with open(paths["fold1_gold"], encoding="utf-8") as f:
    gold1 = json.load(f)
with open(paths["fold2_text"], encoding="utf-8") as f:
    text2 = json.load(f)
with open(paths["fold2_gold"], encoding="utf-8") as f:
    gold2 = json.load(f)

with open(paths["labels"], encoding="utf-8") as f:
    ENTITY_TYPES = [line.strip() for line in f if line.strip()]

print("Entity types:", ENTITY_TYPES)
print("Fold 1 documents:", len(text1), "annotations:", len(gold1))
print("Fold 2 documents:", len(text2), "annotations:", len(gold2))

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Entity types: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1 documents: 35 annotations: 35
Fold 2 documents: 35 annotations: 35


In [14]:
def load_model(path=MODEL_PATH):
    if os.path.exists(MODEL_PATH):
        path = MODEL_PATH
    else:
        path = "/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt"
    print(path)
    ck = torch.load(path, map_location=DEVICE, weights_only=False); c = ck["config"]
    m = LegalLM(c["vocab_size"], c["dim"], c["layers"], c["heads"], c["block_size"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()

model = load_model()

/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt


In [15]:
# tokenizer = BPETokenizer.load(TOKENIZER_PATH)
# print("loaded tokenizer, vocab", tokenizer.vocab_size)

In [16]:
def normalize_entity(e):
    if isinstance(e, dict):
        start = e.get("start", e.get("start_offset", e.get("start_char")))
        end = e.get("end", e.get("end_offset", e.get("end_char")))
        label = e.get("label", e.get("entity_label", e.get("type")))
        entity_text = e.get("text", e.get("entity_text"))
    elif isinstance(e, (list, tuple)) and len(e) >= 3:
        start, end, label = e[:3]
        entity_text = e[3] if len(e) >= 4 else None
    else:
        raise ValueError(f"Unrecognized entity annotation: {e!r}")

    if start is None or end is None or label is None:
        raise ValueError(f"Missing start/end/label: {e!r}")

    return int(start), int(end), str(label), entity_text

def get_entity_list(gold_record):
    # Supports a direct list of entities or a wrapper with an entities field.
    if isinstance(gold_record, dict) and "entities" in gold_record:
        gold_record = gold_record["entities"]
    if not isinstance(gold_record, list):
        raise ValueError(f"Expected entity list; got {type(gold_record)}")
    return [normalize_entity(e) for e in gold_record]

def validate_document(text, entities, doc_id):
    for start, end, label, entity_text in entities:
        assert 0 <= start < end <= len(text), (
            f"Invalid offsets for document {doc_id}: {(start, end, label)}"
        )
        assert label in ENTITY_TYPES, (
            f"Unknown label {label!r} in document {doc_id}"
        )
        if entity_text is not None and text[start:end] != entity_text:
            raise ValueError(
                f"Text/offset mismatch in {doc_id}: "
                f"{text[start:end]!r} != {entity_text!r}; "
                "check the dataset's end-offset convention."
            )

documents = {}
annotations = {}

for texts, golds in ((text1, gold1), (text2, gold2)):
    for doc_id, text in texts.items():
        if doc_id not in golds:
            raise KeyError(f"No gold annotations for document {doc_id}")
        entities = get_entity_list(golds[doc_id])
        validate_document(text, entities, doc_id)
        documents[str(doc_id)] = text
        annotations[str(doc_id)] = entities

if MAX_DOCS is not None:
    selected_ids = list(documents)[:MAX_DOCS]
    documents = {k: documents[k] for k in selected_ids}
    annotations = {k: annotations[k] for k in selected_ids}

print("Total documents:", len(documents))
example_id = next(iter(documents))
print("Example ID:", example_id)
print("Example text:", documents[example_id][:500])
print("Example entities:", annotations[example_id][:10])

Total documents: 70
Example ID: 115651329
Example text: REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 
Example entities: [(137, 153, 'RESP', 'STATE OF HARYANA'), (252, 261, 'DATE', '19.8.2011'), (276, 308, 'COURT', 'High Court of Punjab and Haryana'), (359, 369, 'COURT', 'High Court'), (575, 585, 'DATE', '18.11.1994'), (637, 644, 'WIT', 'Jagdish'), (695, 704, 'WIT', 'Mange Ram'), (792, 800, 'APP', 'Jage Ram'), (1122, 1130, 'APP', 'Jage Ram'), (1341, 1349, 'APP', 'Jage Ram')]


In [17]:
def encode_with_offsets(text, tok):
    ids, offsets = [], []
    pattern = tok._split

    for match in pattern.finditer(text):
        word = match.group()
        if not word:
            continue

        # Initial byte tokens, each with its byte interval.
        raw = word.encode("utf-8")
        pieces = [(bytes([b]), i, i + 1) for i, b in enumerate(raw)]

        # Convert byte intervals to character intervals within this pre-token.
        char_byte_intervals = []
        cursor = 0
        for ch in word:
            nbytes = len(ch.encode("utf-8"))
            char_byte_intervals.append((cursor, cursor + nbytes))
            cursor += nbytes

        # Apply the same lowest-rank adjacent-pair merges as BPETokenizer._word.
        while len(pieces) > 1:
            best_i, best_rank = -1, float("inf")
            for i in range(len(pieces) - 1):
                rank = tok.ranks.get((pieces[i][0], pieces[i + 1][0]), float("inf"))
                if rank < best_rank:
                    best_i, best_rank = i, rank
            if best_i < 0:
                break
            a, b = pieces[best_i], pieces[best_i + 1]
            pieces[best_i:best_i + 2] = [(a[0] + b[0], a[1], b[2])]

        for piece, byte_start, byte_end in pieces:
            if piece not in tok.stoi:
                raise ValueError(f"Token not in pretrained vocabulary: {piece!r}")
            char_start = next(
                (i for i, (_, end) in enumerate(char_byte_intervals)
                 if end > byte_start),
                len(word) - 1
            )
            char_end = next(
                (i for i, (start, _) in enumerate(char_byte_intervals)
                 if start >= byte_end),
                len(word)
            )
            ids.append(tok.stoi[piece])
            offsets.append((
                match.start() + char_start,
                match.start() + char_end
            ))

    return ids, offsets

# Sanity checks before training.
for sample in [
    "IN THE SUPREME COURT OF INDIA",
    "Civil Appeal No. 123 of 2024",
    "Justice R. Kumar",
    "12 January 2024"
]:
    ids, offsets = encode_with_offsets(sample, tokenizer)
    assert len(ids) == len(offsets)
    assert all(0 <= s < e <= len(sample) for s, e in offsets)
    print(sample, "=>", len(ids), "tokens")

IN THE SUPREME COURT OF INDIA => 6 tokens
Civil Appeal No. 123 of 2024 => 7 tokens
Justice R. Kumar => 4 tokens
12 January 2024 => 3 tokens


In [18]:
import json, math, random, time, copy, zipfile, os
from collections import Counter
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WIN, STRIDE = 512, 256          # window length / training stride (tokens)
OUT = "/kaggle/working/phase2-custom-mha"; os.makedirs(OUT, exist_ok=True)
CAUSAL = False                  # True = keep the pretrained causal mask (ablation)

In [19]:

def dl(name):
    return hf_hub_download(HF_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", token=HF_TOKEN)

LABELS = [l.strip() for l in open(dl("Train/labels.txt")) if l.strip()]
BIO = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
l2i = {l: i for i, l in enumerate(BIO)}; NL = len(BIO)

def load_fold(k):
    t = json.load(open(dl(f"Train/fold{k}-text.json"), encoding="utf-8"))
    g = json.load(open(dl(f"Train/fold{k}-gold.json"), encoding="utf-8"))
    ents = {}
    for d in t:
        rec = g[d]; rec = rec["entities"] if isinstance(rec, dict) and "entities" in rec else rec
        out = []
        for e in rec:
            if isinstance(e, dict):
                out.append((int(e.get("start", e.get("start_offset"))), int(e.get("end", e.get("end_offset"))),
                            e.get("label", e.get("type")), e.get("text")))
            else:
                out.append((int(e[0]), int(e[1]), e[2], e[3] if len(e) > 3 else None))
        ents[str(d)] = out
    return {str(k_): v for k_, v in t.items()}, ents

text1, ent1 = load_fold(1); text2, ent2 = load_fold(2)
print(len(text1), len(text2), LABELS)


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

35 35 ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [20]:
def encode_with_offsets(text, tok):
    ids, offs = [], []
    for m in tok._split.finditer(text):
        w = m.group()
        if not w: continue
        pieces = [(bytes([b]), i, i + 1) for i, b in enumerate(w.encode("utf-8"))]
        b2c = []
        for ci, ch in enumerate(w): b2c += [ci] * len(ch.encode("utf-8"))
        while len(pieces) > 1:                       # same merge rule as BPETokenizer._word
            bi, br = -1, 1 << 60
            for i in range(len(pieces) - 1):
                r = tok.ranks.get((pieces[i][0], pieces[i + 1][0]), 1 << 60)
                if r < br: bi, br = i, r
            if bi < 0: break
            a, b = pieces[bi], pieces[bi + 1]
            pieces[bi:bi + 2] = [(a[0] + b[0], a[1], b[2])]
        for p, bs, be in pieces:
            s, e = m.start() + b2c[bs], m.start() + b2c[be - 1] + 1
            while s < e and text[s].isspace(): s += 1        # <-- the fix
            ids.append(tok.stoi[p]); offs.append((s, e))
    return ids, offs

def token_labels(entities, offs):
    y = [0] * len(offs); exact = 0
    for s, e, lab, _ in entities:
        idx = [i for i, (a, b) in enumerate(offs) if a < b and a < e and b > s]
        if not idx: continue
        if offs[idx[0]][0] == s and offs[idx[-1]][1] == e: exact += 1
        y[idx[0]] = l2i[f"B-{lab}"]
        for i in idx[1:]: y[i] = l2i[f"I-{lab}"]
    return y, exact

def build_docs(texts, ents):
    docs = {}
    for d, t in tqdm(texts.items(), desc="encode"):
        ids, offs = encode_with_offsets(t, tokenizer)
        y, ex = token_labels(ents[d], offs)
        docs[d] = dict(text=t, ids=ids, offs=offs, y=y, gold=[(s, e, l) for s, e, l, _ in ents[d]], exact=ex)
    return docs

train_docs, val_docs = build_docs(text1, ent1), build_docs(text2, ent2)
n_ent = sum(len(d["gold"]) for d in train_docs.values())
print("exact-boundary entities (train):", sum(d["exact"] for d in train_docs.values()), "/", n_ent)


encode:   0%|          | 0/35 [00:00<?, ?it/s]

encode:   0%|          | 0/35 [00:00<?, ?it/s]

exact-boundary entities (train): 4051 / 4120


In [21]:
def viterbi(logp):
    T = logp.shape[0]
    if T == 0: return []
    allow = np.zeros((NL, NL), bool)       # allow[prev, cur]
    for p in range(NL):
        for c in range(NL):
            n = BIO[c]
            allow[p, c] = not (n.startswith("I-") and not (BIO[p] != "O" and BIO[p][2:] == n[2:]))
    trans = np.where(allow, 0.0, -1e9)
    start = np.array([0.0 if not BIO[c].startswith("I-") else -1e9 for c in range(NL)])
    dp = start + logp[0]; back = np.zeros((T, NL), int)
    for t in range(1, T):
        sc = dp[:, None] + trans
        back[t] = sc.argmax(0); dp = sc.max(0) + logp[t]
    path = [int(dp.argmax())]
    for t in range(T - 1, 0, -1): path.append(int(back[t][path[-1]]))
    return path[::-1]

def tags_to_spans(tags, offs):
    spans, cur = [], None
    for t, (s, e) in zip(tags, offs):
        if s == e: continue                  # whitespace token: skip, never breaks a span
        n = BIO[t]
        if n == "O":
            if cur: spans.append(tuple(cur)); cur = None
        elif n.startswith("B-") or cur is None or cur[2] != n[2:]:
            if cur: spans.append(tuple(cur))
            cur = [s, e, n[2:]]
        else:
            cur[1] = e
    if cur: spans.append(tuple(cur))
    return spans

def strict_prf(gold, pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P: (tp if x in G else fp)[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    per = {}
    for l in LABELS:
        if tp[l] + fp[l] + fn[l] == 0: continue
        p = tp[l] / max(1, tp[l] + fp[l]); r = tp[l] / max(1, tp[l] + fn[l])
        per[l] = (p, r, 2 * p * r / max(1e-9, p + r), tp[l] + fn[l])
    m = lambda i: float(np.mean([v[i] for v in per.values()])) if per else 0.0
    return dict(P=m(0), R=m(1), F1=m(2), per=per)

def error_analysis(gold, pred):
    c = Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P:
            if x in G: c["exact"] += 1; continue
            if any((g[0], g[1]) == (x[0], x[1]) for g in G): c["label_confusion"] += 1
            elif any(g[2] == x[2] and g[0] < x[1] and x[0] < g[1] for g in G): c["boundary_error"] += 1
            else: c["false_positive"] += 1
        for g in G - P:
            if not any(p[0] < g[1] and g[0] < p[1] for p in P): c["missed(FN)"] += 1
    return dict(c)

def show(res, title=""):
    print(f"{title} macro P {res['P']:.4f} R {res['R']:.4f} F1 {res['F1']:.4f}")
    for l, (p, r, f, n) in res["per"].items(): print(f"   {l:10s} P {p:.3f} R {r:.3f} F1 {f:.3f} n={n}")

# Oracle: gold BIO tags -> spans.  Upper bound of what the tokenizer alignment allows.
gold_v = {d: x["gold"] for d, x in val_docs.items()}
oracle = {d: tags_to_spans(x["y"], x["offs"]) for d, x in val_docs.items()}
show(strict_prf(gold_v, oracle), "ORACLE (should be ~1.0)")

ORACLE (should be ~1.0) macro P 0.9849 R 0.9849 F1 0.9849
   APP        P 0.990 R 0.990 F1 0.990 n=200
   RESP       P 0.963 R 0.963 F1 0.963 n=190
   A.COUNSEL  P 0.992 R 0.992 F1 0.992 n=125
   R.COUNSEL  P 1.000 R 1.000 F1 1.000 n=97
   JUDGE      P 1.000 R 1.000 F1 1.000 n=122
   WIT        P 1.000 R 1.000 F1 1.000 n=120
   AUTH       P 0.991 R 0.991 F1 0.991 n=468
   COURT      P 0.978 R 0.978 F1 0.978 n=558
   STAT       P 0.976 R 0.976 F1 0.976 n=735
   PREC       P 0.928 R 0.928 F1 0.928 n=559
   DATE       P 1.000 R 1.000 F1 1.000 n=762
   CASENO     P 1.000 R 1.000 F1 1.000 n=337


In [22]:
BOS, PAD = 2, 0
def make_windows(n, win, step):
    if n <= win: return [(0, n)]
    st = list(range(0, n - win + 1, step))
    if st[-1] + win < n: st.append(n - win)
    return [(s, s + win) for s in st]

def make_items(docs, step=STRIDE):
    items = []
    for d, x in docs.items():
        for a, b in make_windows(len(x["ids"]), WIN, step):
            items.append(([BOS] + x["ids"][a:b], [-100] + x["y"][a:b]))
    return items

def collate(batch):
    L = max(len(i) for i, _ in batch)
    ids = torch.full((len(batch), L), PAD); y = torch.full((len(batch), L), -100); m = torch.zeros(len(batch), L, dtype=torch.bool)
    for k, (i, l) in enumerate(batch):
        ids[k, :len(i)] = torch.tensor(i); y[k, :len(l)] = torch.tensor(l); m[k, :len(i)] = True
    return ids, y, m

# _ST = {"causal": CAUSAL, "mask": None}
# if not hasattr(MLA, "_orig_forward"): MLA._orig_forward = MLA.forward
# def _mla_forward(self, x, mask=None, cache=None, start=0):
#     if cache is not None: return MLA._orig_forward(self, x, mask, cache, start)
#     b, t, _ = x.shape
#     q = self.uq(self.q_norm(self.dq(x))).reshape(b, t, self.h, self.dn + self.dr).transpose(1, 2)
#     q_nope, q_rope = q.split([self.dn, self.dr], -1)
#     ckv, k_rope = self.dkv(x).split([self.dc, self.dr], -1); ckv = self.kv_norm(ckv)
#     q_rope = self.rope(q_rope, start); k_rope = self.rope(k_rope.unsqueeze(1), start)
#     k_nope = self.uk(ckv).reshape(b, t, self.h, self.dn).transpose(1, 2)
#     v = self.uv(ckv).reshape(b, t, self.h, self.dv).transpose(1, 2)
#     qf = torch.cat([q_nope, q_rope], -1); kf = torch.cat([k_nope, k_rope.expand(-1, self.h, -1, -1)], -1)
#     if _ST["causal"] or _ST["mask"] is None: y = F.scaled_dot_product_attention(qf, kf, v, is_causal=True)
#     else: y = F.scaled_dot_product_attention(qf, kf, v, attn_mask=_ST["mask"])   # bidirectional + padding mask
#     return self.o(y.transpose(1, 2).reshape(b, t, self.h * self.dv)), None
# MLA.forward = _mla_forward


class Adapter(nn.Module):
    def __init__(self, d, r):
        super().__init__()
        self.down = nn.Linear(d, r)
        self.up = nn.Linear(r, d)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


class Backbone(nn.Module):
    """
    Backbone for the supplied RoPE LegalLM:
      - lm.wte: token embedding
      - lm.blocks: Transformer blocks returning (x, cache)
      - lm.ln_f: final RMSNorm
    """
    def __init__(self, lm, causal=False, adapter_dim=0):
        super().__init__()
        self.lm = lm
        self.causal = causal
        self.dim = lm.wte.embedding_dim
        self.adapters = (
            nn.ModuleList([
                Adapter(self.dim, adapter_dim)
                for _ in lm.blocks
            ])
            if adapter_dim > 0 else None
        )

    def forward(self, ids, attn):
        b, t = ids.shape
        valid = attn.bool()

        # SDPA boolean masks use True for positions that may be attended to.
        # Shape: (batch, 1, query_length, key_length).
        key_mask = valid[:, None, None, :]

        if self.causal:
            causal_mask = torch.ones(
                (t, t), dtype=torch.bool, device=ids.device
            ).tril()
            attention_mask = key_mask & causal_mask[None, None, :, :]
        else:
            # Bidirectional attention, excluding padding keys.
            attention_mask = key_mask

        x = self.lm.wte(ids)

        for i, blk in enumerate(self.lm.blocks):
            x, _ = blk(
                x,
                mask=attention_mask,
                cache=None,
                start=0
            )
            if self.adapters is not None:
                x = self.adapters[i](x)

        return self.lm.ln_f(x)



class NERModel(nn.Module):
    def __init__(self, bb, head="linear", drop=0.1):
        super().__init__()
        self.backbone = bb
        self.drop = nn.Dropout(drop)
        d = bb.dim

        if head == "linear":
            self.head = nn.Linear(d, NL)
        elif head == "mlp":
            self.head = nn.Sequential(
                nn.Linear(d, d),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(d, NL)
            )
        else:
            raise ValueError(f"Unknown NER head: {head}")

    def forward(self, ids, attn):
        hidden = self.backbone(ids, attn)
        return self.head(self.drop(hidden))


In [23]:
class LoRALinear(nn.Module):             # y = Wx + s * B A x
    def __init__(s, base, r=8, alpha=16):
        super().__init__(); s.base, s.s = base, alpha / r
        s.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(s, x): return s.base(x) + (x @ s.A.t() @ s.B.t()) * s.s

class LoRMALinear(nn.Module):            # W' = (I + s*B A) W  ->  y = Wx + s * B A (Wx)   (multiplicative, output side)
    def __init__(s, base, r=8, alpha=16):
        super().__init__(); s.base, s.s = base, alpha / r
        s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(s, x): y = s.base(x); return y + (y @ s.A.t() @ s.B.t()) * s.s

In [24]:

def load_lm(path=None):
    # ck = torch.load(path or MODEL_PATH, map_location="cpu", weights_only=False)
    # lm = LegalLM(ck["config"])
    # state = {k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}
    # lm.load_state_dict(state)
    # return lm
    if os.path.exists(MODEL_PATH):
        path = MODEL_PATH
    else:
        path = "/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt"
    print(path)
    ck = torch.load(path, map_location=DEVICE, weights_only=False); c = ck["config"]
    m = LegalLM(c["vocab_size"], c["dim"], c["layers"], c["heads"], c["block_size"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()


TARGETS = {"qkv", "proj", "w", "w13", "w2"}  # Must match your actual LegalLM layer names

def inject(lm, cls, r=8, alpha=16):
    count = 0
    for blk in lm.blocks:
        for path, mod in list(blk.named_modules()):
            if "experts" in path.split("."):
                continue
            for name, child in list(mod.named_children()):
                if isinstance(child, nn.Linear) and name in TARGETS:
                    setattr(mod, name, cls(child, r=r, alpha=alpha))
                    count += 1
    if count == 0:
        raise RuntimeError(
            f"No layers matched TARGETS={TARGETS}. "
            "Inspect the Linear layer names in lm.blocks."
        )
    print(f"Injected {cls.__name__} into {count} Linear layers.")
    return count


def build(kind, head="linear", r=8, k_top=3, path=None):
    valid = {"full", "lora", "lorma", "adapter", "partial"}
    if kind not in valid:
        raise ValueError(f"Unknown kind {kind!r}; expected one of {sorted(valid)}")

    lm = load_lm(path)

    if kind == "full":
        # Train the entire language model.
        for p in lm.parameters():
            p.requires_grad = True

    else:
        # Freeze the pretrained language model first.
        for p in lm.parameters():
            p.requires_grad = False

        if kind in {"lora", "lorma"}:
            cls = LoRALinear if kind == "lora" else LoRMALinear
            inject(lm, cls, r=r, alpha=2 * r)

        elif kind == "partial":
            n = min(max(k_top, 0), len(lm.blocks))
            for blk in lm.blocks[len(lm.blocks) - n:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in lm.ln_f.parameters():
                p.requires_grad = True

    bb = Backbone(
        lm,
        causal=CAUSAL,
        adapter_dim=64 if kind == "adapter" else 0
    )
    model = NERModel(bb, head=head).to(DEVICE)

    total, trainable = n_params(model)
    print(
        f"{kind}: {total:,} total parameters, "
        f"{trainable:,} trainable ({100 * trainable / max(total, 1):.2f}%)"
    )
    return model



def n_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


In [25]:
@torch.no_grad()
def predict_docs(model, docs, decode="viterbi", bs=8):
    model.eval(); out = {}
    for d, x in docs.items():
        n = len(x["ids"]); acc = np.zeros((n, NL), np.float32); cnt = np.zeros(n, np.float32)
        if n == 0:
            out[d] = []
            continue
        wins = make_windows(n, WIN, WIN // 2)
        for i in range(0, len(wins), bs):
            chunk = wins[i:i + bs]
            ids, _, m = collate([([BOS] + x["ids"][a:b], [0] * (b - a + 1)) for a, b in chunk])
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg = model(ids.to(DEVICE), m.to(DEVICE))
            lp = lg.float().log_softmax(-1).cpu().numpy()
            for k, (a, b) in enumerate(chunk): acc[a:b] += lp[k, 1:b - a + 1]; cnt[a:b] += 1
        lp = acc / np.maximum(cnt, 1)[:, None]
        tags = viterbi(lp) if decode == "viterbi" else lp.argmax(-1).tolist()
        out[d] = tags_to_spans(tags, x["offs"])
    return out

def evaluate(model, docs, decode="viterbi"):
    pred = predict_docs(model, docs, decode); return strict_prf({d: x["gold"] for d, x in docs.items()}, pred), pred

In [26]:
from tqdm.auto import tqdm
def class_weights(docs, power=0.5, cap=10.0):
    c = np.bincount(np.concatenate([x["y"] for x in docs.values()]), minlength=NL).astype(float) + 1
    w = np.clip((c[0] / c) ** power, 0, cap); w[0] = 1.0
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train(model, name, epochs, lr, head_lr=1e-3, bs=8, wd=0.01, cw=None, teacher=None, T=2.0, alpha=0.5, warm=0.1, snapshot_eval=True, resume_path=None):
    # ADD: Checkpoint directory creation
    ckpt_dir = f"{OUT}/checkpoints/{name}"
    print(ckpt_dir)
    os.makedirs(ckpt_dir, exist_ok=True)
    items = make_items(train_docs); steps = epochs * math.ceil(len(items) / bs)
    bb_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head")]
    hd_p = [p for n, p in model.named_parameters() if n.startswith("head")]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": lr}, {"params": hd_p, "lr": head_lr}], weight_decay=wd)
    base = [lr, head_lr]; scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time(); hist = []
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time()
    hist = []
    start_epoch = 0  # ADD: Track starting epoch
    print(resume_path)

    # ADD: Load checkpoint if resume_path is provided
    if resume_path and os.path.exists(resume_path):
        print(f"--> Resuming from: {resume_path}")
        ckpt = torch.load(resume_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"], strict=False)
        opt.load_state_dict(ckpt["optimizer_state"])
        if USE_AMP and AMP_DTYPE == torch.float16 and "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        step = ckpt["step"]
        best = ckpt["best_f1"]
        best_state = ckpt["best_state"]
        hist = ckpt.get("hist", [])
        print(start_epoch)
        model.load_state_dict(best_state, strict=False)
        peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else float("nan")
        return dict(best_val_f1=best, train_sec=time.time() - t0, peak_gb=peak, hist=hist)
    for ep in tqdm(range(start_epoch, epochs)):
        model.train(); random.shuffle(items); tl = 0
        for i in range(0, len(items), bs):
            f = (step + 1) / (warm * steps) if step < warm * steps else 0.5 * (1 + math.cos(math.pi * (step - warm * steps) / max(1, steps - warm * steps)))
            for g, b0 in zip(opt.param_groups, base): g["lr"] = b0 * f
            ids, y, m = [t.to(DEVICE) for t in collate(items[i:i + bs])]
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg = model(ids, m)
                loss = F.cross_entropy(lg.float().reshape(-1, NL), y.reshape(-1), weight=cw, ignore_index=-100)
                if teacher is not None:
                    with torch.no_grad(): tl_ = teacher(ids, m).float()
                    sel = y != -100
                    kd = F.kl_div((lg.float()[sel] / T).log_softmax(-1), (tl_[sel] / T).softmax(-1), reduction="batchmean") * T * T
                    loss = alpha * loss + (1 - alpha) * kd
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1; tl += loss.item()
        res, _ = evaluate(model, val_docs); hist.append((ep + 1, tl / math.ceil(len(items) / bs), res["F1"]))
        print(f"[{name}] ep {ep+1}/{epochs} loss {hist[-1][1]:.4f} | val strict macro P {res['P']:.3f} R {res['R']:.3f} F1 {res['F1']:.4f}")
        if res["F1"] > best: best = res["F1"]; best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
        # inside epoch loop, after calculating `res`:
        if res["F1"] > best:
            best = res["F1"]
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
            # ADD: Save best weights separately
            torch.save(best_state, f"{ckpt_dir}/{name}_best_weights.pt")

        # ADD: Save latest checkpoint for resuming
        ckpt_payload = {
            "epoch": ep,
            "step": step,
            "model_state": {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
            "optimizer_state": opt.state_dict(),
            "scaler_state": scaler.state_dict() if USE_AMP else None,
            "best_f1": best,
            "best_state": best_state,
            "hist": hist,
        }
        torch.save(ckpt_payload, f"{ckpt_dir}/latest_checkpoint.pt")
    model.load_state_dict(best_state, strict=False)
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else float("nan")
    return dict(best_val_f1=best, train_sec=time.time() - t0, peak_gb=peak, hist=hist)

In [27]:
test_texts = json.load(open(dl("Test/fold3-text.json"), encoding="utf-8")); test_texts = {str(k): v for k, v in test_texts.items()}
test_docs = {}
for d, t in test_texts.items():
    ids, offs = encode_with_offsets(t, tokenizer); test_docs[d] = dict(text=t, ids=ids, offs=offs, y=[0] * len(ids), gold=[])

def write_test(model, name):
    pred = predict_docs(model, test_docs)
    js = {d: [[s, e, l, test_texts[d][s:e]] for s, e, l in sp] for d, sp in pred.items()}
    p = f"{OUT}/prediction_{name}.json"; json.dump(js, open(p, "w"), ensure_ascii=False)
    with zipfile.ZipFile(f"{OUT}/submission_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(p, "prediction.json")

CW = class_weights(train_docs)
RESULTS, MODELS = {}, {}
def run(name, kind, epochs, lr, head_lr=1e-3, head="linear", cw=CW, resume = False, **kw):
    model = build(kind, head=head, **kw); tot, tr = n_params(model)
    print(f"\n=== {name}: total {tot/1e6:.1f}M params, trainable {tr/1e6:.2f}M ({100*tr/tot:.1f}%)")
    ckpt_path = f"{OUT}/{name}/latest_checkpoint.pt"
    resume_path = ckpt_path if (resume and os.path.exists(ckpt_path)) else None

    # CHANGE: Pass resume_path to train()
    info = train(model, name, epochs, lr, head_lr, cw=cw, resume_path=resume_path)
    t0 = time.time(); res, pred = evaluate(model, val_docs); info["infer_sec"] = time.time() - t0
    info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
    RESULTS[name] = info; MODELS[name] = model; write_test(model, name); show(res, f"[{name}] FINAL VAL"); return model

# Exp 1: full fine-tuning (also the KD teacher)
run("full_ft", "full", epochs=4, lr=5e-5, resume = True)
# Exp 2: partial / layer-wise (top-3 blocks)
# run("partial_top3", "partial", epochs=10, lr=1e-4, k_top=3, resume = True)
# Exp 3: LoRA
run("lora_r8", "lora", epochs=4, lr=1e-3, r=8, resume = True)
# Exp 4: LoRMA
run("lorma_r8", "lorma", epochs=4, lr=1e-3, r=8, resume = True)
# Exp 5: Houlsby-style adapters
# run("adapter64", "adapter", epochs=15, lr=1e-3, resume = True)

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt
full: 75,087,385 total parameters, 75,087,385 trainable (100.00%)

=== full_ft: total 75.1M params, trainable 75.09M (100.0%)
/kaggle/working/phase2-custom-mha/checkpoints/full_ft
/kaggle/working/phase2-custom-mha/full_ft/latest_checkpoint.pt
--> Resuming from: /kaggle/working/phase2-custom-mha/full_ft/latest_checkpoint.pt
8
[full_ft] FINAL VAL macro P 0.4528 R 0.4325 F1 0.4324
   APP        P 0.223 R 0.125 F1 0.160 n=200
   RESP       P 0.207 R 0.121 F1 0.153 n=190
   A.COUNSEL  P 0.462 R 0.336 F1 0.389 n=125
   R.COUNSEL  P 0.346 R 0.464 F1 0.396 n=97
   JUDGE      P 0.497 R 0.598 F1 0.543 n=122
   WIT        P 0.320 R 0.133 F1 0.188 n=120
   AUTH       P 0.366 R 0.199 F1 0.258 n=468
   COURT      P 0.766 R 0.849 F1 0.805 n=558
   STAT       P 0.665 R 0.623 F1 0.643 n=735
   PREC       P 0.267 R 0.358 F1 0.306 n=559
   DATE       P 0.900 R 0.920 F1 0.910 n=762
   CASENO     P 0.415 R 0.463 F1 0.438 n=337
/kaggle/working/cs781_phase1

NERModel(
  (backbone): Backbone(
    (lm): LegalLM(
      (wte): Embedding(24000, 768)
      (blocks): ModuleList(
        (0-7): 8 x Block(
          (n1): RMSNorm()
          (n2): RMSNorm()
          (attn): Attention(
            (qkv): LoRMALinear(
              (base): Linear(in_features=768, out_features=2304, bias=False)
            )
            (proj): LoRMALinear(
              (base): Linear(in_features=768, out_features=768, bias=False)
            )
            (rope): RoPE()
          )
          (mlp): SwiGLU(
            (w13): LoRMALinear(
              (base): Linear(in_features=768, out_features=4096, bias=False)
            )
            (w2): LoRMALinear(
              (base): Linear(in_features=2048, out_features=768, bias=False)
            )
          )
        )
      )
      (ln_f): RMSNorm()
      (lm_head): Linear(in_features=768, out_features=24000, bias=False)
    )
  )
  (drop): Dropout(p=0.1, inplace=False)
  (head): Linear(in_features=768, out_feature

In [28]:
teacher = MODELS["full_ft"].eval()
for p in teacher.parameters(): p.requires_grad = False
p1 = load_lm(); keep = [0, 2, 4, 7];
slm = LegalLM(vocab_size=24000); slm.wte.load_state_dict(p1.wte.state_dict()); slm.ln_f.load_state_dict(p1.ln_f.state_dict())
for i, j in enumerate(keep): slm.blocks[i].load_state_dict(p1.blocks[j].state_dict())   # init from Phase-1 weights
student = NERModel(Backbone(slm, CAUSAL)).to(DEVICE); tot, tr = n_params(student)
print(f"=== KD student: {tot/1e6:.1f}M params (teacher {RESULTS['full_ft']['total_M']:.1f}M)")
info = train(student, "kd_student", epochs=4, lr=1e-4, cw=CW, teacher=teacher, T=2.0, alpha=0.5, resume_path='/kaggle/working/phase2-custom-mha/kd_student/latest_checkpoint.pt')
t0 = time.time(); res, pred = evaluate(student, val_docs); info["infer_sec"] = time.time() - t0
info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
RESULTS["kd_student"] = info; MODELS["kd_student"] = student; write_test(student, "kd_student"); show(res, "[kd_student] FINAL VAL")
# Control: same student WITHOUT distillation (does KD actually help?)
student0 = NERModel(Backbone(copy.deepcopy(slm), CAUSAL)).to(DEVICE)
i0 = train(student0, "student_noKD", 10, 1e-4, cw=CW, resume_path='/kaggle/working/phase2-custom-mha/student_noKD/latest_checkpoint.pt')
RESULTS["student_noKD"] = dict(best_val_f1=i0["best_val_f1"], F1=i0["best_val_f1"], total_M=tot / 1e6, trainable_M=tr / 1e6, train_sec=i0["train_sec"], peak_gb=i0["peak_gb"])

/kaggle/working/cs781_phase1/legal_lm_custom_mha.pt
=== KD student: 75.1M params (teacher 75.1M)
/kaggle/working/phase2-custom-mha/checkpoints/kd_student
/kaggle/working/phase2-custom-mha/kd_student/latest_checkpoint.pt
--> Resuming from: /kaggle/working/phase2-custom-mha/kd_student/latest_checkpoint.pt
10
[kd_student] FINAL VAL macro P 0.4345 R 0.4295 F1 0.4226
   APP        P 0.182 R 0.110 F1 0.137 n=200
   RESP       P 0.256 R 0.111 F1 0.154 n=190
   A.COUNSEL  P 0.400 R 0.368 F1 0.383 n=125
   R.COUNSEL  P 0.273 R 0.392 F1 0.322 n=97
   JUDGE      P 0.451 R 0.607 F1 0.517 n=122
   WIT        P 0.234 R 0.125 F1 0.163 n=120
   AUTH       P 0.424 R 0.256 F1 0.320 n=468
   COURT      P 0.774 R 0.855 F1 0.813 n=558
   STAT       P 0.662 R 0.612 F1 0.636 n=735
   PREC       P 0.237 R 0.354 F1 0.284 n=559
   DATE       P 0.888 R 0.923 F1 0.905 n=762
   CASENO     P 0.433 R 0.442 F1 0.438 n=337
/kaggle/working/phase2-custom-mha/checkpoints/student_noKD
/kaggle/working/phase2-custom-mha/stu

---
## Architecture: `mla_moe`



In [29]:
# ── Activate backbone: mla_moe ──────────────────────────────────────────────
BACKBONE = "mla_moe"
TOKENIZER_PATH = TOK_PATHS["mla_moe"]
PHASE1_CKPT = CKPTS["mla_moe"]
MODEL_PATH = PHASE1_CKPT
CKPT_PATH = PHASE1_CKPT
if "OUT_DIR" not in dir() or not OUT_DIR:
    OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"[active] BACKBONE={BACKBONE}")
print(f"[active] TOKENIZER_PATH={TOKENIZER_PATH}  exists={os.path.exists(TOKENIZER_PATH)}")
print(f"[active] PHASE1_CKPT={PHASE1_CKPT}  exists={os.path.exists(PHASE1_CKPT)}")

# generation
TARGET_WORDS, MAX_NEW_TOKENS, GEN_BATCH = 128, 320, 16
TEMPERATURE, TOP_P, TOP_K, REP_PENALTY = 0.7, 0.9, 50, 1.15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BLOCK_SIZE, N_EMBD, N_LAYER, N_HEAD = 1024, 768, 8, 12
BATCH_SIZE, GRAD_ACCUM = 8, 8    # 16*4*1024 = 65k tokens / optimizer step
TRAIN_TOKENS = 150_000_000        # total tokens seen in training (several passes over small sources is fine)
LR, MIN_LR, WEIGHT_DECAY, WARMUP_STEPS = 6e-4, 6e-5, 0.1, 100
EVAL_EVERY, EVAL_WINDOWS, LOG_EVERY, CKPT_EVERY = 250, 48, 10, 250
MAX_TRAIN_HOURS = 8.0             # schedule is shortened automatically if throughput is too low
USE_GRAD_CKPT = True             # set True only if you hit OOM (about 30% slower)
USE_COMPILE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

[active] BACKBONE=mla_moe
[active] TOKENIZER_PATH=/kaggle/working/cs781_phase1/legal_bpe_mla_moe.json  exists=True
[active] PHASE1_CKPT=/kaggle/working/cs781_phase1/legal_lm_mla_moe.pt  exists=True


### Phase-1 classes for `mla_moe` (tokenizer + model, needed to reload checkpoints)

These cells are taken from Phase 1 of the same source notebook. They define `BPETokenizer`,
the model stack (`LegalLM` / `DualPathLM` / …), and `load_model`. Internal bodies are unchanged.


## 1. Text cleaning + byte-level BPE tokenizer

In [30]:
import os, sys, re, gc, json, math, time, heapq, random, hashlib, unicodedata, zipfile, subprocess
import multiprocessing as mp
from collections import Counter, defaultdict, deque
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint
from tqdm.auto import tqdm
_QUOTES = str.maketrans({"“": '"', "”": '"', "„": '"', "‘": "'", "’": "'", "‚": "'", "`": "'", "´": "'",
                         "–": "-", "—": "-", "−": "-", "\u00ad": ""})
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
_WS = re.compile(r"[ \t\u00a0\u1680\u2000-\u200b\u202f\u205f\u3000\ufeff]+")
_DOTS = re.compile(r"\.{4,}|(?:\. ){4,}")
_RULE = re.compile(r"[_\-=*~#]{3,}")
_PAGE = re.compile(r"(?mi)^[ \t]*(?:page\s+)?\d{1,4}[ \t]*$\n?")
_HYPH = re.compile(r"(?<=[a-z])-\n(?=[a-z])")
_WRAP = re.compile(r"(?<=[a-z,;])\n(?=[a-z(])")

def clean_text(t):
    """Light, conservative normalisation of legal text (used for training data AND test prompts)."""
    t = unicodedata.normalize("NFKC", str(t)).replace("\r\n", "\n").replace("\r", "\n")
    t = t.translate(_QUOTES)
    t = _CTRL.sub(" ", t)
    t = _HYPH.sub("", t)                 # de-hyphenate line-wrapped words
    t = _DOTS.sub("...", t)              # ".........." table-of-contents leaders
    t = _RULE.sub(" ", t)                # ------ / ====== separators
    t = _PAGE.sub("", t)                 # lines that are only a page number
    t = _WS.sub(" ", t)
    t = re.sub(r" *\n *", "\n", t)
    t = _WRAP.sub(" ", t)                # join hard-wrapped lines inside a sentence
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def keep_doc(c):
    """Quality filter for cleaned documents."""
    if len(c) < 600: return False
    head = c[:5000]
    if sum(ch.isalpha() for ch in head) / len(head) < 0.6: return False
    if sum(ord(ch) > 127 for ch in head) / len(head) > 0.2: return False   # mostly non-English OCR/Indic
    w = c.split()
    if len(w) < 100: return False
    w = w[:3000]
    return len(set(w)) / len(w) >= 0.15                                       # drop repetitive junk

def extract_text(rec):
    if isinstance(rec, str): return rec
    if isinstance(rec, dict):
        for k in ("text", "train_text", "content", "body", "judgment", "document"):
            v = rec.get(k)
            if isinstance(v, str) and v.strip(): return v
        for v in rec.values():
            if isinstance(v, (dict, list)):
                s = extract_text(v)
                if s: return s
    if isinstance(rec, list):
        return "\n".join(filter(None, map(extract_text, rec)))
    return ""

def iter_docs(path):
    """Constant-memory iteration over a huge JSON file ({id: record} or [record, ...])."""
    with open(path, "rb") as f:
        head = f.read(256).lstrip(b"\xef\xbb\xbf \n\r\t")[:1]
        f.seek(0)
        it = (v for _, v in ijson.kvitems(f, "")) if head == b"{" else ijson.items(f, "item")
        for rec in it:
            t = extract_text(rec)
            if t: yield t


class BPETokenizer:
    SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
    # GPT-2 style: a leading space stays attached to the word, so "the" / " the" are one token each
    _split = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+")

    def __init__(self, vocab_size=12000):
        self.vocab_size = vocab_size
        self.merges = []
        self._reset()

    def _reset(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        self._cache = {}

    def _build(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        seen = set(self.itos[4:])
        for a, b in self.merges:
            m = a + b
            if m not in seen:
                seen.add(m); self.itos.append(m)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.ranks = {pair: r for r, pair in enumerate(self.merges)}
        self.merged = {pair: pair[0] + pair[1] for pair in self.merges}
        self._cache = {}
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        self.vocab_size = len(self.itos)

    def train(self, texts, max_unique_words=200_000, min_freq=2):
        counts = Counter()
        for t in tqdm(texts, desc="counting pre-tokens"):
            counts.update(self._split.findall(t))
        if len(counts) > max_unique_words:
            counts = Counter(dict(counts.most_common(max_unique_words)))
        words, freqs = [], []
        pair_counts, pair_words = defaultdict(int), defaultdict(set)
        for wid, (w, c) in enumerate(counts.items()):
            toks = [bytes([b]) for b in w.encode("utf-8")]
            words.append(toks); freqs.append(c)
            for p in zip(toks, toks[1:]):
                pair_counts[p] += c; pair_words[p].add(wid)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)
        self.merges = []
        n_target = self.vocab_size - (4 + 256)
        seen = set(bytes([i]) for i in range(256))
        pbar = tqdm(total=n_target, desc="BPE merges")
        while len(self.merges) < n_target and heap:
            best = None
            while heap:                                  # lazy-deletion max-heap
                negc, p = heapq.heappop(heap)
                cur = pair_counts.get(p, 0)
                if cur <= 0: continue
                if cur != -negc:
                    heapq.heappush(heap, (-cur, p)); continue
                best = p; break
            if best is None or pair_counts[best] < min_freq: break
            a, b = best; merged = a + b
            self.merges.append(best); seen.add(merged)
            for wid in list(pair_words.pop(best, ())):
                w, f = words[wid], freqs[wid]
                new, i = [], 0
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        new.append(merged); i += 2
                    else:
                        new.append(w[i]); i += 1
                if len(new) == len(w): continue
                for p in zip(w, w[1:]): pair_counts[p] -= f
                for p in zip(new, new[1:]):
                    pair_counts[p] += f; pair_words[p].add(wid)
                    heapq.heappush(heap, (-pair_counts[p], p))
                words[wid] = new
            pbar.update(1)
        pbar.close()
        self._build()
        return self

    def _word(self, w):
        r = self._cache.get(w)
        if r is not None: return r
        toks = [bytes([b]) for b in w.encode("utf-8")]
        while len(toks) > 1:
            best_i, best_r = -1, 1 << 60
            for i in range(len(toks) - 1):
                r_ = self.ranks.get((toks[i], toks[i + 1]), 1 << 60)
                if r_ < best_r: best_i, best_r = i, r_
            if best_i < 0: break
            toks[best_i:best_i + 2] = [toks[best_i] + toks[best_i + 1]]
        r = [self.stoi[t] for t in toks]
        if len(self._cache) > 500_000: self._cache.clear()
        self._cache[w] = r
        return r

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.bos_id] if add_bos else []
        for ch in self._split.findall(text): ids.extend(self._word(ch))
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return b"".join(self.itos[i] for i in ids if 4 <= i < len(self.itos)).decode("utf-8", errors="replace")

    def save(self, path):
        with open(path, "w") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": [[a.hex(), b.hex()] for a, b in self.merges]}, f)

    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        t = cls(d["vocab_size"]); t.merges = [(bytes.fromhex(a), bytes.fromhex(b)) for a, b in d["merges"]]
        t._build(); return t

## 4. Model: MLA attention (with weight absorption at inference) + DeepSeekMoE feed-forward
* **MLA**: keys/values are compressed into a small latent `c_kv` (+ a tiny decoupled RoPE key). Training uses the
  expanded form; inference uses the *absorbed* form: `W_UK` is folded into the query and `W_UV` into the output,
  so attention runs directly on the cached latent (cache = d_c + d_rope numbers per token instead of 2*heads*head_dim).
* **MoE**: 1 shared expert + routed experts (top-k). Experts are split into one group per data source; during training the
  router gets a small supervised "domain" loss so each group specialises (Acts / Supreme Court / High Courts / Tribunals).
  At inference there are no labels - the router infers the domain from the prompt text.

In [31]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.w).type_as(x)

class RoPE(nn.Module):
    def __init__(self, hd, max_len, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, hd, 2).float() / hd))
        fr = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", fr.cos(), persistent=False)
        self.register_buffer("sin", fr.sin(), persistent=False)
    def forward(self, x, start=0):                      # x: (b, heads, t, hd)
        t = x.size(2)
        cos, sin = self.cos[start:start + t].to(x.dtype), self.sin[start:start + t].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((x1 * cos - x2 * sin, x2 * cos + x1 * sin), dim=-1)


class MLA(nn.Module):
    """Multi-head Latent Attention (DeepSeek-V2/V3) with decoupled RoPE and weight absorption for inference."""
    def __init__(self, c):
        super().__init__()
        dim, H = c["dim"], c["heads"]
        self.h, self.dc, self.dn, self.dr, self.dv = H, c["d_c"], c["d_nope"], c["d_rope"], c["d_v"]
        self.dq = nn.Linear(dim, c["d_cq"], bias=False);  self.q_norm = RMSNorm(c["d_cq"])
        self.uq = nn.Linear(c["d_cq"], H * (self.dn + self.dr), bias=False)           # -> [q_nope | q_rope] per head
        self.dkv = nn.Linear(dim, self.dc + self.dr, bias=False); self.kv_norm = RMSNorm(self.dc)   # -> [c_kv | k_rope]
        self.uk = nn.Linear(self.dc, H * self.dn, bias=False)                          # W_UK
        self.uv = nn.Linear(self.dc, H * self.dv, bias=False)                          # W_UV
        self.o = nn.Linear(H * self.dv, dim, bias=False)
        self.rope = RoPE(self.dr, c["block_size"])
        self.scale = 1.0 / math.sqrt(self.dn + self.dr)

    def forward(self, x, mask=None, cache=None, start=0):
        b, t, _ = x.shape
        q = self.uq(self.q_norm(self.dq(x))).reshape(b, t, self.h, self.dn + self.dr).transpose(1, 2)
        q_nope, q_rope = q.split([self.dn, self.dr], dim=-1)
        ckv, k_rope = self.dkv(x).split([self.dc, self.dr], dim=-1)
        ckv = self.kv_norm(ckv)                                                       # (b, t, dc)  <- this is what is cached
        q_rope = self.rope(q_rope, start)
        k_rope = self.rope(k_rope.unsqueeze(1), start)                                # (b, 1, t, dr) shared by all heads

        if cache is None:                                                             # ---- training: expanded form
            k_nope = self.uk(ckv).reshape(b, t, self.h, self.dn).transpose(1, 2)
            v = self.uv(ckv).reshape(b, t, self.h, self.dv).transpose(1, 2)
            qf = torch.cat([q_nope, q_rope], -1)
            kf = torch.cat([k_nope, k_rope.expand(-1, self.h, -1, -1)], -1)
            y = F.scaled_dot_product_attention(qf, kf, v, is_causal=True)             # default scale = 1/sqrt(dn+dr)
            return self.o(y.transpose(1, 2).reshape(b, t, self.h * self.dv)), None

        # ---- inference: ABSORBED form, attention runs in latent space on the compressed cache
        r = k_rope.squeeze(1)
        if cache[0] is not None:
            ckv, r = torch.cat([cache[0], ckv], 1), torch.cat([cache[1], r], 1)
        new_cache = (ckv, r)
        Wuk = self.uk.weight.reshape(self.h, self.dn, self.dc)
        q_abs = torch.einsum("bhtn,hnc->bhtc", q_nope, Wuk.to(q_nope.dtype))          # q_nope @ W_UK  (absorb W_UK into q)
        qq = torch.cat([q_abs, q_rope], -1)                                           # (b, H, t, dc+dr)
        kk = torch.cat([ckv, r], -1).to(qq.dtype)[:, None].expand(-1, self.h, -1, -1) # (b, H, T, dc+dr)  (MQA-like, shared)
        vv = ckv.to(qq.dtype)[:, None].expand(-1, self.h, -1, -1)                     # values ARE the latent
        if mask is not None:  ctx = F.scaled_dot_product_attention(qq, kk, vv, attn_mask=mask, scale=self.scale)
        elif t > 1:           ctx = F.scaled_dot_product_attention(qq, kk, vv, is_causal=True, scale=self.scale)
        else:                 ctx = F.scaled_dot_product_attention(qq, kk, vv, scale=self.scale)
        Wuv = self.uv.weight.reshape(self.h, self.dv, self.dc)
        out = torch.einsum("bhtc,hvc->bthv", ctx, Wuv.to(ctx.dtype)).reshape(b, t, self.h * self.dv)   # absorb W_UV after attention
        return self.o(out), new_cache


class FFN(nn.Module):
    def __init__(self, dim, hid):
        super().__init__()
        self.w13 = nn.Linear(dim, 2 * hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
    def forward(self, x):
        a, b = self.w13(x).chunk(2, dim=-1)
        return self.w2(F.silu(a) * b)


class MoE(nn.Module):
    """DeepSeekMoE-style: shared expert(s) + top-k routed experts. Routed experts are grouped per domain."""
    def __init__(self, c):
        super().__init__()
        dim, self.n, self.k, self.D = c["dim"], c["n_routed"], c["topk_experts"], c["n_domains"]
        assert self.n % self.D == 0, "n_routed must be a multiple of n_domains"
        self.router = nn.Linear(dim, self.n, bias=False)
        self.experts = nn.ModuleList([FFN(dim, c["expert_hid"]) for _ in range(self.n)])
        self.shared = FFN(dim, c["shared_hid"])

    def forward(self, x, domain=None):
        b, t, d = x.shape
        xf = x.reshape(-1, d); N = xf.size(0)
        probs = self.router(xf).float().softmax(-1)                                   # (N, n_routed)
        topv, topi = probs.topk(self.k, dim=-1)
        topv = topv / topv.sum(-1, keepdim=True)
        flat_e = topi.reshape(-1)
        order = flat_e.argsort()
        tok = order // self.k                                                         # token index for each (sorted) assignment
        w = topv.reshape(-1)[order]
        counts = torch.bincount(flat_e, minlength=self.n)
        xs, outs, pos = xf[tok], [], 0
        for e, cnt in enumerate(counts.tolist()):
            if cnt:
                outs.append(self.experts[e](xs[pos:pos + cnt])); pos += cnt
        ys = torch.cat(outs, 0)
        routed = torch.zeros(N, d, dtype=ys.dtype, device=x.device).index_add_(0, tok, ys * w[:, None].to(ys.dtype))
        y = (self.shared(xf) + routed).reshape(b, t, d)

        z = probs.new_zeros(())
        bal, dom, acc = z, z, z
        if self.training:                                                             # switch-style load balancing
            f = counts.float() / (N * self.k)
            bal = self.n * (f * probs.mean(0)).sum()
        if domain is not None:                                                        # domain-guided routing
            g = probs.reshape(N, self.D, self.n // self.D).sum(-1)                       # router mass per expert group
            dt = domain[:, None].expand(b, t).reshape(-1)
            dom = -g.gather(1, dt[:, None]).clamp_min(1e-9).log().mean()
            acc = (g.argmax(-1) == dt).float().mean()
        return y, torch.stack([bal, dom, acc])


class Block(nn.Module):
    def __init__(self, c, dense):
        super().__init__()
        self.n1, self.n2, self.dense = RMSNorm(c["dim"]), RMSNorm(c["dim"]), dense
        self.attn = MLA(c)
        self.ffn = FFN(c["dim"], c["dense_hid"]) if dense else MoE(c)
    def forward(self, x, mask=None, cache=None, start=0, domain=None):
        h, cache = self.attn(self.n1(x), mask, cache, start)
        x = x + h
        if self.dense:
            return x + self.ffn(self.n2(x)), cache, x.new_zeros(3, dtype=torch.float32)
        y, aux = self.ffn(self.n2(x), domain)
        return x + y, cache, aux


class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.block_size = cfg, cfg["block_size"]
        self.wte = nn.Embedding(cfg["vocab_size"], cfg["dim"])
        self.blocks = nn.ModuleList([Block(cfg, i < cfg["dense_first"]) for i in range(cfg["layers"])])
        self.n_moe = max(1, cfg["layers"] - cfg["dense_first"])
        self.ln_f = RMSNorm(cfg["dim"])
        self.lm_head = nn.Linear(cfg["dim"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.wte.weight                                          # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("attn.o.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg["layers"]))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, domain=None):
        """Returns logits, CE loss (or None) and aux = mean over MoE layers of [balance_loss, domain_loss, routing_acc]."""
        x, aux = self.wte(idx), 0.0
        for blk in self.blocks:
            if self.training and USE_GRAD_CKPT:
                x, a = torch.utils.checkpoint.checkpoint(lambda t, b=blk: b(t, domain=domain)[0::2], x, use_reentrant=False)
            else:
                x, _, a = blk(x, domain=domain)
            aux = aux + a
        logits = self.lm_head(self.ln_f(x))
        loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1)) if targets is not None else None
        return logits, loss, aux / self.n_moe

    def step(self, idx, caches, start, mask):
        """Inference step on the compressed MLA cache. Returns logits of the LAST position and the new caches."""
        x = self.wte(idx); new = []
        for blk, c in zip(self.blocks, caches):
            x, c, _ = blk(x, mask, c, start); new.append(c)
        return self.lm_head(self.ln_f(x[:, -1:]))[:, 0], new


def make_cfg(vocab_size):
    return dict(vocab_size=vocab_size, dim=N_EMBD, layers=N_LAYER, heads=N_HEAD, block_size=BLOCK_SIZE,
                d_c=D_C, d_cq=D_CQ, d_nope=D_NOPE, d_rope=D_ROPE, d_v=D_V,
                n_routed=N_ROUTED, topk_experts=TOPK_EXPERTS, expert_hid=EXPERT_HID, shared_hid=SHARED_HID,
                dense_hid=DENSE_HID, n_domains=N_DOMAINS, dense_first=DENSE_FIRST)

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    routed = sum(p.numel() for n, p in m.named_parameters() if ".experts." in n)
    c = m.cfg
    return total, total - routed * (1 - c["topk_experts"] / c["n_routed"])


@torch.inference_mode()
def generate_batch(model, tok, prompts, max_new, temperature, top_p, top_k, rep_pen, target_words):
    """Batched sampling with left-padding on the MLA latent cache. Returns a list of generated id-lists."""
    model.eval()
    dev = next(model.parameters()).device
    b, L = len(prompts), max(len(p) for p in prompts)
    assert L + max_new <= model.block_size, "prompt too long for the context window"
    idx = torch.full((b, L), tok.pad_id, dtype=torch.long)
    valid = torch.zeros(b, L, dtype=torch.bool)
    for i, p in enumerate(prompts):
        idx[i, L - len(p):] = torch.tensor(p); valid[i, L - len(p):] = True
    idx, valid = idx.to(dev), valid.to(dev)
    causal = torch.ones(L, L, dtype=torch.bool, device=dev).tril()
    eye = torch.eye(L, dtype=torch.bool, device=dev)
    mask = causal[None, None] & (valid[:, None, None, :] | eye[None, None])   # pad rows attend to themselves (no NaN)
    V = len(tok.itos)
    seen = torch.zeros(b, V, dtype=torch.bool, device=dev)
    gen = []
    with torch.autocast(device_type=dev.type, dtype=AMP_DTYPE, enabled=dev.type == "cuda"):
        logits, caches = model.step(idx, [(None, None)] * len(model.blocks), 0, mask)
        for n in range(max_new):
            lg = logits.float()
            lg[:, [tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id]] = float("-inf")   # always keep writing text
            if rep_pen != 1.0:
                lg = torch.where(seen, torch.where(lg > 0, lg / rep_pen, lg * rep_pen), lg)
            lg = lg / max(temperature, 1e-5)
            if top_k:
                kth = lg.topk(min(top_k, V), dim=-1).values[:, -1:]
                lg = lg.masked_fill(lg < kth, float("-inf"))
            if top_p < 1.0:
                sl, si = lg.sort(descending=True)
                cp = sl.softmax(-1).cumsum(-1)
                rm = cp - sl.softmax(-1) > top_p
                lg = lg.masked_fill(rm.scatter(1, si, rm), float("-inf"))
            nxt = torch.multinomial(lg.softmax(-1), 1)
            gen.append(nxt); seen.scatter_(1, nxt, True)
            valid = torch.cat([valid, torch.ones(b, 1, dtype=torch.bool, device=dev)], 1)
            if (n + 1) % 32 == 0:
                g = torch.cat(gen, 1).tolist()
                if all(len(tok.decode(r).split()) >= target_words + 8 for r in g): break
            if n + 1 < max_new:
                logits, caches = model.step(nxt, caches, L + n, valid[:, None, None, :])
    return torch.cat(gen, 1).tolist()

## 6. Generation: exactly 128 words per prompt

In [32]:
# Load tokenizer from the path resolved for this backbone (handles 24k vs 30k vocabs)
if os.path.exists(TOKENIZER_PATH):
    try:
        tokenizer = BPETokenizer.load(TOKENIZER_PATH)
        print(f"Loaded tokenizer vocab={tokenizer.vocab_size} from {TOKENIZER_PATH}")
    except Exception as e:
        print("Tokenizer load failed:", e)
else:
    print(f"WARNING: tokenizer not found at {TOKENIZER_PATH}")


Loaded tokenizer vocab=30000 from /kaggle/working/cs781_phase1/legal_bpe_mla_moe.json


In [33]:
def load_model(path=MODEL_PATH):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = LegalLM(ck["config"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()

def continue_texts(model, prompts, target_words=TARGET_WORDS, bs=GEN_BATCH, rounds=4, progress=True):
    max_ctx = BLOCK_SIZE - MAX_NEW_TOKENS
    ctx = [tokenizer.encode(clean_text(p), add_bos=True)[-max_ctx:] for p in prompts]   # no </s> after the prompt
    gen = [[] for _ in prompts]
    todo = list(range(len(prompts)))
    for _ in range(rounds):
        if not todo: break
        todo.sort(key=lambda i: len(ctx[i]))
        bar = tqdm(range(0, len(todo), bs), desc="generating", disable=not progress)
        for s in bar:
            chunk = todo[s:s + bs]
            outs = generate_batch(model, tokenizer, [ctx[i] for i in chunk], MAX_NEW_TOKENS,
                                  TEMPERATURE, TOP_P, TOP_K, REP_PENALTY, target_words)
            for i, o in zip(chunk, outs):
                gen[i] += o; ctx[i] = (ctx[i] + o)[-max_ctx:]
        todo = [i for i in todo if len(tokenizer.decode(gen[i]).split()) < target_words]
    return [" ".join(tokenizer.decode(g).split()[:target_words]) for g in gen]

model = load_model()
samples = ["The respondent submits that the appeal should be dismissed.",
           "Section 302 of the Indian Penal Code shall apply.",
           "The Court, having considered the evidence, hereby orders"]
for p, o in zip(samples, continue_texts(model, samples, progress=False)):
    print(f"PROMPT: {p}\n -> [{len(o.split())} words] {o}\n")

PROMPT: The respondent submits that the appeal should be dismissed.
 -> [128 words] The respondent is directed to take a decision on the record of the case. 2. The writ petition stands disposed of with the aforesaid observations, if any. It is also directed that in accordance with law, within a period of four months from today, the respondents shall be entitled to file appropriate representation before the first respondent in accordance with the provisions of Sections 12 and 10 of the Act. The Writ Petition is allowed accordingly. As a sequel, this revision filed by the petitioner, the petitioner is directed to furnish an additional reply and the copy of the order in the High Court's favour which is sought for, along with certified copies of the judgment and decree dated 06.02.2021 (Annexure P-4). The matter is disposed of in

PROMPT: Section 302 of the Indian Penal Code shall apply.
 -> [128 words] 2. The application for bail is allowed and the same shall be decided in accordance with

## 7. Predict on the test set -> prediction.json + submission.zip

In [34]:
def load_model(path=MODEL_PATH):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = LegalLM(ck["config"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()

def continue_texts(model, prompts, target_words=TARGET_WORDS, bs=GEN_BATCH, rounds=4, progress=True):
    max_ctx = BLOCK_SIZE - MAX_NEW_TOKENS
    ctx = [tokenizer.encode(clean_text(p), add_bos=True)[-max_ctx:] for p in prompts]   # no </s> after the prompt
    gen = [[] for _ in prompts]
    # max_prompt_len = max(len(p) for p in prompts)
    # safe_max_new = min(MAX_NEW_TOKENS, model.block_size - max_prompt_len)
    todo = list(range(len(prompts)))
    for _ in range(rounds):
        if not todo: break
        todo.sort(key=lambda i: len(ctx[i]))
        bar = tqdm(range(0, len(todo), bs), desc="generating", disable=not progress)
        for s in bar:
            chunk = todo[s:s + bs]
            outs = generate_batch(model, tokenizer, [ctx[i] for i in chunk], MAX_NEW_TOKENS,
                                  TEMPERATURE, TOP_P, TOP_K, REP_PENALTY, target_words)
            for i, o in zip(chunk, outs):
                gen[i] += o; ctx[i] = (ctx[i] + o)[-max_ctx:]
        todo = [i for i in todo if len(tokenizer.decode(gen[i]).split()) < target_words]
    return [" ".join(tokenizer.decode(g).split()[:target_words]) for g in gen]

model = load_model()
print(model)

LegalLM(
  (wte): Embedding(30000, 640)
  (blocks): ModuleList(
    (0): Block(
      (n1): RMSNorm()
      (n2): RMSNorm()
      (attn): MLA(
        (dq): Linear(in_features=640, out_features=256, bias=False)
        (q_norm): RMSNorm()
        (uq): Linear(in_features=256, out_features=960, bias=False)
        (dkv): Linear(in_features=640, out_features=160, bias=False)
        (kv_norm): RMSNorm()
        (uk): Linear(in_features=128, out_features=640, bias=False)
        (uv): Linear(in_features=128, out_features=640, bias=False)
        (o): Linear(in_features=640, out_features=640, bias=False)
        (rope): RoPE()
      )
      (ffn): FFN(
        (w13): Linear(in_features=640, out_features=3456, bias=False)
        (w2): Linear(in_features=1728, out_features=640, bias=False)
      )
    )
    (1-7): 7 x Block(
      (n1): RMSNorm()
      (n2): RMSNorm()
      (attn): MLA(
        (dq): Linear(in_features=640, out_features=256, bias=False)
        (q_norm): RMSNorm()
        (

### Phase content (`p2`) – cells [28:55] from source


## Phase 2: NER(Named Entity Recognition for Legal Documents)

In [35]:
import os, json, pickle, random, math, time
from pathlib import Path
from dataclasses import fields
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# IMPORTANT: point these to your actual Phase-1 LEGAL model artifacts.
# Example only: change this directory to the folder containing your LegalLM.
LEGAL_MODEL_DIR = Path("/kaggle/working/cs781_phase1")
# MODEL_CONFIG_PATH = LEGAL_MODEL_DIR / "config.json"
MODEL_WEIGHTS_PATH = LEGAL_MODEL_DIR / "legal_lm_v3.pt"
TOKENIZER_STATE_PATH = LEGAL_MODEL_DIR / "legal_bpe_v2.json"

OUTPUT_DIR = Path("/kaggle/working/lner_full_finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DOCS = None             # None = use all available annotated documents
VALIDATION_FRACTION = 0.10
EPOCHS = 4
BATCH_SIZE = 4              # reduce to 1 or 2 if GPU memory is limited
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WINDOW_OVERLAP = 64

In [36]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold1-gold.json"]
VAL_FILES = ["Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json"]

In [37]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [38]:
DATA_DIR = Path("/kaggle/working/Phase_2_LNER")
TRAIN_DIR = DATA_DIR / "Train"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def download_lner_file(filename):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename=f"Phase_2_LNER/Train/{filename}",
        local_dir="/kaggle/working/hf_legal_data"
    )

paths = {
    "fold1_text": download_lner_file("fold1-text.json"),
    "fold1_gold": download_lner_file("fold1-gold.json"),
    "fold2_text": download_lner_file("fold2-text.json"),
    "fold2_gold": download_lner_file("fold2-gold.json"),
    "labels": download_lner_file("labels.txt"),
}

with open(paths["fold1_text"], encoding="utf-8") as f:
    text1 = json.load(f)
with open(paths["fold1_gold"], encoding="utf-8") as f:
    gold1 = json.load(f)
with open(paths["fold2_text"], encoding="utf-8") as f:
    text2 = json.load(f)
with open(paths["fold2_gold"], encoding="utf-8") as f:
    gold2 = json.load(f)

with open(paths["labels"], encoding="utf-8") as f:
    ENTITY_TYPES = [line.strip() for line in f if line.strip()]

print("Entity types:", ENTITY_TYPES)
print("Fold 1 documents:", len(text1), "annotations:", len(gold1))
print("Fold 2 documents:", len(text2), "annotations:", len(gold2))

Entity types: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1 documents: 35 annotations: 35
Fold 2 documents: 35 annotations: 35


In [39]:
model = load_model()

In [40]:
tokenizer = BPETokenizer.load(TOKENIZER_PATH)
print("loaded tokenizer, vocab", tokenizer.vocab_size)

loaded tokenizer, vocab 30000


In [41]:
def normalize_entity(e):
    if isinstance(e, dict):
        start = e.get("start", e.get("start_offset", e.get("start_char")))
        end = e.get("end", e.get("end_offset", e.get("end_char")))
        label = e.get("label", e.get("entity_label", e.get("type")))
        entity_text = e.get("text", e.get("entity_text"))
    elif isinstance(e, (list, tuple)) and len(e) >= 3:
        start, end, label = e[:3]
        entity_text = e[3] if len(e) >= 4 else None
    else:
        raise ValueError(f"Unrecognized entity annotation: {e!r}")

    if start is None or end is None or label is None:
        raise ValueError(f"Missing start/end/label: {e!r}")

    return int(start), int(end), str(label), entity_text

def get_entity_list(gold_record):
    # Supports a direct list of entities or a wrapper with an entities field.
    if isinstance(gold_record, dict) and "entities" in gold_record:
        gold_record = gold_record["entities"]
    if not isinstance(gold_record, list):
        raise ValueError(f"Expected entity list; got {type(gold_record)}")
    return [normalize_entity(e) for e in gold_record]

def validate_document(text, entities, doc_id):
    for start, end, label, entity_text in entities:
        assert 0 <= start < end <= len(text), (
            f"Invalid offsets for document {doc_id}: {(start, end, label)}"
        )
        assert label in ENTITY_TYPES, (
            f"Unknown label {label!r} in document {doc_id}"
        )
        if entity_text is not None and text[start:end] != entity_text:
            raise ValueError(
                f"Text/offset mismatch in {doc_id}: "
                f"{text[start:end]!r} != {entity_text!r}; "
                "check the dataset's end-offset convention."
            )

documents = {}
annotations = {}

for texts, golds in ((text1, gold1), (text2, gold2)):
    for doc_id, text in texts.items():
        if doc_id not in golds:
            raise KeyError(f"No gold annotations for document {doc_id}")
        entities = get_entity_list(golds[doc_id])
        validate_document(text, entities, doc_id)
        documents[str(doc_id)] = text
        annotations[str(doc_id)] = entities

if MAX_DOCS is not None:
    selected_ids = list(documents)[:MAX_DOCS]
    documents = {k: documents[k] for k in selected_ids}
    annotations = {k: annotations[k] for k in selected_ids}

print("Total documents:", len(documents))
example_id = next(iter(documents))
print("Example ID:", example_id)
print("Example text:", documents[example_id][:500])
print("Example entities:", annotations[example_id][:10])

Total documents: 70
Example ID: 115651329
Example text: REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 
Example entities: [(137, 153, 'RESP', 'STATE OF HARYANA'), (252, 261, 'DATE', '19.8.2011'), (276, 308, 'COURT', 'High Court of Punjab and Haryana'), (359, 369, 'COURT', 'High Court'), (575, 585, 'DATE', '18.11.1994'), (637, 644, 'WIT', 'Jagdish'), (695, 704, 'WIT', 'Mange Ram'), (792, 800, 'APP', 'Jage Ram'), (1122, 1130, 'APP', 'Jage Ram'), (1341, 1349, 'APP', 'Jage Ram')]


In [42]:
def encode_with_offsets(text, tok):
    ids, offsets = [], []
    pattern = tok._split

    for match in pattern.finditer(text):
        word = match.group()
        if not word:
            continue

        # Initial byte tokens, each with its byte interval.
        raw = word.encode("utf-8")
        pieces = [(bytes([b]), i, i + 1) for i, b in enumerate(raw)]

        # Convert byte intervals to character intervals within this pre-token.
        char_byte_intervals = []
        cursor = 0
        for ch in word:
            nbytes = len(ch.encode("utf-8"))
            char_byte_intervals.append((cursor, cursor + nbytes))
            cursor += nbytes

        # Apply the same lowest-rank adjacent-pair merges as BPETokenizer._word.
        while len(pieces) > 1:
            best_i, best_rank = -1, float("inf")
            for i in range(len(pieces) - 1):
                rank = tok.ranks.get((pieces[i][0], pieces[i + 1][0]), float("inf"))
                if rank < best_rank:
                    best_i, best_rank = i, rank
            if best_i < 0:
                break
            a, b = pieces[best_i], pieces[best_i + 1]
            pieces[best_i:best_i + 2] = [(a[0] + b[0], a[1], b[2])]

        for piece, byte_start, byte_end in pieces:
            if piece not in tok.stoi:
                raise ValueError(f"Token not in pretrained vocabulary: {piece!r}")
            char_start = next(
                (i for i, (_, end) in enumerate(char_byte_intervals)
                 if end > byte_start),
                len(word) - 1
            )
            char_end = next(
                (i for i, (start, _) in enumerate(char_byte_intervals)
                 if start >= byte_end),
                len(word)
            )
            ids.append(tok.stoi[piece])
            offsets.append((
                match.start() + char_start,
                match.start() + char_end
            ))

    return ids, offsets

# Sanity checks before training.
for sample in [
    "IN THE SUPREME COURT OF INDIA",
    "Civil Appeal No. 123 of 2024",
    "Justice R. Kumar",
    "12 January 2024"
]:
    ids, offsets = encode_with_offsets(sample, tokenizer)
    assert len(ids) == len(offsets)
    assert all(0 <= s < e <= len(sample) for s, e in offsets)
    print(sample, "=>", len(ids), "tokens")

IN THE SUPREME COURT OF INDIA => 6 tokens
Civil Appeal No. 123 of 2024 => 7 tokens
Justice R. Kumar => 4 tokens
12 January 2024 => 3 tokens


In [43]:
import json, math, random, time, copy, zipfile, os
from collections import Counter
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WIN, STRIDE = 512, 256          # window length / training stride (tokens)
OUT = "/kaggle/working/phase2-mla-moe"; os.makedirs(OUT, exist_ok=True)
CAUSAL = False                  # True = keep the pretrained causal mask (ablation)

In [44]:

def dl(name):
    return hf_hub_download(HF_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", token=HF_TOKEN)

LABELS = [l.strip() for l in open(dl("Train/labels.txt")) if l.strip()]
BIO = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
l2i = {l: i for i, l in enumerate(BIO)}; NL = len(BIO)

def load_fold(k):
    t = json.load(open(dl(f"Train/fold{k}-text.json"), encoding="utf-8"))
    g = json.load(open(dl(f"Train/fold{k}-gold.json"), encoding="utf-8"))
    ents = {}
    for d in t:
        rec = g[d]; rec = rec["entities"] if isinstance(rec, dict) and "entities" in rec else rec
        out = []
        for e in rec:
            if isinstance(e, dict):
                out.append((int(e.get("start", e.get("start_offset"))), int(e.get("end", e.get("end_offset"))),
                            e.get("label", e.get("type")), e.get("text")))
            else:
                out.append((int(e[0]), int(e[1]), e[2], e[3] if len(e) > 3 else None))
        ents[str(d)] = out
    return {str(k_): v for k_, v in t.items()}, ents

text1, ent1 = load_fold(1); text2, ent2 = load_fold(2)
print(len(text1), len(text2), LABELS)


35 35 ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [45]:
def encode_with_offsets(text, tok):
    ids, offs = [], []
    for m in tok._split.finditer(text):
        w = m.group()
        if not w: continue
        pieces = [(bytes([b]), i, i + 1) for i, b in enumerate(w.encode("utf-8"))]
        b2c = []
        for ci, ch in enumerate(w): b2c += [ci] * len(ch.encode("utf-8"))
        while len(pieces) > 1:                       # same merge rule as BPETokenizer._word
            bi, br = -1, 1 << 60
            for i in range(len(pieces) - 1):
                r = tok.ranks.get((pieces[i][0], pieces[i + 1][0]), 1 << 60)
                if r < br: bi, br = i, r
            if bi < 0: break
            a, b = pieces[bi], pieces[bi + 1]
            pieces[bi:bi + 2] = [(a[0] + b[0], a[1], b[2])]
        for p, bs, be in pieces:
            s, e = m.start() + b2c[bs], m.start() + b2c[be - 1] + 1
            while s < e and text[s].isspace(): s += 1        # <-- the fix
            ids.append(tok.stoi[p]); offs.append((s, e))
    return ids, offs

def token_labels(entities, offs):
    y = [0] * len(offs); exact = 0
    for s, e, lab, _ in entities:
        idx = [i for i, (a, b) in enumerate(offs) if a < b and a < e and b > s]
        if not idx: continue
        if offs[idx[0]][0] == s and offs[idx[-1]][1] == e: exact += 1
        y[idx[0]] = l2i[f"B-{lab}"]
        for i in idx[1:]: y[i] = l2i[f"I-{lab}"]
    return y, exact

def build_docs(texts, ents):
    docs = {}
    for d, t in tqdm(texts.items(), desc="encode"):
        ids, offs = encode_with_offsets(t, tokenizer)
        y, ex = token_labels(ents[d], offs)
        docs[d] = dict(text=t, ids=ids, offs=offs, y=y, gold=[(s, e, l) for s, e, l, _ in ents[d]], exact=ex)
    return docs

train_docs, val_docs = build_docs(text1, ent1), build_docs(text2, ent2)
n_ent = sum(len(d["gold"]) for d in train_docs.values())
print("exact-boundary entities (train):", sum(d["exact"] for d in train_docs.values()), "/", n_ent)


encode:   0%|          | 0/35 [00:00<?, ?it/s]

encode:   0%|          | 0/35 [00:00<?, ?it/s]

exact-boundary entities (train): 4048 / 4120


In [46]:
def viterbi(logp):
    T = logp.shape[0]
    if T == 0: return []
    allow = np.zeros((NL, NL), bool)       # allow[prev, cur]
    for p in range(NL):
        for c in range(NL):
            n = BIO[c]
            allow[p, c] = not (n.startswith("I-") and not (BIO[p] != "O" and BIO[p][2:] == n[2:]))
    trans = np.where(allow, 0.0, -1e9)
    start = np.array([0.0 if not BIO[c].startswith("I-") else -1e9 for c in range(NL)])
    dp = start + logp[0]; back = np.zeros((T, NL), int)
    for t in range(1, T):
        sc = dp[:, None] + trans
        back[t] = sc.argmax(0); dp = sc.max(0) + logp[t]
    path = [int(dp.argmax())]
    for t in range(T - 1, 0, -1): path.append(int(back[t][path[-1]]))
    return path[::-1]

def tags_to_spans(tags, offs):
    spans, cur = [], None
    for t, (s, e) in zip(tags, offs):
        if s == e: continue                  # whitespace token: skip, never breaks a span
        n = BIO[t]
        if n == "O":
            if cur: spans.append(tuple(cur)); cur = None
        elif n.startswith("B-") or cur is None or cur[2] != n[2:]:
            if cur: spans.append(tuple(cur))
            cur = [s, e, n[2:]]
        else:
            cur[1] = e
    if cur: spans.append(tuple(cur))
    return spans

def strict_prf(gold, pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P: (tp if x in G else fp)[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    per = {}
    for l in LABELS:
        if tp[l] + fp[l] + fn[l] == 0: continue
        p = tp[l] / max(1, tp[l] + fp[l]); r = tp[l] / max(1, tp[l] + fn[l])
        per[l] = (p, r, 2 * p * r / max(1e-9, p + r), tp[l] + fn[l])
    m = lambda i: float(np.mean([v[i] for v in per.values()])) if per else 0.0
    return dict(P=m(0), R=m(1), F1=m(2), per=per)

def error_analysis(gold, pred):
    c = Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P:
            if x in G: c["exact"] += 1; continue
            if any((g[0], g[1]) == (x[0], x[1]) for g in G): c["label_confusion"] += 1
            elif any(g[2] == x[2] and g[0] < x[1] and x[0] < g[1] for g in G): c["boundary_error"] += 1
            else: c["false_positive"] += 1
        for g in G - P:
            if not any(p[0] < g[1] and g[0] < p[1] for p in P): c["missed(FN)"] += 1
    return dict(c)

def show(res, title=""):
    print(f"{title} macro P {res['P']:.4f} R {res['R']:.4f} F1 {res['F1']:.4f}")
    for l, (p, r, f, n) in res["per"].items(): print(f"   {l:10s} P {p:.3f} R {r:.3f} F1 {f:.3f} n={n}")

# Oracle: gold BIO tags -> spans.  Upper bound of what the tokenizer alignment allows.
gold_v = {d: x["gold"] for d, x in val_docs.items()}
oracle = {d: tags_to_spans(x["y"], x["offs"]) for d, x in val_docs.items()}
show(strict_prf(gold_v, oracle), "ORACLE (should be ~1.0)")

ORACLE (should be ~1.0) macro P 0.9846 R 0.9846 F1 0.9846
   APP        P 0.990 R 0.990 F1 0.990 n=200
   RESP       P 0.963 R 0.963 F1 0.963 n=190
   A.COUNSEL  P 0.992 R 0.992 F1 0.992 n=125
   R.COUNSEL  P 1.000 R 1.000 F1 1.000 n=97
   JUDGE      P 1.000 R 1.000 F1 1.000 n=122
   WIT        P 1.000 R 1.000 F1 1.000 n=120
   AUTH       P 0.991 R 0.991 F1 0.991 n=468
   COURT      P 0.978 R 0.978 F1 0.978 n=558
   STAT       P 0.976 R 0.976 F1 0.976 n=735
   PREC       P 0.925 R 0.925 F1 0.925 n=559
   DATE       P 1.000 R 1.000 F1 1.000 n=762
   CASENO     P 1.000 R 1.000 F1 1.000 n=337


In [47]:
BOS, PAD = 2, 0
def make_windows(n, win, step):
    if n <= win: return [(0, n)]
    st = list(range(0, n - win + 1, step))
    if st[-1] + win < n: st.append(n - win)
    return [(s, s + win) for s in st]

def make_items(docs, step=STRIDE):
    items = []
    for d, x in docs.items():
        for a, b in make_windows(len(x["ids"]), WIN, step):
            items.append(([BOS] + x["ids"][a:b], [-100] + x["y"][a:b]))
    return items

def collate(batch):
    L = max(len(i) for i, _ in batch)
    ids = torch.full((len(batch), L), PAD); y = torch.full((len(batch), L), -100); m = torch.zeros(len(batch), L, dtype=torch.bool)
    for k, (i, l) in enumerate(batch):
        ids[k, :len(i)] = torch.tensor(i); y[k, :len(l)] = torch.tensor(l); m[k, :len(i)] = True
    return ids, y, m

_ST = {"causal": CAUSAL, "mask": None}
if not hasattr(MLA, "_orig_forward"): MLA._orig_forward = MLA.forward
def _mla_forward(self, x, mask=None, cache=None, start=0):
    if cache is not None: return MLA._orig_forward(self, x, mask, cache, start)
    b, t, _ = x.shape
    q = self.uq(self.q_norm(self.dq(x))).reshape(b, t, self.h, self.dn + self.dr).transpose(1, 2)
    q_nope, q_rope = q.split([self.dn, self.dr], -1)
    ckv, k_rope = self.dkv(x).split([self.dc, self.dr], -1); ckv = self.kv_norm(ckv)
    q_rope = self.rope(q_rope, start); k_rope = self.rope(k_rope.unsqueeze(1), start)
    k_nope = self.uk(ckv).reshape(b, t, self.h, self.dn).transpose(1, 2)
    v = self.uv(ckv).reshape(b, t, self.h, self.dv).transpose(1, 2)
    qf = torch.cat([q_nope, q_rope], -1); kf = torch.cat([k_nope, k_rope.expand(-1, self.h, -1, -1)], -1)
    if _ST["causal"] or _ST["mask"] is None: y = F.scaled_dot_product_attention(qf, kf, v, is_causal=True)
    else: y = F.scaled_dot_product_attention(qf, kf, v, attn_mask=_ST["mask"])   # bidirectional + padding mask
    return self.o(y.transpose(1, 2).reshape(b, t, self.h * self.dv)), None
MLA.forward = _mla_forward

class Adapter(nn.Module):
    def __init__(s, d, r): super().__init__(); s.down = nn.Linear(d, r); s.up = nn.Linear(r, d); nn.init.zeros_(s.up.weight); nn.init.zeros_(s.up.bias)
    def forward(s, x): return x + s.up(F.gelu(s.down(x)))

class Backbone(nn.Module):
    """Interface for ANY Phase-1 model: forward(ids, attn_mask)->(b,t,dim) and .dim.  Implement it for your other 3 architectures."""
    def __init__(s, lm, causal=False, adapter_dim=0):
        super().__init__(); s.lm, s.causal, s.dim = lm, causal, lm.cfg["dim"]
        s.adapters = nn.ModuleList([Adapter(s.dim, adapter_dim) for _ in lm.blocks]) if adapter_dim else None
    def forward(s, ids, attn):
        _ST["causal"] = s.causal; _ST["mask"] = None if s.causal else attn[:, None, None, :]
        x = s.lm.wte(ids)
        for i, blk in enumerate(s.lm.blocks):
            x, _, _ = blk(x)
            if s.adapters is not None: x = s.adapters[i](x)
        return s.lm.ln_f(x)

class NERModel(nn.Module):
    def __init__(s, bb, head="linear", drop=0.1):
        super().__init__(); s.backbone, s.drop = bb, nn.Dropout(drop); d = bb.dim
        s.head = nn.Linear(d, NL) if head == "linear" else nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, NL))
    def forward(s, ids, attn): return s.head(s.drop(s.backbone(ids, attn)))

In [48]:
class LoRALinear(nn.Module):             # y = Wx + s * B A x
    def __init__(s, base, r=8, alpha=16):
        super().__init__(); s.base, s.s = base, alpha / r
        s.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(s, x): return s.base(x) + (x @ s.A.t() @ s.B.t()) * s.s

class LoRMALinear(nn.Module):            # W' = (I + s*B A) W  ->  y = Wx + s * B A (Wx)   (multiplicative, output side)
    def __init__(s, base, r=8, alpha=16):
        super().__init__(); s.base, s.s = base, alpha / r
        s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(s, x): y = s.base(x); return y + (y @ s.A.t() @ s.B.t()) * s.s

In [49]:
class LoRALinear(nn.Module):             # y = Wx + s * B A x
    def __init__(s, base, r=8, alpha=16):
        super().__init__(); s.base, s.s = base, alpha / r
        s.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(s, x): return s.base(x) + (x @ s.A.t() @ s.B.t()) * s.s

class LoRMALinear(nn.Module):            # W' = (I + s*B A) W  ->  y = Wx + s * B A (Wx)   (multiplicative, output side)
    def __init__(s, base, r=8, alpha=16):
        super().__init__(); s.base, s.s = base, alpha / r
        s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(s, x): y = s.base(x); return y + (y @ s.A.t() @ s.B.t()) * s.s

TARGETS = {"dq", "uq", "dkv", "uk", "uv", "o", "w13", "w2"}
def inject(lm, cls, r, alpha):
    for blk in lm.blocks:
        for name, mod in list(blk.named_modules()):
            if "experts" in name.split("."): continue            # keep routed experts frozen & untouched
            for n, ch in list(mod.named_children()):
                if isinstance(ch, nn.Linear) and n in TARGETS: setattr(mod, n, cls(ch, r, alpha))

def load_lm(path=None):
    ck = torch.load(path or MODEL_PATH, map_location="cpu", weights_only=False)
    m = LegalLM(ck["config"]); m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m

def build(kind, head="linear", r=8, k_top=3, path=None):
    lm = load_lm(path); bb = Backbone(lm, CAUSAL, 64 if kind == "adapter" else 0)
    if kind in ("lora", "lorma"):
        for p in lm.parameters(): p.requires_grad = False
        inject(lm, LoRALinear if kind == "lora" else LoRMALinear, r, 2 * r)
    elif kind == "adapter":
        for p in lm.parameters(): p.requires_grad = False
    elif kind == "partial":
        for p in lm.parameters(): p.requires_grad = False
        for blk in lm.blocks[-k_top:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in lm.ln_f.parameters(): p.requires_grad = True
    return NERModel(bb, head).to(DEVICE)

def n_params(m):
    tot = sum(p.numel() for p in m.parameters()); tr = sum(p.numel() for p in m.parameters() if p.requires_grad); return tot, tr


In [50]:
@torch.no_grad()
def predict_docs(model, docs, decode="viterbi", bs=8):
    model.eval(); out = {}
    for d, x in docs.items():
        n = len(x["ids"]); acc = np.zeros((n, NL), np.float32); cnt = np.zeros(n, np.float32)
        if n == 0:
            out[d] = []
            continue
        wins = make_windows(n, WIN, WIN // 2)
        for i in range(0, len(wins), bs):
            chunk = wins[i:i + bs]
            ids, _, m = collate([([BOS] + x["ids"][a:b], [0] * (b - a + 1)) for a, b in chunk])
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg = model(ids.to(DEVICE), m.to(DEVICE))
            lp = lg.float().log_softmax(-1).cpu().numpy()
            for k, (a, b) in enumerate(chunk): acc[a:b] += lp[k, 1:b - a + 1]; cnt[a:b] += 1
        lp = acc / np.maximum(cnt, 1)[:, None]
        tags = viterbi(lp) if decode == "viterbi" else lp.argmax(-1).tolist()
        out[d] = tags_to_spans(tags, x["offs"])
    return out

def evaluate(model, docs, decode="viterbi"):
    pred = predict_docs(model, docs, decode); return strict_prf({d: x["gold"] for d, x in docs.items()}, pred), pred

In [51]:
from tqdm.auto import tqdm
def class_weights(docs, power=0.5, cap=10.0):
    c = np.bincount(np.concatenate([x["y"] for x in docs.values()]), minlength=NL).astype(float) + 1
    w = np.clip((c[0] / c) ** power, 0, cap); w[0] = 1.0
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train(model, name, epochs, lr, head_lr=1e-3, bs=8, wd=0.01, cw=None, teacher=None, T=2.0, alpha=0.5, warm=0.1, snapshot_eval=True, resume_path=None):
    # ADD: Checkpoint directory creation
    ckpt_dir = f"{OUT}/checkpoints/{name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    items = make_items(train_docs); steps = epochs * math.ceil(len(items) / bs)
    bb_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head")]
    hd_p = [p for n, p in model.named_parameters() if n.startswith("head")]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": lr}, {"params": hd_p, "lr": head_lr}], weight_decay=wd)
    base = [lr, head_lr]; scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time(); hist = []
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time()
    hist = []
    start_epoch = 0  # ADD: Track starting epoch

    # ADD: Load checkpoint if resume_path is provided
    if resume_path and os.path.exists(resume_path):
        print(f"--> Resuming from: {resume_path}")
        ckpt = torch.load(resume_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"], strict=False)
        opt.load_state_dict(ckpt["optimizer_state"])
        if USE_AMP and AMP_DTYPE == torch.float16 and "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        step = ckpt["step"]
        best = ckpt["best_f1"]
        best_state = ckpt["best_state"]
        hist = ckpt.get("hist", [])
    for ep in tqdm(range(start_epoch, epochs)):
        model.train(); random.shuffle(items); tl = 0
        for i in range(0, len(items), bs):
            f = (step + 1) / (warm * steps) if step < warm * steps else 0.5 * (1 + math.cos(math.pi * (step - warm * steps) / max(1, steps - warm * steps)))
            for g, b0 in zip(opt.param_groups, base): g["lr"] = b0 * f
            ids, y, m = [t.to(DEVICE) for t in collate(items[i:i + bs])]
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg = model(ids, m)
                loss = F.cross_entropy(lg.float().reshape(-1, NL), y.reshape(-1), weight=cw, ignore_index=-100)
                if teacher is not None:
                    with torch.no_grad(): tl_ = teacher(ids, m).float()
                    sel = y != -100
                    kd = F.kl_div((lg.float()[sel] / T).log_softmax(-1), (tl_[sel] / T).softmax(-1), reduction="batchmean") * T * T
                    loss = alpha * loss + (1 - alpha) * kd
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1; tl += loss.item()
        res, _ = evaluate(model, val_docs); hist.append((ep + 1, tl / math.ceil(len(items) / bs), res["F1"]))
        print(f"[{name}] ep {ep+1}/{epochs} loss {hist[-1][1]:.4f} | val strict macro P {res['P']:.3f} R {res['R']:.3f} F1 {res['F1']:.4f}")
        if res["F1"] > best: best = res["F1"]; best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
        # inside epoch loop, after calculating `res`:
        if res["F1"] > best:
            best = res["F1"]
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
            # ADD: Save best weights separately
            torch.save(best_state, f"{ckpt_dir}/{name}_best_weights.pt")

        # ADD: Save latest checkpoint for resuming
        ckpt_payload = {
            "epoch": ep,
            "step": step,
            "model_state": {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
            "optimizer_state": opt.state_dict(),
            "scaler_state": scaler.state_dict() if USE_AMP else None,
            "best_f1": best,
            "best_state": best_state,
            "hist": hist,
        }
        torch.save(ckpt_payload, f"{ckpt_dir}/latest_checkpoint.pt")
    model.load_state_dict(best_state, strict=False)
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else float("nan")
    return dict(best_val_f1=best, train_sec=time.time() - t0, peak_gb=peak, hist=hist)

In [52]:
# !rm -f /kaggle/working/phase2-mla-moe/checkpoints/full_ft/latest_checkpoint.pt
# !rm -f /kaggle/working/phase2-mla-moe/checkpoints/lora_r8/latest_checkpoint.pt
# !rm -f /kaggle/working/phase2-mla-moe/checkpoints/partial_top3/latest_checkpoint.pt
# !rm -f /kaggle/working/phase2-mla-moe/checkpoints/lorma_r8/latest_checkpoint.pt
# !rm -f /kaggle/working/phase2-mla-moe/checkpoints/adapter64/latest_checkpoint.pt

In [54]:
test_texts = json.load(open(dl("Test/fold3-text.json"), encoding="utf-8")); test_texts = {str(k): v for k, v in test_texts.items()}
test_docs = {}
for d, t in test_texts.items():
    ids, offs = encode_with_offsets(t, tokenizer); test_docs[d] = dict(text=t, ids=ids, offs=offs, y=[0] * len(ids), gold=[])

def write_test(model, name):
    pred = predict_docs(model, test_docs)
    js = {d: [[s, e, l, test_texts[d][s:e]] for s, e, l in sp] for d, sp in pred.items()}
    p = f"{OUT}/prediction_{name}.json"; json.dump(js, open(p, "w"), ensure_ascii=False)
    with zipfile.ZipFile(f"{OUT}/submission_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(p, "prediction.json")

CW = class_weights(train_docs)
RESULTS, MODELS = {}, {}
def run(name, kind, epochs, lr, head_lr=1e-3, head="linear", cw=CW, resume = False, **kw):
    model = build(kind, head=head, **kw); tot, tr = n_params(model)
    print(f"\n=== {name}: total {tot/1e6:.1f}M params, trainable {tr/1e6:.2f}M ({100*tr/tot:.1f}%)")
    ckpt_path = f"{OUT}/checkpoints/{name}/latest_checkpoint.pt"
    resume_path = ckpt_path if (resume and os.path.exists(ckpt_path)) else None

    # CHANGE: Pass resume_path to train()
    info = train(model, name, epochs, lr, head_lr, cw=cw, resume_path=resume_path)
    t0 = time.time(); res, pred = evaluate(model, val_docs); info["infer_sec"] = time.time() - t0
    info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
    RESULTS[name] = info; MODELS[name] = model; write_test(model, name); show(res, f"[{name}] FINAL VAL"); return model

# Exp 1: full fine-tuning (also the KD teacher)
run("full_ft", "full", epochs=4, lr=5e-5, resume = True)
# Exp 2: partial / layer-wise (top-3 blocks)
run("partial_top3", "partial", epochs=4, lr=1e-4, k_top=3, resume = True)
# Exp 3: LoRA
run("lora_r8", "lora", epochs=4, lr=1e-3, r=8, resume = True)
# Exp 4: LoRMA
run("lorma_r8", "lorma", epochs=4, lr=1e-3, r=8, resume = True)
# Exp 5: Houlsby-style adapters
run("adapter64", "adapter", epochs=4, lr=1e-3, resume = True)


=== full_ft: total 100.1M params, trainable 100.08M (100.0%)
--> Resuming from: /kaggle/working/phase2-mla-moe/checkpoints/full_ft/latest_checkpoint.pt


0it [00:00, ?it/s]

[full_ft] FINAL VAL macro P 0.4111 R 0.4713 F1 0.4324
   APP        P 0.128 R 0.135 F1 0.131 n=200
   RESP       P 0.218 R 0.168 F1 0.190 n=190
   A.COUNSEL  P 0.366 R 0.600 F1 0.455 n=125
   R.COUNSEL  P 0.353 R 0.371 F1 0.362 n=97
   JUDGE      P 0.543 R 0.721 F1 0.620 n=122
   WIT        P 0.250 R 0.108 F1 0.151 n=120
   AUTH       P 0.314 R 0.271 F1 0.291 n=468
   COURT      P 0.692 R 0.842 F1 0.760 n=558
   STAT       P 0.623 R 0.660 F1 0.641 n=735
   PREC       P 0.284 R 0.369 F1 0.321 n=559
   DATE       P 0.843 R 0.932 F1 0.885 n=762
   CASENO     P 0.319 R 0.478 F1 0.382 n=337

=== partial_top3: total 100.1M params, trainable 32.78M (32.8%)
--> Resuming from: /kaggle/working/phase2-mla-moe/checkpoints/partial_top3/latest_checkpoint.pt


0it [00:00, ?it/s]

[partial_top3] FINAL VAL macro P 0.4304 R 0.4375 F1 0.4251
   APP        P 0.210 R 0.130 F1 0.160 n=200
   RESP       P 0.264 R 0.126 F1 0.171 n=190
   A.COUNSEL  P 0.377 R 0.440 F1 0.406 n=125
   R.COUNSEL  P 0.360 R 0.371 F1 0.365 n=97
   JUDGE      P 0.507 R 0.598 F1 0.549 n=122
   WIT        P 0.229 R 0.067 F1 0.103 n=120
   AUTH       P 0.352 R 0.301 F1 0.325 n=468
   COURT      P 0.756 R 0.837 F1 0.794 n=558
   STAT       P 0.602 R 0.595 F1 0.598 n=735
   PREC       P 0.306 R 0.390 F1 0.343 n=559
   DATE       P 0.877 R 0.929 F1 0.902 n=762
   CASENO     P 0.326 R 0.466 F1 0.384 n=337

=== lora_r8: total 100.7M params, trainable 0.68M (0.7%)
--> Resuming from: /kaggle/working/phase2-mla-moe/checkpoints/lora_r8/latest_checkpoint.pt


0it [00:00, ?it/s]

... (2 outputs left out)

[lorma_r8] FINAL VAL macro P 0.4161 R 0.4430 F1 0.4229
   APP        P 0.119 R 0.105 F1 0.111 n=200
   RESP       P 0.140 R 0.095 F1 0.113 n=190
   A.COUNSEL  P 0.387 R 0.384 F1 0.386 n=125
   R.COUNSEL  P 0.328 R 0.412 F1 0.365 n=97
   JUDGE      P 0.438 R 0.607 F1 0.509 n=122
   WIT        P 0.382 R 0.175 F1 0.240 n=120
   AUTH       P 0.357 R 0.331 F1 0.344 n=468
   COURT      P 0.733 R 0.864 F1 0.793 n=558
   STAT       P 0.635 R 0.669 F1 0.652 n=735
   PREC       P 0.263 R 0.318 F1 0.288 n=559
   DATE       P 0.902 R 0.928 F1 0.915 n=762
   CASENO     P 0.311 R 0.427 F1 0.360 n=337

=== adapter64: total 100.7M params, trainable 0.68M (0.7%)


  0%|          | 0/4 [00:00<?, ?it/s]

[adapter64] ep 1/4 loss 0.9649 | val strict macro P 0.232 R 0.275 F1 0.2446
[adapter64] ep 2/4 loss 0.2374 | val strict macro P 0.301 R 0.414 F1 0.3387
[adapter64] ep 3/4 loss 0.1418 | val strict macro P 0.317 R 0.400 F1 0.3490
[adapter64] ep 4/4 loss 0.1036 | val strict macro P 0.350 R 0.425 F1 0.3753
[adapter64] FINAL VAL macro P 0.3504 R 0.4253 F1 0.3753
   APP        P 0.136 R 0.130 F1 0.133 n=200
   RESP       P 0.037 R 0.047 F1 0.042 n=190
   A.COUNSEL  P 0.366 R 0.360 F1 0.363 n=125
   R.COUNSEL  P 0.319 R 0.474 F1 0.382 n=97
   JUDGE      P 0.313 R 0.516 F1 0.390 n=122
   WIT        P 0.348 R 0.133 F1 0.193 n=120
   AUTH       P 0.273 R 0.378 F1 0.317 n=468
   COURT      P 0.666 R 0.832 F1 0.739 n=558
   STAT       P 0.508 R 0.631 F1 0.563 n=735
   PREC       P 0.196 R 0.277 F1 0.230 n=559
   DATE       P 0.818 R 0.906 F1 0.859 n=762
   CASENO     P 0.225 R 0.418 F1 0.293 n=337


NERModel(
  (backbone): Backbone(
    (lm): LegalLM(
      (wte): Embedding(30000, 640)
      (blocks): ModuleList(
        (0): Block(
          (n1): RMSNorm()
          (n2): RMSNorm()
          (attn): MLA(
            (dq): Linear(in_features=640, out_features=256, bias=False)
            (q_norm): RMSNorm()
            (uq): Linear(in_features=256, out_features=960, bias=False)
            (dkv): Linear(in_features=640, out_features=160, bias=False)
            (kv_norm): RMSNorm()
            (uk): Linear(in_features=128, out_features=640, bias=False)
            (uv): Linear(in_features=128, out_features=640, bias=False)
            (o): Linear(in_features=640, out_features=640, bias=False)
            (rope): RoPE()
          )
          (ffn): FFN(
            (w13): Linear(in_features=640, out_features=3456, bias=False)
            (w2): Linear(in_features=1728, out_features=640, bias=False)
          )
        )
        (1-7): 7 x Block(
          (n1): RMSNorm()
          

In [55]:
teacher = MODELS["full_ft"].eval()
for p in teacher.parameters(): p.requires_grad = False
p1 = load_lm(); cfg = dict(p1.cfg); keep = [0, 2, 4, 7]; cfg["layers"] = len(keep)
slm = LegalLM(cfg); slm.wte.load_state_dict(p1.wte.state_dict()); slm.ln_f.load_state_dict(p1.ln_f.state_dict())
for i, j in enumerate(keep): slm.blocks[i].load_state_dict(p1.blocks[j].state_dict())   # init from Phase-1 weights
student = NERModel(Backbone(slm, CAUSAL)).to(DEVICE); tot, tr = n_params(student)
print(f"=== KD student: {tot/1e6:.1f}M params (teacher {RESULTS['full_ft']['total_M']:.1f}M)")
info = train(student, "kd_student", epochs=4, lr=1e-4, cw=CW, teacher=teacher, T=2.0, alpha=0.5)
t0 = time.time(); res, pred = evaluate(student, val_docs); info["infer_sec"] = time.time() - t0
info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
RESULTS["kd_student"] = info; MODELS["kd_student"] = student; write_test(student, "kd_student"); show(res, "[kd_student] FINAL VAL")
# Control: same student WITHOUT distillation (does KD actually help?)
student0 = NERModel(Backbone(copy.deepcopy(slm), CAUSAL)).to(DEVICE)
i0 = train(student0, "student_noKD", 10, 1e-4, cw=CW)
RESULTS["student_noKD"] = dict(best_val_f1=i0["best_val_f1"], F1=i0["best_val_f1"], total_M=tot / 1e6, trainable_M=tr / 1e6, train_sec=i0["train_sec"], peak_gb=i0["peak_gb"])

=== KD student: 56.4M params (teacher 100.1M)


  0%|          | 0/4 [00:00<?, ?it/s]

[kd_student] ep 1/4 loss 1.2884 | val strict macro P 0.361 R 0.358 F1 0.3110
[kd_student] ep 2/4 loss 0.1636 | val strict macro P 0.371 R 0.410 F1 0.3796
[kd_student] ep 3/4 loss 0.0566 | val strict macro P 0.403 R 0.416 F1 0.3982
[kd_student] ep 4/4 loss 0.0300 | val strict macro P 0.410 R 0.418 F1 0.4045
[kd_student] FINAL VAL macro P 0.4098 R 0.4181 F1 0.4045
   APP        P 0.158 R 0.105 F1 0.126 n=200
   RESP       P 0.134 R 0.079 F1 0.099 n=190
   A.COUNSEL  P 0.344 R 0.360 F1 0.352 n=125
   R.COUNSEL  P 0.281 R 0.423 F1 0.337 n=97
   JUDGE      P 0.436 R 0.582 F1 0.498 n=122
   WIT        P 0.375 R 0.125 F1 0.188 n=120
   AUTH       P 0.297 R 0.241 F1 0.267 n=468
   COURT      P 0.759 R 0.853 F1 0.803 n=558
   STAT       P 0.671 R 0.605 F1 0.637 n=735
   PREC       P 0.258 R 0.315 F1 0.284 n=559
   DATE       P 0.854 R 0.925 F1 0.888 n=762
   CASENO     P 0.351 R 0.404 F1 0.376 n=337


  0%|          | 0/10 [00:00<?, ?it/s]

[student_noKD] ep 1/10 loss 0.5897 | val strict macro P 0.361 R 0.374 F1 0.3507
[student_noKD] ep 2/10 loss 0.0453 | val strict macro P 0.322 R 0.423 F1 0.3553
[student_noKD] ep 3/10 loss 0.0275 | val strict macro P 0.419 R 0.412 F1 0.4031
[student_noKD] ep 4/10 loss 0.0158 | val strict macro P 0.394 R 0.398 F1 0.3825
[student_noKD] ep 5/10 loss 0.0081 | val strict macro P 0.393 R 0.413 F1 0.3907
[student_noKD] ep 6/10 loss 0.0027 | val strict macro P 0.419 R 0.428 F1 0.4096
[student_noKD] ep 7/10 loss 0.0010 | val strict macro P 0.413 R 0.423 F1 0.4037
[student_noKD] ep 8/10 loss 0.0005 | val strict macro P 0.416 R 0.427 F1 0.4086
[student_noKD] ep 9/10 loss 0.0004 | val strict macro P 0.416 R 0.427 F1 0.4077
[student_noKD] ep 10/10 loss 0.0003 | val strict macro P 0.415 R 0.426 F1 0.4069


In [56]:
import pandas as pd
rows = [dict(exp=k, total_M=round(v["total_M"], 1), trainable_M=round(v["trainable_M"], 2), P=round(v.get("P", np.nan), 4), R=round(v.get("R", np.nan), 4),
             F1=round(v["F1"], 4), train_min=round(v["train_sec"] / 60, 1), peak_GB=round(v["peak_gb"], 2), infer_sec=round(v.get("infer_sec", np.nan), 1)) for k, v in RESULTS.items()]
df = pd.DataFrame(rows).sort_values("F1", ascending=False); print(df.to_string(index=False))
best = df.iloc[0]["exp"]; print("\nBEST on val:", best)
for dec in ("greedy", "viterbi"):          # span-decoding ablation
    r, _ = evaluate(MODELS[best], val_docs, dec); print(f"decode={dec}: macro F1 {r['F1']:.4f}")
print("error breakdown:", RESULTS[best]["errors"])
show(strict_prf(gold_v, evaluate(MODELS[best], val_docs)[1]), f"per-label ({best})")
print(f"\nSubmit: {OUT}/submission_{best}.zip  (pick 4 submissions from DIFFERENT Phase-1 models/methods)")

         exp  total_M  trainable_M      P      R     F1  train_min  peak_GB  infer_sec
     full_ft    100.1       100.08 0.4111 0.4713 0.4324        0.0     5.41       26.5
     lora_r8    100.7         0.68 0.4442 0.4427 0.4278        0.0     4.68       29.2
partial_top3    100.1        32.78 0.4304 0.4375 0.4251        0.0     4.78       26.2
    lorma_r8    100.9         0.80 0.4161 0.4430 0.4229        0.0     5.06       29.2
student_noKD     56.4        56.39    NaN    NaN 0.4096        8.4     6.88        NaN
  kd_student     56.4        56.39 0.4098 0.4181 0.4045        5.0     6.97       14.6
   adapter64    100.7         0.68 0.3504 0.4253 0.3753        5.2     7.27       26.9

BEST on val: full_ft
decode=greedy: macro F1 0.3227
decode=viterbi: macro F1 0.4324
error breakdown: {'exact': 2430, 'boundary_error': 727, 'false_positive': 1432, 'label_confusion': 224, 'missed(FN)': 788}
per-label (full_ft) macro P 0.4111 R 0.4713 F1 0.4324
   APP        P 0.128 R 0.135 F1 0.131 n=2

In [57]:
import pandas as pd
import numpy as np

def inspect_ner_failures(model, val_docs, gold_v, max_examples=5):
    """
    Runs diagnostic inspection on validation documents to identify
    where and why the model is failing (boundary errors vs false positives vs label confusion).
    """
    model.eval()
    # Get predictions using your existing predict_docs function
    pred_spans = predict_docs(model, val_docs, decode="viterbi")

    # Run error analysis breakdown
    analysis = error_analysis(gold_v, pred_spans)
    print("==========================================")
    print(" GLOBAL ERROR BREAKDOWN:")
    print("==========================================")
    for err_type, count in analysis.items():
        print(f" - {err_type}: {count}")

    print("\n==========================================")
    print(" DETAILED EXAMPLES OF MODEL ERRORS:")
    print("==========================================")

    printed_count = 0
    for doc_id, doc in val_docs.items():
        text = doc["text"]
        g_spans = set(gold_v.get(doc_id, []))
        p_spans = set(pred_spans.get(doc_id, []))

        # Find documents where predictions don't match gold exactly
        if g_spans != p_spans and printed_count < max_examples:
            print(f"\n--- Document ID: {doc_id} ---")
            print(f"Snippet: {text[:300]}...\n")

            print("Gold Entities:")
            for s, e, l in sorted(g_spans):
                print(f"  [{l}] {text[s:e]} (chars {s}:{e})")

            print("\nPredicted Entities:")
            for s, e, l in sorted(p_spans):
                status = "CORRECT" if (s, e, l) in g_spans else "ERROR"
                print(f"  [{l}] {text[s:e]} (chars {s}:{e}) -> {status}")

            printed_count += 1

# Run inspection using your best performing model (e.g., lora_r8 or full_ft)
best_model_name = "lora_r8" if "lora_r8" in MODELS else list(MODELS.keys())[0]
print(f"Inspecting errors for model: {best_model_name}")
inspect_ner_failures(MODELS[best_model_name], val_docs, gold_v)

Inspecting errors for model: lora_r8
 GLOBAL ERROR BREAKDOWN:
 - exact: 2328
 - false_positive: 1023
 - boundary_error: 710
 - label_confusion: 206
 - missed(FN): 962

 DETAILED EXAMPLES OF MODEL ERRORS:

--- Document ID: 184861797 ---
Snippet: IN THE HIGH COURT AT CALCUTTA CIVIL REVISIONAL JURISDICTION APPELLATE SIDE Before: The Hon'ble Justice Hiranmay Bhattacharyya C.O. 3346 of 2018 The Kolkata Municipal Corporation Vs. Smt. Kajari Banerjee For the petitioner : Mr. Alok Kr. Ghosh, Mr. S.K. Debnath..... advocates For the opposite party :...

Gold Entities:
  [JUDGE] Hiranmay Bhattacharyya (chars 103:125)
  [CASENO] C.O. 3346 of 2018 (chars 126:143)
  [APP] Kolkata Municipal Corporation (chars 148:177)
  [RESP] Smt. Kajari Banerjee (chars 182:202)
  [A.COUNSEL] Ghosh (chars 237:242)
  [R.COUNSEL] R.N. Chakraborty (chars 305:321)
  [R.COUNSEL] M. Ahmed (chars 327:335)
  [CASENO] 18.12.2020 (chars 369:379)
  [CASENO] 22.12.2020 (chars 394:404)
  [JUDGE] Hiranmay Bhattacharyya (chars 405

In [58]:
%%writefile evaluate_phase2.py
#!/usr/bin/env python3
"""Local evaluator for Phase 2.

Usage:
  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt
"""
import argparse, json, sys

class DuplicateJSONKeyError(ValueError): pass
def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")
def pairs_no_duplicates(pairs):
    out={}; dup=[]
    for k,v in pairs:
        if k in out: dup.append(k)
        out[k]=v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out
def load_json(path):
    with open(path,encoding='utf-8') as f:
        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e
def labels(path):
    with open(path,encoding='utf-8') as f: xs=[x.strip() for x in f if x.strip()]
    if not xs: raise ValueError('labels.txt is empty.')
    if len(xs)!=12: raise ValueError(f'Expected 12 labels, found {len(xs)}.')
    if len(set(xs))!=len(xs): raise ValueError('labels.txt contains duplicate labels.')
    return xs
def spans(x, allowed, where):
    if x is None: return set()
    if not isinstance(x,list): raise ValueError(f'{where}: expected a list of entities.')
    out=set()
    for j,s in enumerate(x):
        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].')
        a,b,l=s[:3]
        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f'{where}, entity {j}: start/end must be integers.')
        if a<0 or b<=a: raise ValueError(f'{where}, entity {j}: require 0 <= start < end.')
        if not isinstance(l,str) or l not in allowed: raise ValueError(f'{where}, entity {j}: invalid label {l!r}.')
        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f'{where}, entity {j}: entity_text must be a string.')
        t=(a,b,l)
        if t in out: raise ValueError(f'{where}, entity {j}: duplicate span {t}.')
        out.add(t)
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 2 entity predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); ap.add_argument('--labels',required=True)
    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)
    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError('Truth and prediction must be non-empty JSON objects.')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    stats={l:[0,0,0] for l in labs}; allowed=set(labs)
    for did in truth:
        g=spans(truth[did],allowed,f'truth[{did}]'); p=spans(pred[did],allowed,f'prediction[{did}]')
        for l in labs:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    precisions=[]; recalls=[]; f1s=[]
    print(f'Examples: {len(truth)}')
    for l in labs:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0
        precisions.append(pr); recalls.append(rc); f1s.append(f)
        print(f'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}')
    print(f'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}')
    print(f'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}')
    print(f'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}')
# if __name__=='__main__':
#     try: main()
#     except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

Overwriting evaluate_phase2.py


In [59]:

from collections import defaultdict

def inspect_ner_errors(model, docs, top_k=15):
    predictions = predict_docs(model, docs)

    def norm(items):
        return {(int(x[0]), int(x[1]), str(x[2])) for x in items}

    def overlap(a, b):
        return max(a[0], b[0]) < min(a[1], b[1])

    label_errors = []
    boundary_errors = []
    wrong_label_overlaps = []
    missed = []
    false_positives = []

    total_gold = total_pred = exact = 0

    for doc_id, doc in docs.items():
        text = doc["text"]
        gold = norm(doc["gold"])
        pred = norm(predictions.get(doc_id, []))
        total_gold += len(gold)
        total_pred += len(pred)
        exact += len(gold & pred)

        # Same exact boundaries, different label.
        gold_by_span = defaultdict(set)
        pred_by_span = defaultdict(set)
        for s, e, lab in gold:
            gold_by_span[(s, e)].add(lab)
        for s, e, lab in pred:
            pred_by_span[(s, e)].add(lab)

        for span in gold_by_span.keys() & pred_by_span.keys():
            for gl in gold_by_span[span]:
                for pl in pred_by_span[span]:
                    if gl != pl:
                        s, e = span
                        label_errors.append({
                            "doc": doc_id, "span": span,
                            "gold_label": gl, "pred_label": pl,
                            "text": text[s:e]
                        })

        unmatched_g = gold - pred
        unmatched_p = pred - gold

        # Pair same-label overlapping entities: likely boundary errors.
        candidates = []
        for g in unmatched_g:
            for p in unmatched_p:
                if g[2] == p[2] and overlap(g, p):
                    inter = min(g[1], p[1]) - max(g[0], p[0])
                    union = max(g[1], p[1]) - min(g[0], p[0])
                    iou = inter / union if union else 0
                    candidates.append((iou, g, p))

        used_g, used_p = set(), set()
        for iou, g, p in sorted(candidates, reverse=True):
            if g in used_g or p in used_p:
                continue
            used_g.add(g)
            used_p.add(p)
            boundary_errors.append({
                "doc": doc_id,
                "gold_span": g[:2], "pred_span": p[:2],
                "label": g[2], "iou": round(iou, 3),
                "gold_text": text[g[0]:g[1]],
                "pred_text": text[p[0]:p[1]]
            })

        # Remaining unmatched predictions with overlapping gold spans,
        # but a different label, are likely label-confusion errors.
        for g in unmatched_g - used_g:
            for p in unmatched_p - used_p:
                if overlap(g, p) and g[2] != p[2]:
                    wrong_label_overlaps.append({
                        "doc": doc_id,
                        "gold_span": g[:2], "pred_span": p[:2],
                        "gold_label": g[2], "pred_label": p[2],
                        "gold_text": text[g[0]:g[1]],
                        "pred_text": text[p[0]:p[1]]
                    })

        # Anything not paired remains a miss or a false positive.
        for g in unmatched_g - used_g:
            missed.append({
                "doc": doc_id, "span": g[:2], "label": g[2],
                "text": text[g[0]:g[1]]
            })

        for p in unmatched_p - used_p:
            false_positives.append({
                "doc": doc_id, "span": p[:2], "label": p[2],
                "text": text[p[0]:p[1]]
            })

    def show(title, rows):
        print(f"\n{'=' * 25} {title}: {len(rows)} {'=' * 25}")
        for i, x in enumerate(rows[:top_k], 1):
            print(f"\n{i}. {x}")

    print("Gold entities:", total_gold)
    print("Predicted entities:", total_pred)
    print("Exact entity matches:", exact)
    print("Strict entity precision:", round(exact / total_pred, 4)
          if total_pred else 0)
    print("Strict entity recall:", round(exact / total_gold, 4)
          if total_gold else 0)

    show("EXACT SPAN, WRONG LABEL", label_errors)
    show("RIGHT LABEL, WRONG BOUNDARIES (overlapping spans)", boundary_errors)
    show("OVERLAPPING SPANS, DIFFERENT LABELS", wrong_label_overlaps)
    show("MISSED GOLD ENTITIES", missed)
    show("FALSE POSITIVE PREDICTIONS", false_positives)

    print("\nCounts:")
    print("Exact-span label errors:", len(label_errors))
    print("Likely boundary errors:", len(boundary_errors))
    print("Overlapping different-label errors:", len(wrong_label_overlaps))
    print("Missed entities:", len(missed))
    print("False positives:", len(false_positives))

    return {
        "label_errors": label_errors,
        "boundary_errors": boundary_errors,
        "wrong_label_overlaps": wrong_label_overlaps,
        "missed": missed,
        "false_positives": false_positives,
    }


---
## Architecture: `gqa_alibi`



In [25]:
# ── Activate backbone: gqa_alibi ──────────────────────────────────────────────
BACKBONE = "gqa_alibi"
TOKENIZER_PATH = TOK_PATHS["gqa_alibi"]
PHASE1_CKPT = CKPTS["gqa_alibi"]
MODEL_PATH = PHASE1_CKPT
CKPT_PATH = PHASE1_CKPT
if "OUT_DIR" not in dir() or not OUT_DIR:
    OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"[active] BACKBONE={BACKBONE}")
print(f"[active] TOKENIZER_PATH={TOKENIZER_PATH}  exists={os.path.exists(TOKENIZER_PATH)}")
print(f"[active] PHASE1_CKPT={PHASE1_CKPT}  exists={os.path.exists(PHASE1_CKPT)}")


[active] BACKBONE=gqa_alibi
[active] TOKENIZER_PATH=/kaggle/working/cs781_phase1/legal_bpe_tokenizer_gqa_alibi.pkl  exists=True
[active] PHASE1_CKPT=/kaggle/working/cs781_phase1/legal_lm_alibi_gqa.pt  exists=True


### Phase-1 classes for `gqa_alibi` (tokenizer + model, needed to reload checkpoints)

These cells are taken from Phase 1 of the same source notebook. They define `BPETokenizer`,
the model stack (`LegalLM` / `DualPathLM` / …), and `load_model`. Internal bodies are unchanged.


## 1. Cleaning (regex) and de-duplication (exact + fuzzy)

In [26]:
# Deduplication configuration constants
DEDUP_SHINGLE = 3       # Number of words per shingle
DEDUP_MAX_WORDS = 100   # Maximum words to consider per text snippet
DEDUP_THRESHOLD = 0.8   # Jaccard similarity threshold for fuzzy matching
DEDUP_PERM = 64         # Number of permutations for MinHash
MAX_CHARS_PER_DOC = 4000000

In [27]:
import re
import hashlib
import unicodedata

_QUOTES = {
    "‘": "'", "’": "'", "‚": "'", "‛": "'",
    "“": '"', "”": '"', "„": '"',
    "–": "-", "—": "-", "−": "-", "…": "...",
    "\u00a0": " ", "•": "-", "·": "-",
}
_QTABLE = str.maketrans(_QUOTES)

RE_CTRL   = re.compile(r"[\x00-\x08\x0b-\x1f\x7f-\x9f]")
RE_URL    = re.compile(r"(?:https?://|www\.)\S+", re.I)
RE_EMAIL  = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
RE_HTML   = re.compile(r"<[^>\n]{1,200}>")

# Standalone page-number lines
RE_PAGE = re.compile(
    r"(?im)^[ \t]*(?:"
    r"page\s*\d+(?:\s*(?:of|/)\s*\d+)?"
    r"|\d+\s*/\s*\d+"
    r"|-+\s*\d+\s*-+"
    r"|\[?\s*\d{1,4}\s*\]?"
    r")[ \t]*$"
)

RE_SITE = re.compile(
    r"(?im)^[ \t]*(?:"
    r"Indian\s+Kanoon.*|"
    r"Manupatra.*|"
    r"SCC\s+Online.*|"
    r"Digitally\s+signed.*|"
    r"Downloaded\s+from.*|"
    r"True\s+copy.*"
    r")[ \t]*$"
)

RE_PAGE_INLINE = re.compile(
    r"(?i)\s*[.;,:-]?\s*page\s*\d+(?:\s*(?:of|/)\s*\d+)?"
)
RE_REPEAT     = re.compile(r"([^\w\s])\1{2,}")
RE_UNDERSC    = re.compile(r"_{2,}")
RE_NONASCII   = re.compile(r"[^\x20-\x7e\n]")
RE_WS         = re.compile(r"\s+")
RE_SPACE_PUNCT = re.compile(r"\s+([,.;:!?])")


def normalize_text(t):
    """Light normalisation (also applied to test prompts): unicode → ASCII-ish, collapse whitespace."""
    t = unicodedata.normalize("NFKC", str(t)).translate(_QTABLE)
    t = RE_CTRL.sub(" ", t)
    t = RE_NONASCII.sub(" ", t)
    return RE_WS.sub(" ", t).strip()


def clean_text(t):
    t = unicodedata.normalize("NFKC", str(t)).translate(_QTABLE)
    t = RE_CTRL.sub(" ", t)
    t = RE_HTML.sub(" ", t)

    # Remove obvious standalone scraper/watermark lines.
    t = RE_SITE.sub("\n", t)

    # Remove standalone page-number lines.
    t = RE_PAGE.sub("\n", t)

    # Remove inline page markers such as ".Page 3 of 10".
    t = RE_PAGE_INLINE.sub(" ", t)

    # Remove URLs/emails.
    t = RE_URL.sub(" ", t)
    t = RE_EMAIL.sub(" ", t)

    # Remove obvious formatting/OCR noise.
    t = RE_UNDERSC.sub(" ", t)
    t = RE_REPEAT.sub(r"\1", t)
    t = RE_NONASCII.sub(" ", t)

    # Final whitespace normalisation.
    t = RE_WS.sub(" ", t).strip()
    t = RE_SPACE_PUNCT.sub(r"\1", t)

    if len(t) > MAX_CHARS_PER_DOC:
        t = t[:MAX_CHARS_PER_DOC].rsplit(" ", 1)[0]

    return t


def good_doc(t):
    if len(t.split()) < MIN_WORDS_PER_DOC:
        return False
    letters = sum(c.isalpha() for c in t)
    return letters / max(len(t), 1) > 0.6          # drop tables / garbage / OCR noise


def extract_text(rec):
    if isinstance(rec, str):
        return rec
    if isinstance(rec, dict):
        for k in ("train_text", "text", "content", "body", "judgment", "document", "input"):
            v = rec.get(k)
            if isinstance(v, str) and v.strip():
                return v
        return "\n".join(
            filter(None, (extract_text(v) for v in rec.values() if isinstance(v, (dict, list))))
        )
    if isinstance(rec, list):
        return "\n".join(filter(None, map(extract_text, rec)))
    return ""


def iter_records(path):
    import ijson
    with open(path, "rb") as f:
        head = f.read(4096).lstrip()[:1]
        f.seek(0)
        if head == b"[":
            yield from ijson.items(f, "item")
        else:
            for _, v in ijson.kvitems(f, ""):
                yield v


def shingles(text, k=DEDUP_SHINGLE, max_words=DEDUP_MAX_WORDS):
    w = text.lower().split()[:max_words]
    return {" ".join(w[i:i + k]) for i in range(max(1, len(w) - k + 1))}


def dedup(docs, threshold=DEDUP_THRESHOLD):
    """Exact (md5 of lowercase text) then fuzzy (MinHash-LSH, Jaccard >= threshold). Keeps first occurrence."""
    from datasketch import MinHash, MinHashLSH
    seen, kept, exact_dup, fuzzy_dup = set(), [], 0, 0
    lsh = MinHashLSH(threshold=threshold, num_perm=DEDUP_PERM)
    for i, (src, t) in enumerate(tqdm(docs, desc="dedup")):
        h = hashlib.md5(t.lower().encode()).hexdigest()
        if h in seen:
            exact_dup += 1
            continue
        seen.add(h)
        m = MinHash(num_perm=DEDUP_PERM)
        for s in shingles(t):
            m.update(s.encode())
        if lsh.query(m):
            fuzzy_dup += 1
            continue
        lsh.insert(str(i), m)
        kept.append((src, t))
    print(f"dedup: {len(docs):,} → {len(kept):,}  (exact dup {exact_dup:,}, fuzzy dup {fuzzy_dup:,})")
    return kept


# quick regex sanity check
print(clean_text(
    "Having received the first information report PW-11 registered a case against all the accused "
    "on the file of Yadrami Police Station for the offences punishable under .Page 3 of 10\n"
    "Indian Kanoon - http://indiankanoon.org/doc/1/ 5\n"
    "“Held” — see www.x.com ..... ok ____ done Nice, Further reliance. was placed by the counsel "
    "appearing for the appellant on Serial No. 285 of Notification 17 / 2001 Cus d"
))

Having received the first information report PW-11 registered a case against all the accused on the file of Yadrami Police Station for the offences punishable under "Held" - see. ok done Nice, Further reliance. was placed by the counsel appearing for the appellant on Serial No. 285 of Notification 17 / 2001 Cus d


## 3. Tokenizer — byte-level BPE with word-boundary (leading-space) tokens

Every word is encoded as `" " + word`, so spaces are part of the tokens and decoding is lossless.

In [28]:
from pathlib import Path
import os

# Define path constructor
P = Path

# Define configuration variables used in tokenizer training
VOCAB_SIZE = 30000                            # Target vocabulary size
TOKENIZER_TRAIN_CHARS = 4_000_000             # Number of characters to sample for training
CLEAN_TRAIN = "clean_train_corpus.txt"        # Path to your clean training corpus file

# Ensure target directory exists for the tokenizer output path
TOKENIZER_PATH = P("legal_bpe_tokenizer_ws.pkl")
TOKENIZER_PATH.parent.mkdir(parents=True, exist_ok=True)

In [29]:
class BPETokenizer:
    reserved = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self, vocab_size=30000):
        self.vocab_size = vocab_size
        self.itos = list(self.reserved) + [bytes([i]) for i in range(256)]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.merges, self.ranks, self.merged_map, self._cache = [], {}, {}, {}

    pad_id, unk_id, bos_id, eos_id = 0, 1, 2, 3

    def train(self, texts, max_unique_words=200000):
        raw = Counter()
        for text in tqdm(texts, desc="word counts"):
            raw.update(text.split())
        raw = dict(raw.most_common(max_unique_words))

        word_freqs, pair_counts, pair_to_words = {}, defaultdict(int), defaultdict(set)
        for w, c in raw.items():
            b = (" " + w).encode("utf-8")
            bw = tuple(b[i:i + 1] for i in range(len(b)))
            word_freqs[bw] = word_freqs.get(bw, 0) + c

        for bw, c in word_freqs.items():
            for i in range(len(bw) - 1):
                p = (bw[i], bw[i + 1])
                pair_counts[p] += c
                pair_to_words[p].add(bw)

        self.merges = []
        with tqdm(total=self.vocab_size - len(self.itos), desc="BPE merges") as pbar:
            while len(self.itos) < self.vocab_size and pair_counts:
                best = max(pair_counts, key=lambda p: (pair_counts[p], p))
                if pair_counts[best] <= 0:
                    break
                merged = best[0] + best[1]
                self.merges.append((best, merged))
                self.itos.append(merged)

                words = list(pair_to_words.pop(best))
                del pair_counts[best]

                for w in words:
                    c = word_freqs.get(w, 0)
                    if not c:
                        continue
                    nw, i = [], 0
                    while i < len(w):
                        if i + 1 < len(w) and w[i] == best[0] and w[i + 1] == best[1]:
                            nw.append(merged)
                            i += 2
                        else:
                            nw.append(w[i])
                            i += 1
                    nw = tuple(nw)

                    for j in range(len(w) - 1):
                        op = (w[j], w[j + 1])
                        if op in pair_counts:
                            pair_counts[op] -= c
                            if pair_counts[op] <= 0:
                                del pair_counts[op]

                    del word_freqs[w]
                    word_freqs[nw] = word_freqs.get(nw, 0) + c
                    for j in range(len(nw) - 1):
                        np_ = (nw[j], nw[j + 1])
                        pair_counts[np_] += c
                        pair_to_words[np_].add(nw)

                pbar.update(1)

        self.vocab_size = len(self.itos)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.ranks = {p: r for r, (p, _) in enumerate(self.merges)}
        self.merged_map = dict(self.merges)
        self._cache = {}
        return self

    def encode_word(self, word):
        r = self._cache.get(word)
        if r is not None:
            return r
        b = (" " + word).encode("utf-8")
        toks = [b[i:i + 1] for i in range(len(b))]
        while len(toks) > 1:
            best, br = None, 1 << 60
            for i in range(len(toks) - 1):
                rk = self.ranks.get((toks[i], toks[i + 1]), 1 << 60)
                if rk < br:
                    br, best = rk, (toks[i], toks[i + 1])
            if best is None:
                break
            m, nt, i = self.merged_map[best], [], 0
            while i < len(toks):
                if i + 1 < len(toks) and toks[i] == best[0] and toks[i + 1] == best[1]:
                    nt.append(m)
                    i += 2
                else:
                    nt.append(toks[i])
                    i += 1
            toks = nt
        r = [self.stoi.get(t, self.unk_id) for t in toks]
        self._cache[word] = r
        return r

    def encode(self, text, bos=True, eos=True):
        ids = [self.bos_id] if bos else []
        for w in str(text).split():
            ids.extend(self.encode_word(w))
        if eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        bs = b"".join(
            self.itos[i]
            for i in ids
            if 0 <= i < len(self.itos) and isinstance(self.itos[i], bytes)
        )
        return bs.decode("utf-8", errors="ignore").strip()

    def get_vocab_size(self):
        return self.vocab_size

# FIX: load the Phase-1 ALiBi-GQA tokenizer (no corpus needed)
TOKENIZER_PATH = TOK_PATHS["gqa_alibi"]
with open(TOKENIZER_PATH, "rb") as f:
    _obj = pickle.load(f)
if isinstance(_obj, dict):
    tokenizer = BPETokenizer(_obj.get("vocab_size", 30000)); tokenizer.__dict__.update(_obj)
else:
    tokenizer = BPETokenizer(getattr(_obj, "vocab_size", 30000)); tokenizer.__dict__.update(_obj.__dict__)
if not getattr(tokenizer, "stoi", None) or len(tokenizer.stoi) != len(tokenizer.itos):
    tokenizer.stoi = {t: i for i, t in enumerate(tokenizer.itos)}
if not getattr(tokenizer, "ranks", None):
    tokenizer.ranks = {p: r for r, (p, _) in enumerate(tokenizer.merges)}
if not getattr(tokenizer, "merged_map", None):
    tokenizer.merged_map = dict(tokenizer.merges)
tokenizer._cache = {}
tokenizer.vocab_size = len(tokenizer.itos)
print("Loaded ALiBi-GQA tokenizer, vocab", tokenizer.vocab_size, "from", TOKENIZER_PATH)
s_ = "The appellant filed Section 302 IPC appeal; held: not maintainable."
print("round-trip:", tokenizer.decode(tokenizer.encode(s_)) == s_, "|", len(tokenizer.encode(s_)), "tokens")


Loaded ALiBi-GQA tokenizer, vocab 32000 from /kaggle/working/cs781_phase1/legal_bpe_tokenizer_gqa_alibi.pkl
round-trip: True | 13 tokens


## 4. Model: decoder-only Transformer, ALiBi bias, GQA, RMSNorm, SwiGLU, tied embeddings, KV cache

In [30]:
USE_GRAD_CHECKPOINTING = False
def alibi_slopes(heads):
    n = 2 ** math.floor(math.log2(heads))
    s = [2.0 ** (-8.0 / n * (i + 1)) for i in range(n)]
    if n < heads:
        extra = 2.0 ** (-4.0 / n)
        s += [extra ** (2 * i + 1) for i in range(heads - n)]
    return torch.tensor(s, dtype=torch.float32)


def alibi_causal_bias(slopes, tq, tk, device):
    """(1, H, tq, tk): -slope*(i-j) for j<=i, -inf for future.
    Queries are the LAST tq positions of a length-tk key sequence."""
    qpos = torch.arange(tk - tq, tk, device=device)[:, None]
    kpos = torch.arange(tk, device=device)[None, :]
    dist = (qpos - kpos).float()
    bias = -slopes.to(device).view(1, -1, 1, 1) * dist.clamp(min=0)
    return bias.masked_fill((dist < 0).view(1, 1, tq, tk), float("-inf"))


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.weight


class ALiBiGQA(nn.Module):
    def __init__(self, dim, heads, kv_heads, dropout=0.0):
        super().__init__()
        assert dim % heads == 0 and heads % kv_heads == 0
        self.h   = heads
        self.kvh = kv_heads
        self.hd  = dim // heads
        self.rep = heads // kv_heads
        self.dropout = dropout
        self.q   = nn.Linear(dim, dim, bias=False)
        self.kv  = nn.Linear(dim, 2 * kv_heads * self.hd, bias=False)   # smaller K/V projection + cache
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, bias, past=None, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.h, self.hd).transpose(1, 2)
        k, v = self.kv(x).view(b, t, 2, self.kvh, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        if self.rep > 1:
            k = k.repeat_interleave(self.rep, dim=1)
            v = v.repeat_interleave(self.rep, dim=1)

        attn_mask = bias.to(q.dtype) if bias is not None else None
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(b, t, -1)), present


class SwiGLU(nn.Module):
    def __init__(self, dim, dropout=0.0):
        super().__init__()
        hid = int(2 * 4 * dim / 3)
        hid = (hid + 7) // 8 * 8
        self.w1 = nn.Linear(dim, hid, bias=False)
        self.w3 = nn.Linear(dim, hid, bias=False)
        self.w2 = nn.Linear(hid, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class Block(nn.Module):
    def __init__(self, dim, heads, kv_heads, dropout):
        super().__init__()
        self.n1 = RMSNorm(dim)
        self.n2 = RMSNorm(dim)
        self.attn = ALiBiGQA(dim, heads, kv_heads, dropout)
        self.mlp  = SwiGLU(dim, dropout)

    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present


class LegalLM(nn.Module):
    def __init__(self, vocab_size, dim, layers, heads, kv_heads, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.heads = heads
        self.wte = nn.Embedding(vocab_size, dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(dim, heads, kv_heads, dropout) for _ in range(layers)
        ])
        self.norm_f = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight          # weight tying
        self.register_buffer("slopes", alibi_slopes(heads), persistent=False)
        self.apply(self._init)
        # GPT-2 style scaled residual init
        for n, p in self.named_parameters():
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, past=None, use_cache=False, key_pad=None):
        """key_pad: optional (B, Tk) bool, True = padding key (left-padding for batched generation)."""
        b, t = idx.shape
        past_len = 0 if past is None else past[0][0].shape[2]
        bias = alibi_causal_bias(self.slopes, t, past_len + t, idx.device)
        if key_pad is not None:
            # finite mask so fully-padded rows never produce NaN
            bias = bias + key_pad.view(b, 1, 1, -1).float() * (-3e4)
        x = self.drop(self.wte(idx))
        presents = []
        for i, blk in enumerate(self.blocks):
            if self.training and USE_GRAD_CHECKPOINTING and not use_cache:
                x, _ = torch.utils.checkpoint.checkpoint(
                    blk, x, bias, None, False, use_reentrant=False
                )
            else:
                x, pr = blk(x, bias, None if past is None else past[i], use_cache)
                presents.append(pr)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.float().view(-1, logits.size(-1)),
                targets.view(-1),
            )
        return logits, loss, (presents if use_cache else None)




In [31]:
print("ALiBi tokenizer active, vocab", tokenizer.vocab_size)


ALiBi tokenizer active, vocab 32000


### Phase content (`p2`) – cells [24:38] from source


# Phase 2

In [32]:
import os, json, pickle, random, math, time
from pathlib import Path
from dataclasses import fields
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# IMPORTANT: point these to your actual Phase-1 LEGAL model artifacts.
# Example only: change this directory to the folder containing your LegalLM.
LEGAL_MODEL_DIR = Path("/kaggle/working/cs781_phase1")
# MODEL_CONFIG_PATH = LEGAL_MODEL_DIR / "config.json"
MODEL_WEIGHTS_PATH = LEGAL_MODEL_DIR / "legal_lm_alibi_gqa.pt"
TOKENIZER_STATE_PATH = LEGAL_MODEL_DIR / "legal_bpe_tokenizer_ws.pkl"

OUTPUT_DIR = Path("/kaggle/working/lner_full_finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DOCS = None             # None = use all available annotated documents
VALIDATION_FRACTION = 0.10
EPOCHS = 4
BATCH_SIZE = 4              # reduce to 1 or 2 if GPU memory is limited
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WINDOW_OVERLAP = 64

In [33]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold1-gold.json"]
VAL_FILES = ["Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json"]

In [34]:
DATA_DIR = Path("/kaggle/working/Phase_2_LNER")
TRAIN_DIR = DATA_DIR / "Train"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def download_lner_file(filename):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename=f"Phase_2_LNER/Train/{filename}",
        local_dir="/kaggle/working/hf_legal_data"
    )

paths = {
    "fold1_text": download_lner_file("fold1-text.json"),
    "fold1_gold": download_lner_file("fold1-gold.json"),
    "fold2_text": download_lner_file("fold2-text.json"),
    "fold2_gold": download_lner_file("fold2-gold.json"),
    "labels": download_lner_file("labels.txt"),
}

with open(paths["fold1_text"], encoding="utf-8") as f:
    text1 = json.load(f)
with open(paths["fold1_gold"], encoding="utf-8") as f:
    gold1 = json.load(f)
with open(paths["fold2_text"], encoding="utf-8") as f:
    text2 = json.load(f)
with open(paths["fold2_gold"], encoding="utf-8") as f:
    gold2 = json.load(f)

with open(paths["labels"], encoding="utf-8") as f:
    ENTITY_TYPES = [line.strip() for line in f if line.strip()]

print("Entity types:", ENTITY_TYPES)
print("Fold 1 documents:", len(text1), "annotations:", len(gold1))
print("Fold 2 documents:", len(text2), "annotations:", len(gold2))

Entity types: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1 documents: 35 annotations: 35
Fold 2 documents: 35 annotations: 35


In [35]:

import os, json, math, random, time, copy, zipfile, inspect
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WIN, STRIDE = 512, 256
OUT = "/kaggle/working/phase2-alibi-gqa"
os.makedirs(OUT, exist_ok=True)
CKPT_DIR = os.path.join(OUT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
CAUSAL = False
BOS, PAD = 2, 0

def dl(name):
    return hf_hub_download(HF_DATASET_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", token=HF_TOKEN)

LABELS = [x.strip() for x in open(dl("Train/labels.txt"), encoding="utf-8") if x.strip()]
BIO = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
l2i = {l: i for i, l in enumerate(BIO)}
NL = len(BIO)
print("Labels:", LABELS)
print("Number of BIO classes:", NL)

def load_fold(k):
    texts = json.load(open(dl(f"Train/fold{k}-text.json"), encoding="utf-8"))
    raw_gold = json.load(open(dl(f"Train/fold{k}-gold.json"), encoding="utf-8"))
    texts = {str(d): t for d, t in texts.items()}
    entities = {}
    for d in texts:
        rec = raw_gold.get(d, raw_gold.get(str(d), []))
        if isinstance(rec, dict) and "entities" in rec:
            rec = rec["entities"]
        out = []
        for e in rec:
            if isinstance(e, dict):
                s = e.get("start", e.get("start_offset"))
                en = e.get("end", e.get("end_offset"))
                lab = e.get("label", e.get("type"))
                txt = e.get("text")
            else:
                s, en, lab = e[:3]
                txt = e[3] if len(e) > 3 else None
            if lab not in LABELS:
                raise ValueError(f"Unknown label {lab!r} in document {d}")
            out.append((int(s), int(en), lab, txt))
        entities[d] = out
    return texts, entities

text1, ent1 = load_fold(1)
text2, ent2 = load_fold(2)
print("Train documents:", len(text1))
print("Validation documents:", len(text2))


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Number of BIO classes: 25


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Train documents: 35
Validation documents: 35


In [36]:


def encode_with_offsets(text, tok):
    ids, offsets = [], []
    pos = 0

    for match in re.finditer(r"\S+", text):
        word = match.group()
        start, end = match.span()

        word_ids = tok.encode_word(word)
        raw = (" " + word).encode("utf-8")
        byte_cursor = 0

        for tid in word_ids:
            token_bytes = tok.itos[tid]
            if not isinstance(token_bytes, bytes):
                ids.append(tid)
                offsets.append((start, end))
                continue

            b0 = byte_cursor
            b1 = b0 + len(token_bytes)
            byte_cursor = b1

            # Ignore the artificial leading space added by encode_word().
            a0 = max(b0, 1)
            a1 = min(b1, len(raw))
            if a0 >= a1:
                continue

            # Convert byte offsets within the word to character offsets.
            char_start = start + len(raw[1:a0].decode("utf-8", errors="ignore"))
            char_end = start + len(raw[1:a1].decode("utf-8", errors="ignore"))

            ids.append(tid)
            offsets.append((char_start, char_end))

        pos = end

    return ids, offsets

def token_labels(entities, offs):
    y = [0] * len(offs)
    exact = 0
    for s, e, lab, *_ in entities:
        idx = [i for i, (a, b) in enumerate(offs) if a < b and a < e and b > s]
        if not idx:
            continue
        if offs[idx[0]][0] == s and offs[idx[-1]][1] == e:
            exact += 1
        y[idx[0]] = l2i[f"B-{lab}"]
        for i in idx[1:]:
            y[i] = l2i[f"I-{lab}"]
    return y, exact

def build_docs(texts, entities):
    docs = {}
    for d, text in tqdm(texts.items(), desc="Tokenizing"):
        ids, offs = encode_with_offsets(text, tokenizer)
        y, exact = token_labels(entities[d], offs)
        docs[d] = {
            "text": text, "ids": ids, "offs": offs, "y": y,
            "gold": [(s, e, lab) for s, e, lab, *_ in entities[d]],
            "exact": exact
        }
    return docs

train_docs = build_docs(text1, ent1)
val_docs = build_docs(text2, ent2)

n_gold_train = sum(len(x["gold"]) for x in train_docs.values())
n_exact_train = sum(x["exact"] for x in train_docs.values())
print(f"Train entities aligned to exact token boundaries: {n_exact_train}/{n_gold_train}")
print("Train tokens:", sum(len(x["ids"]) for x in train_docs.values()))
print("Validation tokens:", sum(len(x["ids"]) for x in val_docs.values()))


Tokenizing:   0%|          | 0/35 [00:00<?, ?it/s]

Tokenizing:   0%|          | 0/35 [00:00<?, ?it/s]

Train entities aligned to exact token boundaries: 2801/4120
Train tokens: 213148
Validation tokens: 215644


In [37]:

def alibi_causal_bias(slopes, tq, tk, device):
    qpos = torch.arange(tk - tq, tk, device=device)[:, None]
    kpos = torch.arange(tk, device=device)[None, :]
    dist = (qpos - kpos).float()
    bias = -slopes.to(device).view(1, -1, 1, 1) * dist.clamp(min=0).view(1, 1, tq, tk)
    return bias.masked_fill((dist < 0).view(1, 1, tq, tk), float("-inf"))

def alibi_bidirectional_bias(slopes, length, device):
    pos = torch.arange(length, device=device)
    dist = (pos[:, None] - pos[None, :]).abs().float()
    return -slopes.to(device).view(1, -1, 1, 1) * dist.view(1, 1, length, length)

def viterbi(logp):
    T = logp.shape[0]
    if T == 0:
        return []
    allow = np.zeros((NL, NL), dtype=bool)
    for p in range(NL):
        for c in range(NL):
            cur = BIO[c]
            allow[p, c] = not (
                cur.startswith("I-") and
                not (BIO[p] != "O" and BIO[p][2:] == cur[2:])
            )
    trans = np.where(allow, 0.0, -1e9)
    start = np.array([0.0 if not BIO[c].startswith("I-") else -1e9 for c in range(NL)])
    dp = start + logp[0]
    back = np.zeros((T, NL), dtype=np.int64)
    for t in range(1, T):
        sc = dp[:, None] + trans
        back[t] = sc.argmax(0)
        dp = sc.max(0) + logp[t]
    path = [int(dp.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

def tags_to_spans(tags, offs):
    spans, cur = [], None
    for tag, (s, e) in zip(tags, offs):
        if s == e:
            continue
        name = BIO[int(tag)]
        if name == "O":
            if cur is not None:
                spans.append(tuple(cur))
                cur = None
        elif name.startswith("B-") or cur is None or cur[2] != name[2:]:
            if cur is not None:
                spans.append(tuple(cur))
            cur = [s, e, name[2:]]
        else:
            cur[1] = e
    if cur is not None:
        spans.append(tuple(cur))
    return spans

def strict_prf(gold, pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P:
            if x in G:
                tp[x[2]] += 1
            else:
                fp[x[2]] += 1
        for x in G - P:
            fn[x[2]] += 1
    per = {}
    for lab in LABELS:
        den_p = tp[lab] + fp[lab]
        den_r = tp[lab] + fn[lab]
        if den_p + den_r == 0:
            continue
        p = tp[lab] / den_p if den_p else 0.0
        r = tp[lab] / den_r if den_r else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        per[lab] = (p, r, f, tp[lab] + fn[lab])
    mean = lambda i: float(np.mean([v[i] for v in per.values()])) if per else 0.0
    return {"P": mean(0), "R": mean(1), "F1": mean(2), "per": per}

def error_analysis(gold, pred):
    c = Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for p in P:
            if p in G:
                c["exact"] += 1
            elif any((g[0], g[1]) == (p[0], p[1]) for g in G):
                c["label_confusion"] += 1
            elif any(g[2] == p[2] and g[0] < p[1] and p[0] < g[1] for g in G):
                c["boundary_error"] += 1
            else:
                c["false_positive"] += 1
        for g in G - P:
            if not any((p[0], p[1]) == (g[0], g[1]) for p in P) and not any(
                p[0] < g[1] and g[0] < p[1] for p in P
            ):
                c["missed_without_overlap"] += 1
    return dict(c)

def show(res, title=""):
    print(f"{title} macro P={res['P']:.4f} R={res['R']:.4f} F1={res['F1']:.4f}")
    for lab, (p, r, f, n) in res["per"].items():
        print(f"  {lab:12s} P={p:.3f} R={r:.3f} F1={f:.3f} gold={n}")

gold_v = {d: x["gold"] for d, x in val_docs.items()}
oracle = {d: tags_to_spans(x["y"], x["offs"]) for d, x in val_docs.items()}
show(strict_prf(gold_v, oracle), "ORACLE")


ORACLE macro P=0.6599 R=0.6599 F1=0.6599
  APP          P=0.715 R=0.715 F1=0.715 gold=200
  RESP         P=0.721 R=0.721 F1=0.721 gold=190
  A.COUNSEL    P=0.488 R=0.488 F1=0.488 gold=125
  R.COUNSEL    P=0.526 R=0.526 F1=0.526 gold=97
  JUDGE        P=0.459 R=0.459 F1=0.459 gold=122
  WIT          P=0.725 R=0.725 F1=0.725 gold=120
  AUTH         P=0.782 R=0.782 F1=0.782 gold=468
  COURT        P=0.751 R=0.751 F1=0.751 gold=558
  STAT         P=0.641 R=0.641 F1=0.641 gold=735
  PREC         P=0.744 R=0.744 F1=0.744 gold=559
  DATE         P=0.629 R=0.629 F1=0.629 gold=762
  CASENO       P=0.739 R=0.739 F1=0.739 gold=337


In [38]:

def make_windows(n, win=WIN, step=STRIDE):
    if n <= win:
        return [(0, n)]
    starts = list(range(0, n - win + 1, step))
    if starts[-1] + win < n:
        starts.append(n - win)
    return [(s, s + win) for s in starts]

def make_items(docs, step=STRIDE):
    items = []
    for d, x in docs.items():
        for a, b in make_windows(len(x["ids"]), WIN, step):
            items.append(([BOS] + x["ids"][a:b], [-100] + x["y"][a:b]))
    return items

def collate(batch):
    L = max(len(ids) for ids, _ in batch)
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    y = torch.full((len(batch), L), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.bool)
    for k, (seq, labels) in enumerate(batch):
        ids[k, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        y[k, :len(labels)] = torch.tensor(labels, dtype=torch.long)
        attn[k, :len(seq)] = True
    return ids, y, attn

class Adapter(nn.Module):
    def __init__(self, dim, r=64):
        super().__init__()
        self.down = nn.Linear(dim, r)
        self.up = nn.Linear(r, dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))

class Backbone(nn.Module):
    """Feature extractor for the Phase-1 ALiBi-GQA LegalLM."""
    def __init__(self, lm, causal=False, adapter_dim=0):
        super().__init__()
        self.lm = lm
        self.causal = causal
        self.dim = lm.wte.embedding_dim
        self.adapters = nn.ModuleList(
            [Adapter(self.dim, adapter_dim) for _ in lm.blocks]
        ) if adapter_dim else None

    def forward(self, ids, attn):
        x = self.lm.drop(self.lm.wte(ids))
        B, T = ids.shape
        if self.causal:
            bias = alibi_causal_bias(self.lm.slopes, T, T, ids.device)
        else:
            bias = alibi_bidirectional_bias(self.lm.slopes, T, ids.device)

        pad_bias = (~attn).view(B, 1, 1, T).to(bias.dtype) * -3e4
        bias = bias + pad_bias

        for i, blk in enumerate(self.lm.blocks):
            x, _ = blk(x, bias, past=None, use_cache=False)
            if self.adapters is not None:
                x = self.adapters[i](x)
        return self.lm.norm_f(x)

class NERModel(nn.Module):
    def __init__(self, bb, head="linear", drop=0.1):
        super().__init__()
        self.backbone = bb
        self.drop = nn.Dropout(drop)
        d = bb.dim
        if head == "linear":
            self.head = nn.Linear(d, NL)
        elif head == "mlp":
            self.head = nn.Sequential(
                nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, NL)
            )
        else:
            raise ValueError(f"Unknown NER head: {head}")

    def forward(self, ids, attn):
        return self.head(self.drop(self.backbone(ids, attn)))


In [39]:

def get_lm_config(ck):
    cfg = ck.get("config", ck.get("cfg"))
    if cfg is None:
        raise KeyError("Checkpoint has neither 'config' nor 'cfg'.")
    allowed = set(inspect.signature(LegalLM.__init__).parameters) - {"self"}
    cfg = {k: v for k, v in dict(cfg).items() if k in allowed}
    required = {"vocab_size", "dim", "layers", "heads", "kv_heads", "block_size"}
    missing = required - set(cfg)
    if missing:
        raise ValueError(f"Missing model config fields: {sorted(missing)}")
    return cfg

def load_lm(path=None):
    ck = torch.load(path or MODEL_PATH, map_location="cpu", weights_only=False)
    cfg = get_lm_config(ck)
    lm = LegalLM(**cfg)
    state = ck.get("model", ck.get("model_state_dict", ck.get("state_dict")))
    if state is None:
        raise KeyError("Checkpoint has no model weights.")
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
    lm.load_state_dict(state, strict=True)
    return lm

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.s = alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.s

class LoRMALinear(nn.Module):
    """Output-side multiplicative adaptation: y' = y + s*B*A*y."""
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.s = alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + (y @ self.A.t() @ self.B.t()) * self.s

TARGETS = {"q", "kv", "proj", "w1", "w3", "w2"}

def inject(lm, cls, r=8, alpha=16):
    for blk in lm.blocks:
        for mod in list(blk.modules()):
            for name, child in list(mod.named_children()):
                if isinstance(child, nn.Linear) and name in TARGETS:
                    setattr(mod, name, cls(child, r, alpha))

def build(kind, head="linear", r=8, k_top=3, path=None):
    lm = load_lm(path)
    bb = Backbone(lm, causal=CAUSAL, adapter_dim=64 if kind == "adapter" else 0)

    if kind in ("lora", "lorma"):
        for p in lm.parameters():
            p.requires_grad = False
        inject(lm, LoRALinear if kind == "lora" else LoRMALinear, r, 2 * r)

    elif kind == "adapter":
        for p in lm.parameters():
            p.requires_grad = False
        # Adapter parameters are separate and remain trainable.

    elif kind == "partial":
        for p in lm.parameters():
            p.requires_grad = False
        for blk in lm.blocks[-k_top:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.norm_f.parameters():
            p.requires_grad = True

    elif kind == "full":
        pass  # All backbone parameters remain trainable.

    else:
        raise ValueError(f"Unknown training method: {kind}")

    return NERModel(bb, head).to(DEVICE)

def n_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


In [40]:

@torch.no_grad()
def predict_docs(model, docs, decode="viterbi", bs=8):
    model.eval()
    out = {}
    for d, x in tqdm(docs.items(), desc="Predicting"):
        n = len(x["ids"])
        if n == 0:
            out[d] = []
            continue

        acc = np.zeros((n, NL), dtype=np.float32)
        cnt = np.zeros(n, dtype=np.float32)
        wins = make_windows(n, WIN, WIN // 2)

        for i in range(0, len(wins), bs):
            chunk = wins[i:i + bs]
            batch = [([BOS] + x["ids"][a:b], [-100] * (b - a + 1)) for a, b in chunk]
            ids, _, attn = collate(batch)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=AMP_DTYPE,
                enabled=bool(USE_AMP and DEVICE.type == "cuda")
            ):
                logits = model(ids.to(DEVICE), attn.to(DEVICE))
            logp = logits.float().log_softmax(-1).cpu().numpy()
            for k, (a, b) in enumerate(chunk):
                acc[a:b] += logp[k, 1:b - a + 1]
                cnt[a:b] += 1

        logp = acc / np.maximum(cnt[:, None], 1)
        tags = viterbi(logp) if decode == "viterbi" else logp.argmax(-1).tolist()
        out[d] = tags_to_spans(tags, x["offs"])
    return out

def evaluate(model, docs, decode="viterbi"):
    pred = predict_docs(model, docs, decode=decode)
    gold = {d: x["gold"] for d, x in docs.items()}
    return strict_prf(gold, pred), pred


In [41]:
from tqdm.auto import tqdm
def class_weights(docs, power=0.5, cap=10.0):
    c = np.bincount(np.concatenate([x["y"] for x in docs.values()]), minlength=NL).astype(float) + 1
    w = np.clip((c[0] / c) ** power, 0, cap); w[0] = 1.0
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train(model, name, epochs, lr, head_lr=1e-3, bs=8, wd=0.01, cw=None, teacher=None, T=2.0, alpha=0.5, warm=0.1, snapshot_eval=True, resume_path=None):
    # ADD: Checkpoint directory creation
    ckpt_dir = f"{OUT}/checkpoints/{name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    items = make_items(train_docs); steps = epochs * math.ceil(len(items) / bs)
    bb_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head")]
    hd_p = [p for n, p in model.named_parameters() if n.startswith("head")]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": lr}, {"params": hd_p, "lr": head_lr}], weight_decay=wd)
    base = [lr, head_lr]; scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time(); hist = []
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time()
    hist = []
    start_epoch = 0  # ADD: Track starting epoch

    # ADD: Load checkpoint if resume_path is provided
    if resume_path and os.path.exists(resume_path):
        print(f"--> Resuming from: {resume_path}")
        ckpt = torch.load(resume_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"], strict=False)
        opt.load_state_dict(ckpt["optimizer_state"])
        if USE_AMP and AMP_DTYPE == torch.float16 and "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        step = ckpt["step"]
        best = ckpt["best_f1"]
        best_state = ckpt["best_state"]
        hist = ckpt.get("hist", [])
    for ep in tqdm(range(start_epoch, epochs)):
        model.train(); random.shuffle(items); tl = 0
        for i in range(0, len(items), bs):
            f = (step + 1) / (warm * steps) if step < warm * steps else 0.5 * (1 + math.cos(math.pi * (step - warm * steps) / max(1, steps - warm * steps)))
            for g, b0 in zip(opt.param_groups, base): g["lr"] = b0 * f
            ids, y, m = [t.to(DEVICE) for t in collate(items[i:i + bs])]
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg = model(ids, m)
                loss = F.cross_entropy(lg.float().reshape(-1, NL), y.reshape(-1), weight=cw, ignore_index=-100)
                if teacher is not None:
                    with torch.no_grad(): tl_ = teacher(ids, m).float()
                    sel = y != -100
                    kd = F.kl_div((lg.float()[sel] / T).log_softmax(-1), (tl_[sel] / T).softmax(-1), reduction="batchmean") * T * T
                    loss = alpha * loss + (1 - alpha) * kd
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1; tl += loss.item()
        res, _ = evaluate(model, val_docs); hist.append((ep + 1, tl / math.ceil(len(items) / bs), res["F1"]))
        print(f"[{name}] ep {ep+1}/{epochs} loss {hist[-1][1]:.4f} | val strict macro P {res['P']:.3f} R {res['R']:.3f} F1 {res['F1']:.4f}")
        if res["F1"] > best: best = res["F1"]; best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
        # inside epoch loop, after calculating `res`:
        if res["F1"] > best:
            best = res["F1"]
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
            # ADD: Save best weights separately
            torch.save(best_state, f"{ckpt_dir}/{name}_best_weights.pt")

        # ADD: Save latest checkpoint for resuming
        ckpt_payload = {
            "epoch": ep,
            "step": step,
            "model_state": {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
            "optimizer_state": opt.state_dict(),
            "scaler_state": scaler.state_dict() if USE_AMP else None,
            "best_f1": best,
            "best_state": best_state,
            "hist": hist,
        }
        torch.save(ckpt_payload, f"{ckpt_dir}/latest_checkpoint.pt")
    model.load_state_dict(best_state, strict=False)
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else float("nan")
    return dict(best_val_f1=best, train_sec=time.time() - t0, peak_gb=peak, hist=hist)

In [42]:
test_texts = json.load(open(dl("Test/fold3-text.json"), encoding="utf-8")); test_texts = {str(k): v for k, v in test_texts.items()}
test_docs = {}
for d, t in test_texts.items():
    ids, offs = encode_with_offsets(t, tokenizer); test_docs[d] = dict(text=t, ids=ids, offs=offs, y=[0] * len(ids), gold=[])

def write_test(model, name):
    pred = predict_docs(model, test_docs)
    js = {d: [[s, e, l, test_texts[d][s:e]] for s, e, l in sp] for d, sp in pred.items()}
    p = f"{OUT}/prediction_{name}.json"; json.dump(js, open(p, "w"), ensure_ascii=False)
    with zipfile.ZipFile(f"{OUT}/submission_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(p, "prediction.json")

CW = class_weights(train_docs)
RESULTS, MODELS = {}, {}
def run(name, kind, epochs, lr, head_lr=1e-3, head="linear", cw=CW, resume = False, **kw):
    model = build(kind, head=head, **kw); tot, tr = n_params(model)
    print(f"\n=== {name}: total {tot/1e6:.1f}M params, trainable {tr/1e6:.2f}M ({100*tr/tot:.1f}%)")
    ckpt_path = f"{OUT}/checkpoints/{name}/latest_checkpoint.pt"
    resume_path = ckpt_path if (resume and os.path.exists(ckpt_path)) else None

    # CHANGE: Pass resume_path to train()
    info = train(model, name, epochs, lr, head_lr, cw=cw, resume_path=resume_path)
    t0 = time.time(); res, pred = evaluate(model, val_docs); info["infer_sec"] = time.time() - t0
    info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
    RESULTS[name] = info; MODELS[name] = model; write_test(model, name); show(res, f"[{name}] FINAL VAL"); return model

# Exp 1: full fine-tuning (also the KD teacher)
run("full_ft", "full", epochs=4, lr=5e-5, resume = True)
# Exp 2: partial / layer-wise (top-3 blocks)
# run("partial_top3", "partial", epochs=10, lr=1e-4, k_top=3, resume = True)
# Exp 3: LoRA
run("lora_r8", "lora", epochs=4, lr=1e-3, r=8, resume = True)
# Exp 4: LoRMA
run("lorma_r8", "lorma", epochs=4, lr=1e-3, r=8, resume = True)
# Exp 5: Houlsby-style adapters
# run("adapter64", "adapter", epochs=15, lr=1e-3, resume = True)

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]


=== full_ft: total 102.5M params, trainable 102.47M (100.0%)
--> Resuming from: /kaggle/working/phase2-alibi-gqa/checkpoints/full_ft/latest_checkpoint.pt


0it [00:00, ?it/s]

Predicting:   0%|          | 0/35 [00:00<?, ?it/s]

Predicting:   0%|          | 0/35 [00:00<?, ?it/s]

[full_ft] FINAL VAL macro P=0.2627 R=0.2375 F1=0.2421
  APP          P=0.160 R=0.120 F1=0.137 gold=200
  RESP         P=0.141 R=0.053 F1=0.077 gold=190
  A.COUNSEL    P=0.100 R=0.080 F1=0.089 gold=125
  R.COUNSEL    P=0.039 R=0.062 F1=0.048 gold=97
  JUDGE        P=0.204 R=0.262 F1=0.229 gold=122
  WIT          P=0.227 R=0.083 F1=0.122 gold=120
  AUTH         P=0.350 R=0.184 F1=0.241 gold=468
  COURT        P=0.575 R=0.634 F1=0.603 gold=558
  STAT         P=0.406 R=0.351 F1=0.377 gold=735
  PREC         P=0.174 R=0.224 F1=0.196 gold=559
  DATE         P=0.547 R=0.563 F1=0.555 gold=762
  CASENO       P=0.230 R=0.234 F1=0.232 gold=337

=== lora_r8: total 103.7M params, trainable 1.27M (1.2%)
--> Resuming from: /kaggle/working/phase2-alibi-gqa/checkpoints/lora_r8/latest_checkpoint.pt


... (5 outputs left out)

Predicting:   0%|          | 0/35 [00:00<?, ?it/s]

Predicting:   0%|          | 0/35 [00:00<?, ?it/s]

[lorma_r8] FINAL VAL macro P=0.2447 R=0.2531 F1=0.2430
  APP          P=0.126 R=0.090 F1=0.105 gold=200
  RESP         P=0.057 R=0.032 F1=0.041 gold=190
  A.COUNSEL    P=0.100 R=0.136 F1=0.115 gold=125
  R.COUNSEL    P=0.038 R=0.031 F1=0.034 gold=97
  JUDGE        P=0.105 R=0.205 F1=0.139 gold=122
  WIT          P=0.232 R=0.183 F1=0.205 gold=120
  AUTH         P=0.375 R=0.199 F1=0.260 gold=468
  COURT        P=0.561 R=0.629 F1=0.593 gold=558
  STAT         P=0.394 R=0.396 F1=0.395 gold=735
  PREC         P=0.161 R=0.250 F1=0.196 gold=559
  DATE         P=0.506 R=0.568 F1=0.536 gold=762
  CASENO       P=0.282 R=0.318 F1=0.298 gold=337


NERModel(
  (backbone): Backbone(
    (lm): LegalLM(
      (wte): Embedding(32000, 768)
      (drop): Dropout(p=0.0, inplace=False)
      (blocks): ModuleList(
        (0-11): 12 x Block(
          (n1): RMSNorm()
          (n2): RMSNorm()
          (attn): ALiBiGQA(
            (q): LoRMALinear(
              (base): Linear(in_features=768, out_features=768, bias=False)
            )
            (kv): LoRMALinear(
              (base): Linear(in_features=768, out_features=768, bias=False)
            )
            (proj): LoRMALinear(
              (base): Linear(in_features=768, out_features=768, bias=False)
            )
          )
          (mlp): SwiGLU(
            (w1): LoRMALinear(
              (base): Linear(in_features=768, out_features=2048, bias=False)
            )
            (w3): LoRMALinear(
              (base): Linear(in_features=768, out_features=2048, bias=False)
            )
            (w2): LoRMALinear(
              (base): Linear(in_features=2048, out_featu

In [43]:
# ═══ PHASE 2 – bundle everything (weights, resume ckpts, results, preds, submissions, plots) ═══
import zipfile
import os
import shutil
!ls {OUT}/checkpoints
# ═══ PHASE 2 – bundle everything (ckpts, results, probs, submissions) ═══
P2_OUT = OUT
P2_BUNDLE = f"{P2_OUT}/phase2_bundle.zip"
with zipfile.ZipFile(P2_BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(P2_OUT):
        for f in files:
            if f == os.path.basename(P2_BUNDLE):
                continue
            p = os.path.join(root, f)
            z.write(p, arcname=os.path.relpath(p, P2_OUT))
print(f"wrote {P2_BUNDLE} ({os.path.getsize(P2_BUNDLE)/2**30:.2f} GB)")



full_ft  lora_r8  lorma_r8
wrote /kaggle/working/phase2-alibi-gqa/phase2_bundle.zip (1.35 GB)


---
## Architecture: `gqa_conv`



In [44]:
# ── Activate backbone: gqa_conv ──────────────────────────────────────────────
BACKBONE = "gqa_conv"
TOKENIZER_PATH = TOK_PATHS["gqa_conv"]
PHASE1_CKPT = CKPTS["gqa_conv"]
MODEL_PATH = PHASE1_CKPT
CKPT_PATH = PHASE1_CKPT
if "OUT_DIR" not in dir() or not OUT_DIR:
    OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"[active] BACKBONE={BACKBONE}")
print(f"[active] TOKENIZER_PATH={TOKENIZER_PATH}  exists={os.path.exists(TOKENIZER_PATH)}")
print(f"[active] PHASE1_CKPT={PHASE1_CKPT}  exists={os.path.exists(PHASE1_CKPT)}")


[active] BACKBONE=gqa_conv
[active] TOKENIZER_PATH=/kaggle/working/cs781_phase1/legal_bpe_gqa_conv.json  exists=True
[active] PHASE1_CKPT=/kaggle/working/cs781_phase1/legal_lm_gqa_conv.pt  exists=True


### Phase-1 classes for `gqa_conv` (tokenizer + model, needed to reload checkpoints)

These cells are taken from Phase 1 of the same source notebook. They define `BPETokenizer`,
the model stack (`LegalLM` / `DualPathLM` / …), and `load_model`. Internal bodies are unchanged.


In [45]:
import os, sys, re, gc, json, math, time, heapq, random, hashlib, unicodedata, zipfile, subprocess
import multiprocessing as mp
from collections import Counter, defaultdict, deque
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint
from tqdm.auto import tqdm

try:
    import ijson
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ijson"])
    import ijson
from huggingface_hub import hf_hub_download, login

# ------------------------------- Configuration -------------------------------
SEED = 42
HF_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

SOURCE_TOKENS = {"acts": 15_000_000, "supreme_court": 35_000_000, "high_courts": 35_000_000, "tribunals": 25_000_000}
SAMPLE_MIX = {"acts": 1.0, "supreme_court": 1.0, "high_courts": 0.30, "tribunals": 0.40}
VAL_TOKENS_PER_SOURCE = 500_000
VAL_MOD = 50
BYTES_PER_TOKEN = 4.0
NUM_WORKERS = 4

VOCAB_SIZE = 30000
TOK_CHARS_PER_SOURCE = 4_000_000

# Model: Decoder-only GQA + RoPE + Dual-path FFN
BLOCK_SIZE = 1024
N_EMBD = 768
N_LAYER = 8
N_HEAD = 12
KV_HEADS = 4                          # Grouped-Query Attention: 12 Q-heads share 4 KV-heads (3x KV reduction)
QK_NORM = True                        # QK-Norm via RMSNorm per head

FFN_MULT = 3                          # SwiGLU expansion multiplier
CONV_K = 4                            # Causal depthwise 1D conv kernel size
DUAL_PATH = True                      # Enable Dual-Path FFN (SwiGLU + lambda * DWConv)
LAMBDA_PER_CHANNEL = True             # Learnable lambda vector per channel (vs single scalar)
LAMBDA_INIT = 0.0                     # Initial sigmoid(a) = 0.5

# Training / Optimization Schedule
P_DOC_START = 0.5                     # 50% windows start at doc beginning, 50% mid-document
TOKENS_PER_STEP = 65_536
MICRO_TOKENS = 16_384
STAGES = [(1024, 0.50)]  # Context curriculum
TRAIN_TOKENS = 150_000_000
LR, MIN_LR, WEIGHT_DECAY, WARMUP_STEPS = 6e-4, 6e-5, 0.1, 100
EVAL_EVERY, EVAL_WINDOWS, LOG_EVERY, CKPT_EVERY = 10, 48, 10, 50
MAX_TRAIN_HOURS = 8.0
USE_GRAD_CKPT = False
DO_TRAIN = True

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

OUT_DIR = "/kaggle/working/cs781_phase1" if os.path.isdir("/kaggle/working") else "./cs781_phase1"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
TOKENIZER_PATH = os.path.join(OUT_DIR, "legal_bpe_v2.json")
MODEL_PATH = os.path.join(OUT_DIR, "legal_lm_v3_fixed.pt")
CKPT_PATH = os.path.join(OUT_DIR, "latest_checkpoint_v3_fixed.pt")

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def init_distributed():
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        dist.init_process_group(backend="nccl")
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
        return device, local_rank, True
    elif torch.cuda.device_count() > 1:
        # Fallback multi-GPU single-process using DataParallel
        return torch.device("cuda:0"), 0, False
    else:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu"), 0, False

DEVICE, LOCAL_RANK, IS_DISTRIBUTED = init_distributed()
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    if HF_TOKEN: login(token=HF_TOKEN)
except Exception as e:
    print("HF login skipped:", e)
# FIX: point back to the resolved gqa_conv artifacts
TOKENIZER_PATH = TOK_PATHS["gqa_conv"]
MODEL_PATH = CKPTS["gqa_conv"]
CKPT_PATH = MODEL_PATH
print("gqa_conv tokenizer:", TOKENIZER_PATH, "| ckpt:", MODEL_PATH)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


gqa_conv tokenizer: /kaggle/working/cs781_phase1/legal_bpe_gqa_conv.json | ckpt: /kaggle/working/cs781_phase1/legal_lm_gqa_conv.pt


In [46]:
# ------------------------------- 1. Tokenizer -------------------------------
_QUOTES = str.maketrans({"“": '"', "”": '"', "„": '"', "‘": "'", "’": "'", "‚": "'", "`": "'", "´": "'",
                         "–": "-", "—": "-", "−": "-", "\u00ad": ""})
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
_WS = re.compile(r"[ \t\u00a0\u1680\u2000-\u200b\u202f\u205f\u3000\ufeff]+")
_DOTS = re.compile(r"\.{4,}|(?:\. ){4,}")
_RULE = re.compile(r"[_\-=*~#]{3,}")
_PAGE = re.compile(r"(?mi)^[ \t]*(?:page\s+)?\d{1,4}[ \t]*$\n?")
_HYPH = re.compile(r"(?<=[a-z])-\n(?=[a-z])")
_WRAP = re.compile(r"(?<=[a-z,;])\n(?=[a-z(])")

def clean_text(t):
    t = unicodedata.normalize("NFKC", str(t)).replace("\r\n", "\n").replace("\r", "\n")
    t = t.translate(_QUOTES)
    t = _CTRL.sub(" ", t)
    t = _HYPH.sub("", t)
    t = _DOTS.sub("...", t)
    t = _RULE.sub(" ", t)
    t = _PAGE.sub("", t)
    t = _WS.sub(" ", t)
    t = re.sub(r" *\n *", "\n", t)
    t = _WRAP.sub(" ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def keep_doc(c):
    if len(c) < 600: return False
    head = c[:5000]
    if sum(ch.isalpha() for ch in head) / len(head) < 0.6: return False
    if sum(ord(ch) > 127 for ch in head) / len(head) > 0.2: return False
    w = c.split()
    if len(w) < 100: return False
    w = w[:3000]
    return len(set(w)) / len(w) >= 0.15

def extract_text(rec):
    if isinstance(rec, str): return rec
    if isinstance(rec, dict):
        for k in ("text", "train_text", "content", "body", "judgment", "document"):
            v = rec.get(k)
            if isinstance(v, str) and v.strip(): return v
        for v in rec.values():
            if isinstance(v, (dict, list)):
                s = extract_text(v)
                if s: return s
    if isinstance(rec, list):
        return "\n".join(filter(None, map(extract_text, rec)))
    return ""

def iter_docs(path):
    with open(path, "rb") as f:
        head = f.read(256).lstrip(b"\xef\xbb\xbf \n\r\t")[:1]
        f.seek(0)
        it = (v for _, v in ijson.kvitems(f, "")) if head == b"{" else ijson.items(f, "item")
        for rec in it:
            t = extract_text(rec)
            if t: yield t

In [47]:
class BPETokenizer:
    SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
    _split = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+")

    def __init__(self, vocab_size=12000):
        self.vocab_size = vocab_size
        self.merges = []
        self._reset()

    def _reset(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        self._cache = {}

    def _build(self):
        self.itos = list(self.SPECIALS) + [bytes([i]) for i in range(256)]
        seen = set(self.itos[4:])
        for a, b in self.merges:
            m = a + b
            if m not in seen:
                seen.add(m); self.itos.append(m)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.ranks = {pair: r for r, pair in enumerate(self.merges)}
        self.merged = {pair: pair[0] + pair[1] for pair in self.merges}
        self._cache = {}
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        self.vocab_size = len(self.itos)

    def train(self, texts, max_unique_words=200_000, min_freq=2):
        counts = Counter()
        for t in tqdm(texts, desc="counting pre-tokens"):
            counts.update(self._split.findall(t))
        if len(counts) > max_unique_words:
            counts = Counter(dict(counts.most_common(max_unique_words)))
        words, freqs = [], []
        pair_counts, pair_words = defaultdict(int), defaultdict(set)
        for wid, (w, c) in enumerate(counts.items()):
            toks = [bytes([b]) for b in w.encode("utf-8")]
            words.append(toks); freqs.append(c)
            for p in zip(toks, toks[1:]):
                pair_counts[p] += c; pair_words[p].add(wid)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)
        self.merges = []
        n_target = self.vocab_size - (4 + 256)
        seen = set(bytes([i]) for i in range(256))
        pbar = tqdm(total=n_target, desc="BPE merges")
        while len(self.merges) < n_target and heap:
            best = None
            while heap:
                negc, p = heapq.heappop(heap)
                cur = pair_counts.get(p, 0)
                if cur <= 0: continue
                if cur != -negc:
                    heapq.heappush(heap, (-cur, p)); continue
                best = p; break
            if best is None or pair_counts[best] < min_freq: break
            a, b = best; merged = a + b
            self.merges.append(best); seen.add(merged)
            for wid in list(pair_words.pop(best, ())):
                w, f = words[wid], freqs[wid]
                new, i = [], 0
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        new.append(merged); i += 2
                    else:
                        new.append(w[i]); i += 1
                if len(new) == len(w): continue
                for p in zip(w, w[1:]): pair_counts[p] -= f
                for p in zip(new, new[1:]):
                    pair_counts[p] += f; pair_words[p].add(wid)
                    heapq.heappush(heap, (-pair_counts[p], p))
                words[wid] = new
            pbar.update(1)
        pbar.close()
        self._build()
        return self

    def _word(self, w):
        r = self._cache.get(w)
        if r is not None: return r
        toks = [bytes([b]) for b in w.encode("utf-8")]
        while len(toks) > 1:
            best_i, best_r = -1, 1 << 60
            for i in range(len(toks) - 1):
                r_ = self.ranks.get((toks[i], toks[i + 1]), 1 << 60)
                if r_ < best_r: best_i, best_r = i, r_
            if best_i < 0: break
            toks[best_i:best_i + 2] = [toks[best_i] + toks[best_i + 1]]
        r = [self.stoi[t] for t in toks]
        if len(self._cache) > 500_000: self._cache.clear()
        self._cache[w] = r
        return r

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.bos_id] if add_bos else []
        for ch in self._split.findall(text): ids.extend(self._word(ch))
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return b"".join(self.itos[i] for i in ids if 4 <= i < len(self.itos)).decode("utf-8", errors="replace")

    def save(self, path):
        with open(path, "w") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": [[a.hex(), b.hex()] for a, b in self.merges]}, f)

    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        t = cls(d.get("vocab_size", 30000)); t.merges = [(bytes.fromhex(a), bytes.fromhex(b)) for a, b in d["merges"]]
        t._build(); return t

In [48]:
# ------------------------------- 3. Architecture Modules -------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.w).type_as(x)

class RoPE(nn.Module):
    def __init__(self, hd, max_len, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, hd, 2).float() / hd))
        fr = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", fr.cos(), persistent=False); self.register_buffer("sin", fr.sin(), persistent=False)
    def forward(self, x, start=0):                      # x: (b, heads, t, hd)
        t = x.size(2)
        cos, sin = self.cos[start:start + t].to(x.dtype), self.sin[start:start + t].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((x1 * cos - x2 * sin, x2 * cos + x1 * sin), dim=-1)

class GQAAttention(nn.Module):
    """Grouped-query attention + RoPE (+ QK-norm). The cache stores K/V with only `kv_heads` heads."""
    def __init__(self, c):
        super().__init__()
        d, self.h, self.hk = c["dim"], c["heads"], c["kv_heads"]
        assert self.h % self.hk == 0
        self.hd = d // self.h
        self.q = nn.Linear(d, self.h * self.hd, bias=False)
        self.kv = nn.Linear(d, 2 * self.hk * self.hd, bias=False)
        self.o = nn.Linear(self.h * self.hd, d, bias=False)
        self.qn = RMSNorm(self.hd) if c["qk_norm"] else nn.Identity()
        self.kn = RMSNorm(self.hd) if c["qk_norm"] else nn.Identity()
        self.rope = RoPE(self.hd, c["block_size"])

    def forward(self, x, mask=None, cache=None, start=0):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.h, self.hd).transpose(1, 2)
        k, v = self.kv(x).view(b, t, 2, self.hk, self.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope(self.qn(q), start), self.rope(self.kn(k), start)
        if cache is not None:
            if cache[0] is not None: k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            cache = (k, v)
        r = self.h // self.hk
        if t == 1 and cache is not None:           # decode step: fold query group into length axis
            y = F.scaled_dot_product_attention(q.reshape(b, self.hk, r, self.hd), k, v, attn_mask=mask)
            y = y.reshape(b, self.h, 1, self.hd)
        else:
            k, v = k.repeat_interleave(r, 1), v.repeat_interleave(r, 1)
            if mask is not None: y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
            else:               y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(b, t, self.h * self.hd)), cache

class DualPathFFN(nn.Module):
    """Y = Wo[ SwiGLU(X) + lambda * DWConv(X) ],  lambda = sigmoid(a)   (Wo = self.w2)."""
    def __init__(self, c):
        super().__init__()
        d, self.k, self.dual = c["dim"], c["conv_k"], c["dual_path"]
        h = d * c["ffn_mult"]
        self.w13 = nn.Linear(d, 2 * h, bias=False)          # SwiGLU gate + value
        self.w2 = nn.Linear(h, d, bias=False)               # shared output projection Wo
        if self.dual:
            # depthwise (groups = d) with `ffn_mult` filters per input channel -> h channels
            self.dwconv = nn.Conv1d(d, h, self.k, groups=d, bias=False)
            nn.init.normal_(self.dwconv.weight, 0.0, 0.02 * math.sqrt(d / self.k))
            self.lam_logit = nn.Parameter(torch.full((h,) if c["lambda_per_channel"] else (1,), float(c["lambda_init"])))

    def lam(self):
        return torch.sigmoid(self.lam_logit.float()) if self.dual else None

    def forward(self, x, valid=None, state=None, want_state=False):
        a, g = self.w13(x).chunk(2, dim=-1)
        s = F.silu(a) * g                                    # nonlinear path
        new_state = None
        if self.dual:
            xm = x if valid is None else x * valid[..., None].to(x.dtype)
            xt = xm.transpose(1, 2)
            xt = F.pad(xt, (self.k - 1, 0)) if state is None else torch.cat([state.transpose(1, 2).to(xt.dtype), xt], 2)
            local = self.dwconv(xt).transpose(1, 2)          # local path, (b,t,h)
            if want_state: new_state = xt[:, :, -(self.k - 1):].transpose(1, 2)
            s = s + self.lam().to(s.dtype) * local
        return self.w2(s), new_state

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c["dim"]), RMSNorm(c["dim"])
        self.attn, self.ffn = GQAAttention(c), DualPathFFN(c)

    def forward(self, x, mask=None, cache=None, start=0):
        a_c, f_c = (None, None) if cache is None else (cache[:2], cache[2] if len(cache) > 2 else None)
        h, a_c = self.attn(self.n1(x), mask, a_c, start)
        x = x + h
        valid = mask[:, 0, -1, :] if (cache is not None and mask is not None and x.size(1) > 1) else None
        y, f_c = self.ffn(self.n2(x), valid, f_c, want_state=cache is not None)
        return x + y, (None if cache is None else (*a_c, f_c))

    def train_fwd(self, x): return self.forward(x)[0]

class DualPathLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.block_size = cfg, cfg["block_size"]
        self.wte = nn.Embedding(cfg["vocab_size"], cfg["dim"])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg["layers"])])
        self.ln_f = RMSNorm(cfg["dim"])
        self.lm_head = nn.Linear(cfg["dim"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.wte.weight                                          # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("attn.o.weight") or n.endswith("ffn.w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg["layers"]))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, reduce=True):
        x = self.wte(idx)
        for blk in self.blocks:
            x = torch.utils.checkpoint.checkpoint(blk.train_fwd, x, use_reentrant=False) if (self.training and USE_GRAD_CKPT) else blk(x)[0]
        logits = self.lm_head(self.ln_f(x))
        if targets is None: return logits, None
        l = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100, reduction="none").view(targets.shape)
        if not reduce: return logits, l
        return logits, l.sum() / (targets != -100).sum().clamp_min(1)

    def step(self, idx, caches, start, mask):
        x = self.wte(idx); new = []
        for blk, c in zip(self.blocks, caches):
            x, c = blk(x, mask, c, start); new.append(c)
        return self.lm_head(self.ln_f(x[:, -1:]))[:, 0], new

    def lambda_report(self):
        return [(b.ffn.lam().mean().item(), b.ffn.lam().min().item(), b.ffn.lam().max().item()) if b.ffn.dual else None for b in self.blocks]

def make_cfg(vocab_size):
    return dict(vocab_size=vocab_size, dim=N_EMBD, layers=N_LAYER, heads=N_HEAD, kv_heads=KV_HEADS, block_size=BLOCK_SIZE,
                ffn_mult=FFN_MULT, conv_k=CONV_K, dual_path=DUAL_PATH, lambda_per_channel=LAMBDA_PER_CHANNEL,
                lambda_init=LAMBDA_INIT, qk_norm=QK_NORM)


def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    routed = sum(p.numel() for n, p in m.named_parameters() if ".experts." in n)
    c = m.cfg
    return total, total - routed * (1 - c["topk_experts"] / c["n_routed"])


@torch.inference_mode()
def generate_batch(model, tok, prompts, max_new, temperature, top_p, top_k, rep_pen, target_words):
    """Batched sampling with left-padding on the MLA latent cache. Returns a list of generated id-lists."""
    model.eval()
    dev = next(model.parameters()).device
    b, L = len(prompts), max(len(p) for p in prompts)
    assert L + max_new <= model.block_size, "prompt too long for the context window"
    idx = torch.full((b, L), tok.pad_id, dtype=torch.long)
    valid = torch.zeros(b, L, dtype=torch.bool)
    for i, p in enumerate(prompts):
        idx[i, L - len(p):] = torch.tensor(p); valid[i, L - len(p):] = True
    idx, valid = idx.to(dev), valid.to(dev)
    causal = torch.ones(L, L, dtype=torch.bool, device=dev).tril()
    eye = torch.eye(L, dtype=torch.bool, device=dev)
    mask = causal[None, None] & (valid[:, None, None, :] | eye[None, None])   # pad rows attend to themselves (no NaN)
    V = len(tok.itos)
    seen = torch.zeros(b, V, dtype=torch.bool, device=dev)
    gen = []
    with torch.autocast(device_type=dev.type, dtype=AMP_DTYPE, enabled=dev.type == "cuda"):
        logits, caches = model.step(idx, [(None, None)] * len(model.blocks), 0, mask)
        for n in range(max_new):
            lg = logits.float()
            lg[:, [tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id]] = float("-inf")   # always keep writing text
            if rep_pen != 1.0:
                lg = torch.where(seen, torch.where(lg > 0, lg / rep_pen, lg * rep_pen), lg)
            lg = lg / max(temperature, 1e-5)
            if top_k:
                kth = lg.topk(min(top_k, V), dim=-1).values[:, -1:]
                lg = lg.masked_fill(lg < kth, float("-inf"))
            if top_p < 1.0:
                sl, si = lg.sort(descending=True)
                cp = sl.softmax(-1).cumsum(-1)
                rm = cp - sl.softmax(-1) > top_p
                lg = lg.masked_fill(rm.scatter(1, si, rm), float("-inf"))
            nxt = torch.multinomial(lg.softmax(-1), 1)
            gen.append(nxt); seen.scatter_(1, nxt, True)
            valid = torch.cat([valid, torch.ones(b, 1, dtype=torch.bool, device=dev)], 1)
            if (n + 1) % 32 == 0:
                g = torch.cat(gen, 1).tolist()
                if all(len(tok.decode(r).split()) >= target_words + 8 for r in g): break
            if n + 1 < max_new:
                logits, caches = model.step(nxt, caches, L + n, valid[:, None, None, :])
    return torch.cat(gen, 1).tolist()

In [49]:
TARGET_WORDS, CHUNK_TOKENS, GEN_BATCH = 128, 192, 16
MAX_NEW_TOKENS = 768
TEMPERATURE = 0.8
TOP_P = 0.95
TOP_K = 50
REP_PENALTY = 1.1
tokenizer = BPETokenizer.load(TOKENIZER_PATH)   # FIX: gqa_conv tokenizer before generation
def load_model(path=MODEL_PATH):
    if not os.path.exists(path):
        path = "/kaggle/working/cs781_phase1/latest_checkpoint_v3.pt"
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = DualPathLM(ck["config"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()

def continue_texts(model, prompts, target_words=TARGET_WORDS, bs=GEN_BATCH, rounds=4, progress=True):
    max_ctx = BLOCK_SIZE - MAX_NEW_TOKENS
    ctx = [tokenizer.encode(clean_text(p), add_bos=True)[-max_ctx:] for p in prompts]   # no </s> after the prompt
    gen = [[] for _ in prompts]
    todo = list(range(len(prompts)))
    for _ in range(rounds):
        if not todo: break
        todo.sort(key=lambda i: len(ctx[i]))
        bar = tqdm(range(0, len(todo), bs), desc="generating", disable=not progress)
        for s in bar:
            chunk = todo[s:s + bs]
            outs = generate_batch(model, tokenizer, [ctx[i] for i in chunk], MAX_NEW_TOKENS,
                                  TEMPERATURE, TOP_P, TOP_K, REP_PENALTY, target_words)
            for i, o in zip(chunk, outs):
                gen[i] += o; ctx[i] = (ctx[i] + o)[-max_ctx:]
        todo = [i for i in todo if len(tokenizer.decode(gen[i]).split()) < target_words]
    return [" ".join(tokenizer.decode(g).split()[:target_words]) for g in gen]

model = load_model()
samples = ["The respondent submits that the appeal should be dismissed.",
           "Section 302 of the Indian Penal Code shall apply.",
           "The Court, having considered the evidence, hereby orders"]
try:
    for p, o in zip(samples, continue_texts(model, samples, progress=False)):
        print(f"PROMPT: {p}\n -> [{len(o.split())} words] {o}\n")
except Exception as e:
    print("generation demo skipped:", e)
del model; torch.cuda.empty_cache()


PROMPT: The respondent submits that the appeal should be dismissed.
 -> [128 words] The respondent submits that the order be sent to respondent no. 1 under Section 2(2) of the West Bengal Urban Development Act, 2001 (for short "the Act") for short). In our view, this Tribunal has held that under Section 2(3) of the Act and Section 3(2) of the Act, 1986 the Court has no option to direct the respondent no. 1 to pay the refund of the interest paid by them. It is stated in the impugned order that the respondents have not produced any deficiency on account of the appellants and the fact that the petitioners were also claiming refund of the claim amount by way of compensation. For reasons and facts and circumstances, this Court is of the view that the respondents herein had filed a

PROMPT: Section 302 of the Indian Penal Code shall apply.
 -> [128 words] In any offence under this Chapter-A unless the Central Government is of the opinion that it is necessary in the public interest, the owner

In [50]:
# Load tokenizer from the path resolved for this backbone (handles 24k vs 30k vocabs)
if os.path.exists(TOKENIZER_PATH):
    try:
        tokenizer = BPETokenizer.load(TOKENIZER_PATH)
        print(f"Loaded tokenizer vocab={tokenizer.vocab_size} from {TOKENIZER_PATH}")
    except Exception as e:
        print("Tokenizer load failed:", e)
else:
    print(f"WARNING: tokenizer not found at {TOKENIZER_PATH}")


Loaded tokenizer vocab=30000 from /kaggle/working/cs781_phase1/legal_bpe_gqa_conv.json


### Phase content (`p2`) – cells [19:42] from source


### Phase-2-LNER

In [51]:
import os, json, pickle, random, math, time
from pathlib import Path
from dataclasses import fields
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# IMPORTANT: point these to your actual Phase-1 LEGAL model artifacts.
# Example only: change this directory to the folder containing your LegalLM.
LEGAL_MODEL_DIR = Path("/kaggle/working/cs781_phase1")
# MODEL_CONFIG_PATH = LEGAL_MODEL_DIR / "config.json"
MODEL_WEIGHTS_PATH = LEGAL_MODEL_DIR / "legal_lm_v3.pt"
TOKENIZER_STATE_PATH = LEGAL_MODEL_DIR / "legal_bpe_v2.json"

OUTPUT_DIR = Path("/kaggle/working/lner_full_finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DOCS = None             # None = use all available annotated documents
VALIDATION_FRACTION = 0.10
EPOCHS = 4
BATCH_SIZE = 4              # reduce to 1 or 2 if GPU memory is limited
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WINDOW_OVERLAP = 64

In [52]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold1-gold.json"]
VAL_FILES = ["Phase_2_LNER/Train/fold2-text.json", "Phase_2_LNER/Train/fold2-gold.json"]

In [53]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [54]:
DATA_DIR = Path("/kaggle/working/Phase_2_LNER")
TRAIN_DIR = DATA_DIR / "Train"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def download_lner_file(filename):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename=f"Phase_2_LNER/Train/{filename}",
        local_dir="/kaggle/working/hf_legal_data"
    )

paths = {
    "fold1_text": download_lner_file("fold1-text.json"),
    "fold1_gold": download_lner_file("fold1-gold.json"),
    "fold2_text": download_lner_file("fold2-text.json"),
    "fold2_gold": download_lner_file("fold2-gold.json"),
    "labels": download_lner_file("labels.txt"),
}

with open(paths["fold1_text"], encoding="utf-8") as f:
    text1 = json.load(f)
with open(paths["fold1_gold"], encoding="utf-8") as f:
    gold1 = json.load(f)
with open(paths["fold2_text"], encoding="utf-8") as f:
    text2 = json.load(f)
with open(paths["fold2_gold"], encoding="utf-8") as f:
    gold2 = json.load(f)

with open(paths["labels"], encoding="utf-8") as f:
    ENTITY_TYPES = [line.strip() for line in f if line.strip()]

print("Entity types:", ENTITY_TYPES)
print("Fold 1 documents:", len(text1), "annotations:", len(gold1))
print("Fold 2 documents:", len(text2), "annotations:", len(gold2))

Entity types: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 1 documents: 35 annotations: 35
Fold 2 documents: 35 annotations: 35


In [55]:
def load_model(path=MODEL_PATH):
    if not os.path.exists(path):
        path = "/kaggle/working/cs781_phase1/latest_checkpoint_v3.pt"
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = DualPathLM(ck["config"]).to(DEVICE)
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()}); return m.eval()
model = load_model()
tokenizer = BPETokenizer.load(TOKENIZER_PATH)
print("loaded tokenizer, vocab", tokenizer.vocab_size)

loaded tokenizer, vocab 30000


In [56]:
def normalize_entity(e):
    if isinstance(e, dict):
        start = e.get("start", e.get("start_offset", e.get("start_char")))
        end = e.get("end", e.get("end_offset", e.get("end_char")))
        label = e.get("label", e.get("entity_label", e.get("type")))
        entity_text = e.get("text", e.get("entity_text"))
    elif isinstance(e, (list, tuple)) and len(e) >= 3:
        start, end, label = e[:3]
        entity_text = e[3] if len(e) >= 4 else None
    else:
        raise ValueError(f"Unrecognized entity annotation: {e!r}")

    if start is None or end is None or label is None:
        raise ValueError(f"Missing start/end/label: {e!r}")

    return int(start), int(end), str(label), entity_text

def get_entity_list(gold_record):
    # Supports a direct list of entities or a wrapper with an entities field.
    if isinstance(gold_record, dict) and "entities" in gold_record:
        gold_record = gold_record["entities"]
    if not isinstance(gold_record, list):
        raise ValueError(f"Expected entity list; got {type(gold_record)}")
    return [normalize_entity(e) for e in gold_record]

def validate_document(text, entities, doc_id):
    for start, end, label, entity_text in entities:
        assert 0 <= start < end <= len(text), (
            f"Invalid offsets for document {doc_id}: {(start, end, label)}"
        )
        assert label in ENTITY_TYPES, (
            f"Unknown label {label!r} in document {doc_id}"
        )
        if entity_text is not None and text[start:end] != entity_text:
            raise ValueError(
                f"Text/offset mismatch in {doc_id}: "
                f"{text[start:end]!r} != {entity_text!r}; "
                "check the dataset's end-offset convention."
            )

documents = {}
annotations = {}

for texts, golds in ((text1, gold1), (text2, gold2)):
    for doc_id, text in texts.items():
        if doc_id not in golds:
            raise KeyError(f"No gold annotations for document {doc_id}")
        entities = get_entity_list(golds[doc_id])
        validate_document(text, entities, doc_id)
        documents[str(doc_id)] = text
        annotations[str(doc_id)] = entities

if MAX_DOCS is not None:
    selected_ids = list(documents)[:MAX_DOCS]
    documents = {k: documents[k] for k in selected_ids}
    annotations = {k: annotations[k] for k in selected_ids}

print("Total documents:", len(documents))
example_id = next(iter(documents))
print("Example ID:", example_id)
print("Example text:", documents[example_id][:500])
print("Example entities:", annotations[example_id][:10])

Total documents: 70
Example ID: 115651329
Example text: REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 
Example entities: [(137, 153, 'RESP', 'STATE OF HARYANA'), (252, 261, 'DATE', '19.8.2011'), (276, 308, 'COURT', 'High Court of Punjab and Haryana'), (359, 369, 'COURT', 'High Court'), (575, 585, 'DATE', '18.11.1994'), (637, 644, 'WIT', 'Jagdish'), (695, 704, 'WIT', 'Mange Ram'), (792, 800, 'APP', 'Jage Ram'), (1122, 1130, 'APP', 'Jage Ram'), (1341, 1349, 'APP', 'Jage Ram')]


In [57]:
import os, json, math, random, time, copy, zipfile
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WIN, STRIDE = 512, 256
OUT = "/kaggle/working/lner_gqaconv"; os.makedirs(OUT, exist_ok=True)
CAUSAL = False       # False = bidirectional NER attention; True = causal ablation
BOS = 2; PAD = 0


In [58]:
# ----------------------------- Data and labels -----------------------------
def dl(name):
    return hf_hub_download(HF_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", token=HF_TOKEN)

LABELS = [s.strip() for s in open(dl("Train/labels.txt"), encoding="utf-8") if s.strip()]
BIO = ["O"] + [f"{p}-{lab}" for lab in LABELS for p in ("B", "I")]
l2i = {lab:i for i,lab in enumerate(BIO)}; NL = len(BIO)

def load_fold(k):
    texts = json.load(open(dl(f"Train/fold{k}-text.json"), encoding="utf-8"))
    raw = json.load(open(dl(f"Train/fold{k}-gold.json"), encoding="utf-8"))
    ents = {}
    for doc_id in texts:
        rec = raw[str(doc_id)] if str(doc_id) in raw else raw[doc_id]
        rec = rec["entities"] if isinstance(rec, dict) and "entities" in rec else rec
        arr = []
        for e in rec:
            if isinstance(e, dict):
                s = e.get("start", e.get("start_offset")); end = e.get("end", e.get("end_offset")); lab = e.get("label", e.get("type")); txt = e.get("text")
            else: s, end, lab = e[:3]; txt = e[3] if len(e) > 3 else None
            arr.append((int(s), int(end), str(lab), txt))
        ents[str(doc_id)] = arr
    return {str(k):v for k,v in texts.items()}, ents

# Align offsets to the exact byte-level BPE implementation used in Phase 1.
def encode_with_offsets(text, tok):
    ids, offs = [], []
    for m in tok._split.finditer(text):
        w = m.group()
        if not w: continue
        pieces = [(bytes([b]), i, i+1) for i,b in enumerate(w.encode("utf-8"))]
        b2c = []
        for ci,ch in enumerate(w): b2c.extend([ci] * len(ch.encode("utf-8")))
        while len(pieces) > 1:
            bi, br = -1, 1 << 60
            for i in range(len(pieces)-1):
                r = tok.ranks.get((pieces[i][0], pieces[i+1][0]), 1 << 60)
                if r < br: bi, br = i, r
            if bi < 0: break
            a,b = pieces[bi],pieces[bi+1]; pieces[bi:bi+2] = [(a[0]+b[0],a[1],b[2])]
        for piece, bs, be in pieces:
            s, e = m.start()+b2c[bs], m.start()+b2c[be-1]+1
            while s < e and text[s].isspace(): s += 1
            ids.append(tok.stoi[piece]); offs.append((s,e))
    return ids, offs

def token_labels(entities, offs):
    y = [0]*len(offs); exact = 0
    for s,e,lab,_ in entities:
        idx = [i for i,(a,b) in enumerate(offs) if a < b and a < e and b > s]
        if not idx or lab not in LABELS: continue
        if offs[idx[0]][0] == s and offs[idx[-1]][1] == e: exact += 1
        y[idx[0]] = l2i[f"B-{lab}"]
        for i in idx[1:]: y[i] = l2i[f"I-{lab}"]
    return y, exact

def build_docs(texts, ents):
    docs = {}
    for d,t in tqdm(texts.items(), desc="encode documents"):
        ids,offs = encode_with_offsets(t, tokenizer); y,ex = token_labels(ents[d],offs)
        docs[d] = dict(text=t,ids=ids,offs=offs,y=y,gold=[(s,e,l) for s,e,l,_ in ents[d]],exact=ex)
    return docs

text1,ent1 = load_fold(1); text2,ent2 = load_fold(2)
train_docs,val_docs = build_docs(text1,ent1),build_docs(text2,ent2)
gold_v = {d:x["gold"] for d,x in val_docs.items()}
print("train docs",len(train_docs),"validation docs",len(val_docs),"labels",LABELS)
print("train exact token-boundary alignment:",sum(x["exact"] for x in train_docs.values()),"/",sum(len(x["gold"]) for x in train_docs.values()))


encode documents:   0%|          | 0/35 [00:00<?, ?it/s]

encode documents:   0%|          | 0/35 [00:00<?, ?it/s]

train docs 35 validation docs 35 labels ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
train exact token-boundary alignment: 4048 / 4120


In [59]:
from sklearn.model_selection import train_test_split

# 1. Merge both sets of documents and gold annotations
all_docs = {**train_docs, **val_docs}
all_gold = {**{d: x["gold"] for d, x in train_docs.items()}, **gold_v}

# 2. Get all document IDs and define your split ratio (e.g., 80% train, 20% validation)
doc_ids = list(all_docs.keys())
val_size = 0.2  # Change this ratio as needed (e.g., 0.15 for 15%)

train_ids, val_ids = train_test_split(doc_ids, test_size=val_size, random_state=SEED)

# 3. Create the new merged and re-split datasets
train_docs = {d: all_docs[d] for d in train_ids}
val_docs = {d: all_docs[d] for d in val_ids}
gold_v = {d: all_gold[d] for d in val_ids}

print(f"Merged & Re-split -> Train docs: {len(train_docs)}, Validation docs: {len(val_docs)}")

Merged & Re-split -> Train docs: 56, Validation docs: 14


In [60]:
def viterbi(logp):
    T = logp.shape[0]
    if T == 0: return []
    allow = np.zeros((NL, NL), bool)       # allow[prev, cur]
    for p in range(NL):
        for c in range(NL):
            n = BIO[c]
            allow[p, c] = not (n.startswith("I-") and not (BIO[p] != "O" and BIO[p][2:] == n[2:]))
    trans = np.where(allow, 0.0, -1e9)
    start = np.array([0.0 if not BIO[c].startswith("I-") else -1e9 for c in range(NL)])
    dp = start + logp[0]; back = np.zeros((T, NL), int)
    for t in range(1, T):
        sc = dp[:, None] + trans
        back[t] = sc.argmax(0); dp = sc.max(0) + logp[t]
    path = [int(dp.argmax())]
    for t in range(T - 1, 0, -1): path.append(int(back[t][path[-1]]))
    return path[::-1]

def tags_to_spans(tags, offs):
    spans, cur = [], None
    for t, (s, e) in zip(tags, offs):
        if s == e: continue                  # whitespace token: skip, never breaks a span
        n = BIO[t]
        if n == "O":
            if cur: spans.append(tuple(cur)); cur = None
        elif n.startswith("B-") or cur is None or cur[2] != n[2:]:
            if cur: spans.append(tuple(cur))
            cur = [s, e, n[2:]]
        else:
            cur[1] = e
    if cur: spans.append(tuple(cur))
    return spans

def strict_prf(gold, pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P: (tp if x in G else fp)[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    per = {}
    for l in LABELS:
        if tp[l] + fp[l] + fn[l] == 0: continue
        p = tp[l] / max(1, tp[l] + fp[l]); r = tp[l] / max(1, tp[l] + fn[l])
        per[l] = (p, r, 2 * p * r / max(1e-9, p + r), tp[l] + fn[l])
    m = lambda i: float(np.mean([v[i] for v in per.values()])) if per else 0.0
    return dict(P=m(0), R=m(1), F1=m(2), per=per)

def error_analysis(gold, pred):
    c = Counter()
    for d in set(gold) | set(pred):
        G, P = set(gold.get(d, [])), set(pred.get(d, []))
        for x in P:
            if x in G: c["exact"] += 1; continue
            if any((g[0], g[1]) == (x[0], x[1]) for g in G): c["label_confusion"] += 1
            elif any(g[2] == x[2] and g[0] < x[1] and x[0] < g[1] for g in G): c["boundary_error"] += 1
            else: c["false_positive"] += 1
        for g in G - P:
            if not any(p[0] < g[1] and g[0] < p[1] for p in P): c["missed(FN)"] += 1
    return dict(c)

def show(res, title=""):
    print(f"{title} macro P {res['P']:.4f} R {res['R']:.4f} F1 {res['F1']:.4f}")
    for l, (p, r, f, n) in res["per"].items(): print(f"   {l:10s} P {p:.3f} R {r:.3f} F1 {f:.3f} n={n}")

# Oracle: gold BIO tags -> spans.  Upper bound of what the tokenizer alignment allows.
gold_v = {d: x["gold"] for d, x in val_docs.items()}
oracle = {d: tags_to_spans(x["y"], x["offs"]) for d, x in val_docs.items()}
show(strict_prf(gold_v, oracle), "ORACLE (should be ~1.0)")

ORACLE (should be ~1.0) macro P 0.9777 R 0.9777 F1 0.9777
   APP        P 0.934 R 0.934 F1 0.934 n=61
   RESP       P 1.000 R 1.000 F1 1.000 n=66
   A.COUNSEL  P 1.000 R 1.000 F1 1.000 n=26
   R.COUNSEL  P 1.000 R 1.000 F1 1.000 n=21
   JUDGE      P 1.000 R 1.000 F1 1.000 n=64
   WIT        P 1.000 R 1.000 F1 1.000 n=62
   AUTH       P 0.974 R 0.974 F1 0.974 n=190
   COURT      P 1.000 R 1.000 F1 1.000 n=136
   STAT       P 0.948 R 0.948 F1 0.948 n=212
   PREC       P 0.876 R 0.876 F1 0.876 n=201
   DATE       P 1.000 R 1.000 F1 1.000 n=315
   CASENO     P 1.000 R 1.000 F1 1.000 n=161


In [61]:
def make_windows(n,win,step):
    if n == 0: return [(0,0)]
    if n <= win: return [(0,n)]
    starts=list(range(0,n-win+1,step))
    if starts[-1]+win < n: starts.append(n-win)
    return [(s,s+win) for s in starts]

def make_items(docs,step=STRIDE):
    return [([BOS]+x["ids"][a:b],[-100]+x["y"][a:b]) for x in docs.values() for a,b in make_windows(len(x["ids"]),WIN,step)]

def collate(batch):
    L=max(len(ids) for ids,_ in batch)
    ids=torch.full((len(batch),L),PAD,dtype=torch.long); y=torch.full((len(batch),L),-100,dtype=torch.long)
    valid=torch.zeros((len(batch),L),dtype=torch.bool)
    for k,(ii,yy) in enumerate(batch):
        ids[k,:len(ii)]=torch.tensor(ii,dtype=torch.long); y[k,:len(yy)]=torch.tensor(yy,dtype=torch.long); valid[k,:len(ii)]=True
    return ids,y,valid

# -------------------------- GQAConv backbone -------------------------------
# GQAConv's source defines DualPathLM(cfg), with cfg keys dim/layers/heads/kv_heads,
# and each Block.forward(x, mask=None, cache=None, start=0) -> (hidden, cache).
# No MLA monkey patch is used. The boolean attention mask marks allowed key positions.
def load_lm(path=None):
    path = path or globals().get("GQACONV_MODEL_PATH", globals().get("MODEL_PATH", None))
    if path is None: raise NameError("Set GQACONV_MODEL_PATH to the Phase-1 GQAConv checkpoint path.")
    ck=torch.load(path,map_location="cpu",weights_only=False)
    cfg=ck.get("config",ck.get("cfg"))
    if cfg is None:
        # If checkpoint only contains weights, config must be supplied by Phase-1 setup.
        cfg=globals().get("GQACONV_CFG",None)
    if cfg is None: raise KeyError("Checkpoint needs 'config'/'cfg', or define GQACONV_CFG before running.")
    if not isinstance(cfg,dict): cfg=dict(cfg)
    if "DualPathLM" not in globals(): raise NameError("Define/import the Phase-1 DualPathLM class before this NER code.")
    lm=DualPathLM(cfg)
    state=ck.get("model",ck.get("model_state_dict",ck.get("state_dict",ck)))
    state={k.replace("_orig_mod.",""):v for k,v in state.items() if isinstance(v,torch.Tensor)}
    missing,unexpected=lm.load_state_dict(state,strict=False)
    if missing or unexpected: print("Checkpoint load report: missing",len(missing),"unexpected",len(unexpected))
    return lm

In [62]:
class Adapter(nn.Module):
    def __init__(self,d,r):
        super().__init__(); self.down=nn.Linear(d,r); self.up=nn.Linear(r,d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return x+self.up(F.gelu(self.down(x)))

class Backbone(nn.Module):
    def __init__(self,lm,causal=False,adapter_dim=0):
        super().__init__(); self.lm=lm; self.causal=causal; self.dim=lm.cfg["dim"]
        self.adapters=nn.ModuleList([Adapter(self.dim,adapter_dim) for _ in lm.blocks]) if adapter_dim else None
    def forward(self,ids,valid):
        b,t=ids.shape; x=self.lm.wte(ids)
        # For GQAAttention, mask=True means a key may be attended to.
        key_valid=valid[:,None,None,:]
        if self.causal:
            causal=torch.ones((t,t),device=ids.device,dtype=torch.bool).tril()[None,None,:,:]
            attn_mask=key_valid & causal
        else: attn_mask=key_valid
        for i,blk in enumerate(self.lm.blocks):
            x,_=blk(x,mask=attn_mask)
            if self.adapters is not None: x=self.adapters[i](x)
        return self.lm.ln_f(x)

class NERModel(nn.Module):
    def __init__(self,bb,head="linear",drop=0.1):
        super().__init__(); self.backbone=bb; self.drop=nn.Dropout(drop); d=bb.dim
        self.head=nn.Linear(d,NL) if head=="linear" else nn.Sequential(nn.Linear(d,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,NL))
    def forward(self,ids,valid): return self.head(self.drop(self.backbone(ids,valid)))


In [63]:
class LoRALinear(nn.Module):
    def __init__(self,base,r=8,alpha=16):
        super().__init__(); self.base=base; self.scale=alpha/r
        self.A=nn.Parameter(torch.randn(r,base.in_features)/math.sqrt(base.in_features)); self.B=nn.Parameter(torch.zeros(base.out_features,r))
    def forward(self,x): return self.base(x)+(x@self.A.t()@self.B.t())*self.scale

class LoRMALinear(nn.Module):
    def __init__(self,base,r=8,alpha=16):
        super().__init__(); self.base=base; self.scale=alpha/r
        self.A=nn.Parameter(torch.randn(r,base.out_features)/math.sqrt(base.out_features)); self.B=nn.Parameter(torch.zeros(base.out_features,r))
    def forward(self,x):
        y=self.base(x); return y+(y@self.A.t()@self.B.t())*self.scale

In [64]:
TARGETS={"q","kv","o","w13","w2"}
def inject(lm,cls,r,alpha):
    count=0
    for blk in lm.blocks:
        for mod in list(blk.modules()):
            for name,ch in list(mod.named_children()):
                if isinstance(ch,nn.Linear) and name in TARGETS:
                    setattr(mod,name,cls(ch,r,alpha)); count+=1
    if count==0: raise RuntimeError("No LoRA targets found. Check GQAConv projection names.")
    print("Injected low-rank adapters into",count,"linear projections")

def build(kind,head="linear",r=8,k_top=3,path=None):
    lm=load_lm(path); bb=Backbone(lm,CAUSAL,64 if kind=="adapter" else 0)
    if kind in ("lora","lorma"):
        for p in lm.parameters(): p.requires_grad=False
        inject(lm,LoRALinear if kind=="lora" else LoRMALinear,r,2*r)
    elif kind=="adapter":
        for p in lm.parameters(): p.requires_grad=False
    elif kind=="partial":
        for p in lm.parameters(): p.requires_grad=False
        for blk in lm.blocks[-k_top:]:
            for p in blk.parameters(): p.requires_grad=True
        for p in lm.ln_f.parameters(): p.requires_grad=True
    return NERModel(bb,head).to(DEVICE)

def n_params(m):
    total=sum(p.numel() for p in m.parameters()); trainable=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total,trainable


In [65]:
@torch.no_grad()
def predict_docs(model,docs,decode="viterbi",bs=8):
    model.eval(); out={}
    for d,x in docs.items():
        n=len(x["ids"]); acc=np.zeros((n,NL),np.float32); cnt=np.zeros(n,np.float32)
        wins=make_windows(n,WIN,WIN//2)
        for i in range(0,len(wins),bs):
            chunk=wins[i:i+bs]
            batch=[([BOS]+x["ids"][a:b],[0]*(b-a+1)) for a,b in chunk]
            ids,_,valid=collate(batch)
            with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP): logits=model(ids.to(DEVICE),valid.to(DEVICE))
            lp=logits.float().log_softmax(-1).cpu().numpy()
            for k,(a,b) in enumerate(chunk): acc[a:b]+=lp[k,1:b-a+1]; cnt[a:b]+=1
        if n:
            lp=acc/np.maximum(cnt,1)[:,None]; tags=viterbi(lp) if decode=="viterbi" else lp.argmax(-1).tolist()
            out[d]=tags_to_spans(tags,x["offs"])
        else: out[d]=[]
    return out

def evaluate(model,docs,decode="viterbi"):
    pred=predict_docs(model,docs,decode); return strict_prf({d:x["gold"] for d,x in docs.items()},pred),pred


In [66]:
from tqdm.auto import tqdm
def class_weights(docs, power=0.5, cap=10.0):
    c = np.bincount(np.concatenate([x["y"] for x in docs.values()]), minlength=NL).astype(float) + 1
    w = np.clip((c[0] / c) ** power, 0, cap); w[0] = 1.0
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train(model, name, epochs, lr, head_lr=1e-3, bs=8, wd=0.01, cw=None, teacher=None, T=2.0, alpha=0.5, warm=0.1, snapshot_eval=True, resume_path=None):
    # ADD: Checkpoint directory creation
    ckpt_dir = f"{OUT}/checkpoints/{name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    items = make_items(train_docs); steps = epochs * math.ceil(len(items) / bs)
    bb_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head")]
    hd_p = [p for n, p in model.named_parameters() if n.startswith("head")]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": lr}, {"params": hd_p, "lr": head_lr}], weight_decay=wd)
    base = [lr, head_lr]; scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time(); hist = []
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    best, best_state, step, t0 = -1, None, 0, time.time()
    hist = []
    start_epoch = 0  # ADD: Track starting epoch

    # ADD: Load checkpoint if resume_path is provided
    if resume_path and os.path.exists(resume_path):
        print(f"--> Resuming from: {resume_path}")
        ckpt = torch.load(resume_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"], strict=False)
        opt.load_state_dict(ckpt["optimizer_state"])
        if USE_AMP and AMP_DTYPE == torch.float16 and "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        step = ckpt["step"]
        best = ckpt["best_f1"]
        best_state = ckpt["best_state"]
        hist = ckpt.get("hist", [])
    for ep in tqdm(range(start_epoch, epochs)):
        model.train(); random.shuffle(items); tl = 0
        for i in range(0, len(items), bs):
            f = (step + 1) / (warm * steps) if step < warm * steps else 0.5 * (1 + math.cos(math.pi * (step - warm * steps) / max(1, steps - warm * steps)))
            for g, b0 in zip(opt.param_groups, base): g["lr"] = b0 * f
            ids, y, m = [t.to(DEVICE) for t in collate(items[i:i + bs])]
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg = model(ids, m)
                loss = F.cross_entropy(lg.float().reshape(-1, NL), y.reshape(-1), weight=cw, ignore_index=-100)
                if teacher is not None:
                    with torch.no_grad(): tl_ = teacher(ids, m).float()
                    sel = y != -100
                    kd = F.kl_div((lg.float()[sel] / T).log_softmax(-1), (tl_[sel] / T).softmax(-1), reduction="batchmean") * T * T
                    loss = alpha * loss + (1 - alpha) * kd
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1; tl += loss.item()
        res, _ = evaluate(model, val_docs); hist.append((ep + 1, tl / math.ceil(len(items) / bs), res["F1"]))
        print(f"[{name}] ep {ep+1}/{epochs} loss {hist[-1][1]:.4f} | val strict macro P {res['P']:.3f} R {res['R']:.3f} F1 {res['F1']:.4f}")
        if res["F1"] > best: best = res["F1"]; best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
        # inside epoch loop, after calculating `res`:
        if res["F1"] > best:
            best = res["F1"]
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in names}
            # ADD: Save best weights separately
            torch.save(best_state, f"{ckpt_dir}/{name}_best_weights.pt")

        # ADD: Save latest checkpoint for resuming
        ckpt_payload = {
            "epoch": ep,
            "step": step,
            "model_state": {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
            "optimizer_state": opt.state_dict(),
            "scaler_state": scaler.state_dict() if USE_AMP else None,
            "best_f1": best,
            "best_state": best_state,
            "hist": hist,
        }
        torch.save(ckpt_payload, f"{ckpt_dir}/latest_checkpoint.pt")
    model.load_state_dict(best_state, strict=False)
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else float("nan")
    return dict(best_val_f1=best, train_sec=time.time() - t0, peak_gb=peak, hist=hist)

In [67]:
test_texts=json.load(open(dl("Test/fold3-text.json"),encoding="utf-8")); test_texts={str(k):v for k,v in test_texts.items()}
test_docs={}
for d,t in tqdm(test_texts.items(),desc="encode test"):
    ids,offs=encode_with_offsets(t,tokenizer); test_docs[d]=dict(text=t,ids=ids,offs=offs,y=[0]*len(ids),gold=[])

def write_test(model,name):
    pred=predict_docs(model,test_docs)
    js={d:[[s,e,lab,test_texts[d][s:e]] for s,e,lab in spans] for d,spans in pred.items()}
    jp=f"{OUT}/prediction_{name}.json"
    with open(jp,"w",encoding="utf-8") as f: json.dump(js,f,ensure_ascii=False)
    with zipfile.ZipFile(f"{OUT}/submission_{name}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(jp,"prediction.json")
    print("Wrote",f"{OUT}/submission_{name}.zip")

RESULTS, MODELS = {}, {}
CW = class_weights(train_docs)
def run(name, kind, epochs, lr, head_lr=1e-3, head="linear", cw=CW, resume = False, **kw):
    model = build(kind, head=head, **kw); tot, tr = n_params(model)
    print(f"\n=== {name}: total {tot/1e6:.1f}M params, trainable {tr/1e6:.2f}M ({100*tr/tot:.1f}%)")
    ckpt_path = f"{OUT}/checkpoints/{name}/latest_checkpoint.pt"
    resume_path = ckpt_path if (resume and os.path.exists(ckpt_path)) else None

    # CHANGE: Pass resume_path to train()
    info = train(model, name, epochs, lr, head_lr, cw=cw, resume_path=resume_path)
    t0 = time.time(); res, pred = evaluate(model, val_docs); info["infer_sec"] = time.time() - t0
    info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
    RESULTS[name] = info; MODELS[name] = model; write_test(model, name); show(res, f"[{name}] FINAL VAL"); return model

encode test:   0%|          | 0/35 [00:00<?, ?it/s]

In [68]:
RUN_ALL = True
if RUN_ALL:
    run("gqaconv_full_ft","full",epochs=4,lr=5e-5,resume=True)
    run("gqaconv_partial_top3","partial",epochs=10,lr=1e-4,k_top=3)
    run("gqaconv_lora_r8","lora",epochs=4,lr=1e-3,r=8,resume=True)
    run("gqaconv_lorma_r8","lorma",epochs=4,lr=1e-3,r=8,resume=True)
    run("gqaconv_adapter64","adapter",epochs=4,lr=1e-3)


=== gqaconv_full_ft: total 78.2M params, trainable 78.22M (100.0%)
--> Resuming from: /kaggle/working/lner_gqaconv/checkpoints/gqaconv_full_ft/latest_checkpoint.pt


0it [00:00, ?it/s]

Wrote /kaggle/working/lner_gqaconv/submission_gqaconv_full_ft.zip
[gqaconv_full_ft] FINAL VAL macro P 0.4770 R 0.5045 F1 0.4833
   APP        P 0.186 R 0.213 F1 0.198 n=61
   RESP       P 0.288 R 0.227 F1 0.254 n=66
   A.COUNSEL  P 0.368 R 0.269 F1 0.311 n=26
   R.COUNSEL  P 0.424 R 0.667 F1 0.519 n=21
   JUDGE      P 0.549 R 0.703 F1 0.616 n=64
   WIT        P 0.432 R 0.258 F1 0.323 n=62
   AUTH       P 0.326 R 0.442 F1 0.375 n=190
   COURT      P 0.817 R 0.853 F1 0.835 n=136
   STAT       P 0.720 R 0.656 F1 0.686 n=212
   PREC       P 0.303 R 0.313 F1 0.308 n=201
   DATE       P 0.929 R 0.949 F1 0.939 n=315
   CASENO     P 0.382 R 0.503 F1 0.434 n=161

=== gqaconv_partial_top3: total 78.2M params, trainable 20.70M (26.5%)


  0%|          | 0/10 [00:00<?, ?it/s]

[gqaconv_partial_top3] ep 1/10 loss 0.8614 | val strict macro P 0.322 R 0.520 F1 0.3834
[gqaconv_partial_top3] ep 2/10 loss 0.1734 | val strict macro P 0.400 R 0.510 F1 0.4332
[gqaconv_partial_top3] ep 3/10 loss 0.0843 | val strict macro P 0.459 R 0.508 F1 0.4722
[gqaconv_partial_top3] ep 4/10 loss 0.0439 | val strict macro P 0.467 R 0.511 F1 0.4782
[gqaconv_partial_top3] ep 5/10 loss 0.0249 | val strict macro P 0.471 R 0.498 F1 0.4727
[gqaconv_partial_top3] ep 6/10 loss 0.0123 | val strict macro P 0.493 R 0.502 F1 0.4830
[gqaconv_partial_top3] ep 7/10 loss 0.0059 | val strict macro P 0.472 R 0.480 F1 0.4681
[gqaconv_partial_top3] ep 8/10 loss 0.0034 | val strict macro P 0.487 R 0.487 F1 0.4776
[gqaconv_partial_top3] ep 9/10 loss 0.0023 | val strict macro P 0.487 R 0.484 F1 0.4769
[gqaconv_partial_top3] ep 10/10 loss 0.0021 | val strict macro P 0.487 R 0.481 F1 0.4755
Wrote /kaggle/working/lner_gqaconv/submission_gqaconv_partial_top3.zip
[gqaconv_partial_top3] FINAL VAL macro P 0.4932 

  0%|          | 0/4 [00:00<?, ?it/s]

[gqaconv_lora_r8] ep 1/4 loss 0.6873 | val strict macro P 0.308 R 0.439 F1 0.3535
[gqaconv_lora_r8] ep 2/4 loss 0.1370 | val strict macro P 0.413 R 0.540 F1 0.4446
[gqaconv_lora_r8] ep 3/4 loss 0.0602 | val strict macro P 0.414 R 0.495 F1 0.4476
[gqaconv_lora_r8] ep 4/4 loss 0.0301 | val strict macro P 0.436 R 0.487 F1 0.4558
Wrote /kaggle/working/lner_gqaconv/submission_gqaconv_lora_r8.zip
[gqaconv_lora_r8] FINAL VAL macro P 0.4359 R 0.4866 F1 0.4558
   APP        P 0.131 R 0.180 F1 0.152 n=61
   RESP       P 0.185 R 0.182 F1 0.183 n=66
   A.COUNSEL  P 0.417 R 0.385 F1 0.400 n=26
   R.COUNSEL  P 0.286 R 0.381 F1 0.327 n=21
   JUDGE      P 0.551 R 0.672 F1 0.606 n=64
   WIT        P 0.237 R 0.145 F1 0.180 n=62
   AUTH       P 0.342 R 0.532 F1 0.416 n=190
   COURT      P 0.738 R 0.868 F1 0.797 n=136
   STAT       P 0.701 R 0.675 F1 0.688 n=212
   PREC       P 0.292 R 0.318 F1 0.305 n=201
   DATE       P 0.937 R 0.943 F1 0.940 n=315
   CASENO     P 0.415 R 0.559 F1 0.476 n=161
Injected l

  0%|          | 0/4 [00:00<?, ?it/s]

[gqaconv_lorma_r8] ep 1/4 loss 0.6701 | val strict macro P 0.319 R 0.421 F1 0.3459
[gqaconv_lorma_r8] ep 2/4 loss 0.1544 | val strict macro P 0.402 R 0.503 F1 0.4387
[gqaconv_lorma_r8] ep 3/4 loss 0.0684 | val strict macro P 0.440 R 0.521 F1 0.4720
[gqaconv_lorma_r8] ep 4/4 loss 0.0339 | val strict macro P 0.469 R 0.531 F1 0.4928
Wrote /kaggle/working/lner_gqaconv/submission_gqaconv_lorma_r8.zip
[gqaconv_lorma_r8] FINAL VAL macro P 0.4689 R 0.5306 F1 0.4928
   APP        P 0.143 R 0.230 F1 0.176 n=61
   RESP       P 0.224 R 0.227 F1 0.226 n=66
   A.COUNSEL  P 0.593 R 0.615 F1 0.604 n=26
   R.COUNSEL  P 0.393 R 0.524 F1 0.449 n=21
   JUDGE      P 0.519 R 0.625 F1 0.567 n=64
   WIT        P 0.364 R 0.258 F1 0.302 n=62
   AUTH       P 0.296 R 0.521 F1 0.378 n=190
   COURT      P 0.732 R 0.801 F1 0.765 n=136
   STAT       P 0.685 R 0.646 F1 0.665 n=212
   PREC       P 0.314 R 0.358 F1 0.335 n=201
   DATE       P 0.941 R 0.959 F1 0.950 n=315
   CASENO     P 0.424 R 0.602 F1 0.497 n=161

===

  0%|          | 0/4 [00:00<?, ?it/s]

[gqaconv_adapter64] ep 1/4 loss 0.7196 | val strict macro P 0.353 R 0.483 F1 0.3902
[gqaconv_adapter64] ep 2/4 loss 0.1973 | val strict macro P 0.387 R 0.523 F1 0.4304
[gqaconv_adapter64] ep 3/4 loss 0.1163 | val strict macro P 0.395 R 0.540 F1 0.4458
[gqaconv_adapter64] ep 4/4 loss 0.0807 | val strict macro P 0.405 R 0.523 F1 0.4500
Wrote /kaggle/working/lner_gqaconv/submission_gqaconv_adapter64.zip
[gqaconv_adapter64] FINAL VAL macro P 0.4050 R 0.5233 F1 0.4500
   APP        P 0.090 R 0.246 F1 0.132 n=61
   RESP       P 0.233 R 0.258 F1 0.245 n=66
   A.COUNSEL  P 0.419 R 0.500 F1 0.456 n=26
   R.COUNSEL  P 0.250 R 0.381 F1 0.302 n=21
   JUDGE      P 0.500 R 0.672 F1 0.573 n=64
   WIT        P 0.265 R 0.290 F1 0.277 n=62
   AUTH       P 0.288 R 0.605 F1 0.390 n=190
   COURT      P 0.673 R 0.801 F1 0.732 n=136
   STAT       P 0.671 R 0.693 F1 0.682 n=212
   PREC       P 0.294 R 0.343 F1 0.317 n=201
   DATE       P 0.878 R 0.937 F1 0.906 n=315
   CASENO     P 0.299 R 0.553 F1 0.388 n=16

In [69]:
# Summary
if RESULTS:
    import pandas as pd
    rows=[dict(exp=k,total_M=round(v["total_M"],2),trainable_M=round(v["trainable_M"],2),P=round(v["P"],4),R=round(v["R"],4),F1=round(v["F1"],4),train_min=round(v["train_sec"]/60,1),peak_GB=round(v["peak_gb"],2),infer_sec=round(v["infer_sec"],1)) for k,v in RESULTS.items()]
    df=pd.DataFrame(rows).sort_values("F1",ascending=False); print(df.to_string(index=False))
    best=df.iloc[0]["exp"]; print("BEST validation experiment:",best)
    for dec in ("greedy","viterbi"):
        r,_=evaluate(MODELS[best],val_docs,dec); print(f"decode={dec}: macro-F1={r['F1']:.4f}")
    show(strict_prf(gold_v,evaluate(MODELS[best],val_docs)[1]),f"per-label ({best})")
    print("Submission:",f"{OUT}/submission_{best}.zip")

                 exp  total_M  trainable_M      P      R     F1  train_min  peak_GB  infer_sec
    gqaconv_lorma_r8    79.17         0.97 0.4689 0.5306 0.4928        9.7     4.83       12.8
     gqaconv_full_ft    78.22        78.22 0.4770 0.5045 0.4833        0.0     2.47       11.3
gqaconv_partial_top3    78.22        20.70 0.4932 0.5018 0.4830       17.2     2.94       11.6
     gqaconv_lora_r8    79.03         0.84 0.4359 0.4866 0.4558        9.6     4.41       12.7
   gqaconv_adapter64    79.01         0.81 0.4050 0.5233 0.4500        8.1     4.54       11.8
BEST validation experiment: gqaconv_lorma_r8
decode=greedy: macro-F1=0.3705
decode=viterbi: macro-F1=0.4928
per-label (gqaconv_lorma_r8) macro P 0.4689 R 0.5306 F1 0.4928
   APP        P 0.143 R 0.230 F1 0.176 n=61
   RESP       P 0.224 R 0.227 F1 0.226 n=66
   A.COUNSEL  P 0.593 R 0.615 F1 0.604 n=26
   R.COUNSEL  P 0.393 R 0.524 F1 0.449 n=21
   JUDGE      P 0.519 R 0.625 F1 0.567 n=64
   WIT        P 0.364 R 0.258 F1 0.302 

In [70]:
# MODELS
!du -h --max-depth=1 /kaggle/working
# !rm -rf /kaggle/working/phase2-custom-mha

4.0K	/kaggle/working/lner_full_finetune
3.1M	/kaggle/working/hf_legal_data
3.0G	/kaggle/working/phase2-alibi-gqa
1.6G	/kaggle/working/lner_gqaconv
4.5G	/kaggle/working/cs781_phase1
3.7G	/kaggle/working/phase2-mla-moe
3.1G	/kaggle/working/phase2-gqa-conv
8.0K	/kaggle/working/Phase_2_LNER
236K	/kaggle/working/.virtual_documents
16G	/kaggle/working


In [72]:
teacher = MODELS["gqaconv_full_ft"].eval()
for p in teacher.parameters(): p.requires_grad = False
p1 = load_lm(); cfg = dict(p1.cfg); keep = [0, 2, 4, 7]; cfg["layers"] = len(keep)
slm = DualPathLM(cfg); slm.wte.load_state_dict(p1.wte.state_dict()); slm.ln_f.load_state_dict(p1.ln_f.state_dict())
for i, j in enumerate(keep): slm.blocks[i].load_state_dict(p1.blocks[j].state_dict())   # init from Phase-1 weights
student = NERModel(Backbone(slm, CAUSAL)).to(DEVICE); tot, tr = n_params(student)
print(f"=== KD student: {tot/1e6:.1f}M params (teacher {RESULTS['gqaconv_full_ft']['total_M']:.1f}M)")
info = train(student, "kd_student", epochs=4, lr=1e-4, cw=CW, teacher=teacher, T=2.0, alpha=0.5)
t0 = time.time(); res, pred = evaluate(student, val_docs); info["infer_sec"] = time.time() - t0
info.update(total_M=tot / 1e6, trainable_M=tr / 1e6, P=res["P"], R=res["R"], F1=res["F1"], per=res["per"], errors=error_analysis(gold_v, pred))
RESULTS["kd_student"] = info; MODELS["kd_student"] = student; write_test(student, "kd_student"); show(res, "[kd_student] FINAL VAL")
# Control: same student WITHOUT distillation (does KD actually help?)
student0 = NERModel(Backbone(copy.deepcopy(slm), CAUSAL)).to(DEVICE)
i0 = train(student0, "student_noKD", 4, 1e-4, cw=CW)
RESULTS["student_noKD"] = dict(best_val_f1=i0["best_val_f1"], F1=i0["best_val_f1"], total_M=tot / 1e6, trainable_M=tr / 1e6, train_sec=i0["train_sec"], peak_gb=i0["peak_gb"])

=== KD student: 50.6M params (teacher 78.2M)


  0%|          | 0/4 [00:00<?, ?it/s]

[kd_student] ep 1/4 loss 0.9903 | val strict macro P 0.333 R 0.388 F1 0.3500
[kd_student] ep 2/4 loss 0.1248 | val strict macro P 0.407 R 0.451 F1 0.4230
[kd_student] ep 3/4 loss 0.0431 | val strict macro P 0.440 R 0.475 F1 0.4508
[kd_student] ep 4/4 loss 0.0211 | val strict macro P 0.444 R 0.469 F1 0.4483
Wrote /kaggle/working/lner_gqaconv/submission_kd_student.zip
[kd_student] FINAL VAL macro P 0.4396 R 0.4749 F1 0.4508
   APP        P 0.152 R 0.164 F1 0.157 n=61
   RESP       P 0.306 R 0.288 F1 0.297 n=66
   A.COUNSEL  P 0.400 R 0.385 F1 0.392 n=26
   R.COUNSEL  P 0.205 R 0.381 F1 0.267 n=21
   JUDGE      P 0.564 R 0.688 F1 0.620 n=64
   WIT        P 0.290 R 0.145 F1 0.194 n=62
   AUTH       P 0.369 R 0.484 F1 0.419 n=190
   COURT      P 0.717 R 0.838 F1 0.773 n=136
   STAT       P 0.708 R 0.642 F1 0.673 n=212
   PREC       P 0.267 R 0.299 F1 0.282 n=201
   DATE       P 0.921 R 0.921 F1 0.921 n=315
   CASENO     P 0.375 R 0.466 F1 0.416 n=161


  0%|          | 0/4 [00:00<?, ?it/s]

[student_noKD] ep 1/4 loss 0.2622 | val strict macro P 0.369 R 0.460 F1 0.4027
[student_noKD] ep 2/4 loss 0.0285 | val strict macro P 0.420 R 0.471 F1 0.4279
[student_noKD] ep 3/4 loss 0.0084 | val strict macro P 0.425 R 0.456 F1 0.4352
[student_noKD] ep 4/4 loss 0.0021 | val strict macro P 0.438 R 0.451 F1 0.4368
